In [1]:
import pandas as pd
# Set option to display all columns
pd.set_option('display.max_columns', None)

# A) Counts

## import data

In [2]:
import duckdb
from pathlib import Path

con = duckdb.connect()

# Low-memory settings
con.execute("PRAGMA threads=1;")
con.execute("PRAGMA preserve_insertion_order=false;")
con.execute("PRAGMA enable_object_cache=false;")
con.execute("PRAGMA memory_limit='2GB';")           # try 1GB if still unstable
con.execute("PRAGMA temp_directory='data/tmp_duckdb';")

# 2) Build paths robustly from the notebook folder
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

BASE = ROOT / "data" / "by_server"

# IMPORTANT: your files are hive-partitioned like:
all_backends = (BASE / "*" / "*.parquet").as_posix()

con.execute(f"""
CREATE OR REPLACE VIEW all_backends AS
SELECT * FROM read_parquet('{all_backends}', hive_partitioning=true, union_by_name=true);
""")

# A unified "all_rows" view
con.execute("""
CREATE OR REPLACE VIEW all_rows AS
SELECT * FROM all_backends
""")

print(con.execute("SHOW TABLES").fetchall())


[('all_backends',), ('all_rows',)]


In [3]:
con.execute("""
SELECT backend, COUNT(*) AS total, COUNT(record_id) AS with_record_id
FROM (
    SELECT backend, record_id FROM all_backends
)
GROUP BY backend
""").df()


,backend,total,with_record_id
0,crossref,3919449,3919449
1,datacite,3525031,3525031
2,openalex,2352367,2352367
3,jxiv,902,902


## total count

In [4]:
con.execute("""
SELECT COUNT(DISTINCT record_id) AS total_dedup_records
FROM (
    SELECT record_id FROM all_backends
)
""").df()


,total_dedup_records
0,7980925


## count per backend

In [5]:
con.execute("""
SELECT
    backend,
    COUNT(DISTINCT record_id) AS dedup_records
FROM (
    SELECT backend, record_id FROM all_backends
)
GROUP BY backend
ORDER BY dedup_records DESC
""").df()


,backend,dedup_records
0,datacite,3522057
1,crossref,2753774
2,openalex,1704192
3,jxiv,902


## count per server

In [6]:
backend_server_name_df = con.execute("""
SELECT
    backend,
    server_name,
    COUNT(DISTINCT record_id) AS dedup_records
FROM (
    SELECT backend, server_name, record_id FROM all_backends
)
GROUP BY backend, server_name
ORDER BY dedup_records DESC
""").df()
backend_server_name_df

,backend,server_name,dedup_records
0,datacite,arXiv,2920797
1,crossref,SSRN,1259256
2,openalex,HAL,1056424
3,crossref,Research Square,450818
4,openalex,RePEc: Research Papers in Economics,389398
...,...,...,...
107,crossref,MNI Open Research,20
108,crossref,Prepublicaciones OpenCiencia,8
109,crossref,EmeRI,8
110,crossref,Therapoid,7


In [7]:
backend_server_name_df.to_csv("outputs_new/tracker_data/initial_backend_server_total_count_before_preprocessing.csv", index=False)

In [8]:
server_name_df = con.execute("""
SELECT server_name, COUNT(DISTINCT record_id) AS n
FROM all_backends
GROUP BY 1
ORDER BY n DESC
LIMIT 200;
""").df()


In [9]:
server_name_df.head(60)

,server_name,n
0,arXiv,2920797
1,SSRN,1259256
2,HAL,1056424
3,Research Square,450818
4,RePEc: Research Papers in Economics,389398
5,bioRxiv,306948
6,AgEcon Search,188173
7,ResearchGate,181231
8,Zenodo,166786
9,Open Science Framework,119481


In [10]:
server_name_df.tail(52)

,server_name,n
57,APSA Preprints,1470
58,PREPRINTS.RU,1415
59,Keldysh Institute Preprints,1258
60,searchRxiv,1234
61,HRB Open Research,1012
62,Jxiv,902
63,ARPHA Preprints,890
64,MetaArXiv,880
65,SportRxiv,878
66,Gates Open Research,863


In [11]:
server_name_df.to_csv("outputs_new/tracker_data/initial_total_count_before_preprocessing.csv", index=False)

# B) Explorations

In [12]:
# con.execute(f"""
# CREATE OR REPLACE VIEW server_thin AS
# SELECT
#   CAST(record_id AS VARCHAR)           AS record_id,
#   CAST(server_name AS VARCHAR)         AS server_name,
#   CAST(backend AS VARCHAR)             AS backend,

#   CAST(doi AS VARCHAR)                 AS doi,
#   CAST(doi_url AS VARCHAR)             AS doi_url,
#   CAST(landing_page_url AS VARCHAR)    AS landing_page_url,

#   CAST(version_label AS VARCHAR)       AS version_label,

#   -- Relationships (keep these for true version links)
#   CAST(relations_json AS VARCHAR)       AS relations_json,
#   CAST(raw_relationships_json AS VARCHAR)       AS raw_relationships_json,
#   CAST(is_version_of AS VARCHAR)       AS is_version_of,      -- keep as text; we’ll interpret later
#   CAST(version_of_ids_json AS VARCHAR) AS version_of_ids_json,
#   CAST(is_preprint_of AS VARCHAR)      AS is_preprint_of,
#   CAST(published_version_ids_json AS VARCHAR) AS published_version_ids_json,

#   -- Dates (helpful for temporal patterns)
#   CAST(date_posted AS VARCHAR)         AS date_posted,
#   CAST(date_published AS VARCHAR)      AS date_published,
#   CAST(date_published_online AS VARCHAR)      AS date_published_online,
#   CAST(date_issued AS VARCHAR)         AS date_issued,
#   CAST(date_deposited AS VARCHAR)      AS date_deposited,
#   CAST(date_indexed AS VARCHAR)        AS date_indexed,
#   CAST(date_created AS VARCHAR)        AS date_created,
#   CAST(date_registered AS VARCHAR)     AS date_registered,
#   CAST(date_updated AS VARCHAR)        AS date_updated,
#   CAST(publication_year AS VARCHAR)    AS publication_year
# FROM all_backends
# """)

# con.execute("SELECT COUNT(*) AS n FROM server_thin").df()


In [13]:
con.execute(f"""
CREATE OR REPLACE VIEW server_thin AS
SELECT
  CAST(record_id AS VARCHAR)           AS record_id,
  CAST(server_name AS VARCHAR)         AS server_name,
  CAST(backend AS VARCHAR)             AS backend,

  CAST(doi AS VARCHAR)                 AS doi,
  CAST(doi_url AS VARCHAR)             AS doi_url,
  CAST(landing_page_url AS VARCHAR)    AS landing_page_url,
  CAST(type_backend_raw AS VARCHAR)    AS type_backend_raw,
  CAST(subtype_backend_raw AS VARCHAR)    AS subtype_backend_raw,

  CAST(title AS VARCHAR) AS title,
  -- CAST(abstract_text AS VARCHAR)      AS abstract_text,
  CAST(authors_flat AS VARCHAR)      AS authors_flat,
  
  -- Dates (helpful for temporal patterns)
  CAST(publication_year AS VARCHAR)    AS publication_year,
  CAST(date_created AS VARCHAR)        AS date_created,
  -- CAST(date_posted AS VARCHAR)         AS date_posted,
  -- CAST(date_deposited AS VARCHAR)      AS date_deposited,

  -- Relationships (keep these for true version links)
  CAST(relations_json AS VARCHAR)       AS relations_json,
  CAST(version_label AS VARCHAR)       AS version_label,
  CAST(is_version_of AS VARCHAR)       AS is_version_of,      -- keep as text; we’ll interpret later
  CAST(is_preprint_of AS VARCHAR)      AS is_preprint_of,
  CAST(has_preprint AS VARCHAR)      AS has_preprint,
  CAST(has_review AS VARCHAR)      AS has_review,
  CAST(has_published_version AS VARCHAR)      AS has_published_version,
  CAST(published_version_ids_json AS VARCHAR) AS published_version_ids_json,
  CAST(version_of_ids_json AS VARCHAR) AS version_of_ids_json,
  CAST(update_to_json AS VARCHAR)      AS update_to_json,
  CAST(raw_relationships_json AS VARCHAR)       AS raw_relationships_json,
FROM all_backends
""")

con.execute("SELECT COUNT(*) AS n FROM server_thin").df()


,n
0,9797749


## Global Exploration

In [14]:
data = con.execute("SELECT * FROM server_thin").df()
# data.drop_duplicates(subset=['record_id'], keep='first', inplace=False)


In [15]:
# 1. Define the mapping (Old Name -> New Name)
name_mapping = {
    "CERN Document Server": "CERN document server"
}

# 2. Apply the mapping to the server_name column
data["server_name"] = data["server_name"].replace(name_mapping)

# 3. Verify the fix
print(data["server_name"].value_counts().head(20))

server_name
arXiv                                  2920797
SSRN                                   1999168
HAL                                    1320107
Research Square                         870976
RePEc: Research Papers in Economics     702078
bioRxiv                                 306948
AgEcon Search                           189671
ResearchGate                            181231
Zenodo                                  166786
Open Science Framework                  119590
Preprints.org                           115815
EconStor Preprints                      113013
Munich Personal RePEc Archive            99237
medRxiv                                  75743
Authorea Inc.                            65450
PsyArXiv                                 56866
ChemRxiv                                 46475
JMIR Preprints                           37631
Humanities Commons CORE                  30665
eLife                                    29901
Name: count, dtype: int64


In [16]:
data = data.drop_duplicates()
data

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json
0,crossref::10.21467/preprints.48,AIJR Preprints,crossref,10.21467/preprints.48,https://doi.org/10.21467/preprints.48,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,"Bird’s Eye View on the Diagnosis, Treatment, &...","Panchalingala, Sai Bhargavi",2020.0,2020-09-15,None,None,,,,,false,None,None,None,None
1,crossref::10.21467/preprints.43,AIJR Preprints,crossref,10.21467/preprints.43,https://doi.org/10.21467/preprints.43,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,Doxycycline and Minocycline Drugs as a Treatme...,"Mostafa, Mohamed",2020.0,2020-09-15,None,None,,,,,false,None,None,None,None
2,crossref::10.21467/preprints.39,AIJR Preprints,crossref,10.21467/preprints.39,https://doi.org/10.21467/preprints.39,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,A Genetic Perspective of 2019-nCoV in Relation...,"Dasgupta, Rimjhim",2020.0,2020-09-15,None,None,,,,,false,None,None,None,None
3,crossref::10.21467/preprints.38,AIJR Preprints,crossref,10.21467/preprints.38,https://doi.org/10.21467/preprints.38,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,Marine Algae as a Natural Source for Antiviral...,"Musale, Amar S; G., Raja Krishna Kumar; Sapre,...",2020.0,2020-09-17,None,None,,,,,false,None,None,None,None
4,crossref::10.21467/preprints.36,AIJR Preprints,crossref,10.21467/preprints.36,https://doi.org/10.21467/preprints.36,https://preprints.aijr.org/index.php/ap/prepri...,posted-content,preprint,Possible Prevention of COVID 19 by Using Linol...,"Subhash, Venkata; G, Raja Krishna Kumar; Sapre...",2020.0,2020-09-17,None,None,,,,,false,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9797744,openalex::W999325625,viXra,openalex,None,None,https://vixra.org/pdf/1409.0090v1.pdf,preprint,None,Three Objections to Modern Physics,Lubomir Vlcek,2014.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None
9797745,openalex::W999460032,viXra,openalex,None,None,https://vixra.org/abs/1112.0094,preprint,None,Particle Mass Ratios,DT Froedge,2011.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None
9797746,openalex::W99967155,viXra,openalex,None,None,https://vixra.org/pdf/1406.0019v1.pdf,preprint,None,Quantum FFF Theory Proposals for Some Unsolved...,Leo Vuyk,2014.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None
9797747,openalex::W999790414,viXra,openalex,None,None,https://vixra.org/pdf/1306.0105v3.pdf,preprint,None,Investigation of the Formalism of Particle Dyn...,Chi-Yi Chen,2013.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None


In [17]:
data = data.sort_values(by='record_id')

### clean data

#### clean repositories with external doi

In [18]:
# List of servers where you want to keep only records WITHOUT a DOI
servers_no_doi = [
    "IACR Cryptology ePrint Archive",
    "Munich Personal RePEc Archive",
    "Organic Eprints",
    "PhilSci-Archive",
    "Digital Access to Scholarship at Harvard (DASH) (Harvard University)",
    "DSpace@MIT",
    "E-LIS Repository",
    "EconStor Preprints",
    "National Bureau of Economic Research",
    "viXra",
    "CogPrints"
]

# Rows from those servers that have no DOI
mask_no_doi = data['server_name'].isin(servers_no_doi) & data['doi'].isna()

# Rows from all other servers (kept as-is, regardless of DOI)
mask_other = ~data['server_name'].isin(servers_no_doi)

df_filtered = data[mask_no_doi | mask_other]

In [19]:
# Quick sanity check
print("Original shape:", data.shape)
print("Filtered shape:", df_filtered.shape)

# Confirm none of the targeted servers have DOIs remaining
check = df_filtered[df_filtered['server_name'].isin(servers_no_doi)]['doi'].isna().all()
print("All targeted server rows have no DOI:", check)

Original shape: (7984658, 23)
Filtered shape: (7971023, 23)
All targeted server rows have no DOI: True


#### manage type

##### ScienceOpen Preprints

In [20]:
# Mask for ScienceOpen Preprints rows with subtype 'other'
mask_scienceopen_remove = ~(
    (data['server_name'] == 'ScienceOpen Preprints') & 
    (data['subtype_backend_raw'] == 'other')
)

df_filtered = df_filtered[mask_scienceopen_remove]

/tmp/ipykernel_4154/744606357.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_filtered = df_filtered[mask_scienceopen_remove]


In [21]:
check = df_filtered[
    (df_filtered['server_name'] == 'ScienceOpen Preprints') & 
    (df_filtered['subtype_backend_raw'] == 'other')
]
print("Remaining ScienceOpen 'other' rows:", len(check))  # Should be 0

Remaining ScienceOpen 'other' rows: 0


##### OSF

In [22]:
# Mapping of URL pattern -> new server_name
osf_url_to_server = {
    'psyarxiv.com':       'PsyArXiv',
    'eartharxiv.org':     'EarthArXiv',
    'thesiscommons.org':  'Thesis Commons',
    'marxiv.org':         'MarXiv',
    'engrxiv.org':        'engrXiv',
    'arabixiv.org':       'Arabixiv',
    'mindrxiv.org':       'MindRxiv',
    'agrixiv.org':        'AgriRxiv',
    'paleorxiv.org':      'PaleorXiv',
    'ecsarxiv.org':       'ECSarXiv',
}

# Only apply to rows currently assigned to Open Science Framework
osf_mask = df_filtered['server_name'] == 'Open Science Framework'

for url_pattern, new_name in osf_url_to_server.items():
    pattern_mask = osf_mask & df_filtered['landing_page_url'].str.contains(url_pattern, na=False)
    df_filtered.loc[pattern_mask, 'server_name'] = new_name

In [23]:
# See the distribution of OSF-related servers before Mapping
osf_related = ['Open Science Framework'] + list(osf_url_to_server.values())

print(data[data['server_name'].isin(osf_related)]['server_name'].value_counts())

server_name
Open Science Framework    119481
PsyArXiv                   56866
EarthArXiv                  6537
engrXiv                     4929
Thesis Commons              3959
AgriRxiv                     818
MarXiv                       508
Arabixiv                     502
MindRxiv                     335
ECSarXiv                     314
PaleorXiv                    287
Name: count, dtype: int64


In [24]:
# See the distribution of OSF-related servers after renaming
osf_related = ['Open Science Framework'] + list(osf_url_to_server.values())

print(df_filtered[df_filtered['server_name'].isin(osf_related)]['server_name'].value_counts())

server_name
Open Science Framework    117171
PsyArXiv                   58193
EarthArXiv                  6799
engrXiv                     5066
Thesis Commons              4175
AgriRxiv                     858
MarXiv                       666
Arabixiv                     584
MindRxiv                     381
ECSarXiv                     326
PaleorXiv                    317
Name: count, dtype: int64


In [25]:
data=df_filtered.copy()

### Duplicates

In [26]:
dupes = data[data.duplicated(subset=['record_id'], keep=False)]
dupes

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json
275551,crossref::10.1002/essoar.10503117.2,Authorea Inc.,crossref,10.1002/essoar.10503117.2,https://doi.org/10.1002/essoar.10503117.2,https://essopenarchive.org/users/545221/articl...,posted-content,preprint,Carbon Dioxide Removal Estimation Method to Re...,"Fiume, Shannon",2023.0,2023-09-20,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.1002/essoar.10503117.1,,,,false,None,None,None,None
388182,crossref::10.1002/essoar.10503117.2,Earth and Space Science Open Archive,crossref,10.1002/essoar.10503117.2,https://doi.org/10.1002/essoar.10503117.2,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Carbon Dioxide Removal Estimation Method to Re...,"Fiume, Shannon",2023.0,2023-09-20,None,None,,,,,false,None,None,None,None
275529,crossref::10.1002/essoar.10503623.3,Authorea Inc.,crossref,10.1002/essoar.10503623.3,https://doi.org/10.1002/essoar.10503623.3,https://essopenarchive.org/users/574499/articl...,posted-content,preprint,A Bayesian model for quantifying errors in cit...,"Eisma, Jessica A; Schoups, Gerrit; Davids, Jef...",2023.0,2023-04-04,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.1002/essoar.10503623.1,,,,false,None,None,None,None
388127,crossref::10.1002/essoar.10503623.3,Earth and Space Science Open Archive,crossref,10.1002/essoar.10503623.3,https://doi.org/10.1002/essoar.10503623.3,https://essopenarchive.org/users/574499/articl...,posted-content,preprint,A Bayesian model for quantifying errors in cit...,"Eisma, Jessica A; Schoups, Gerrit; Davids, Jef...",2023.0,2023-04-04,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.1002/essoar.10503623.1,,,,false,None,None,None,None
275562,crossref::10.1002/essoar.10503738.2,Authorea Inc.,crossref,10.1002/essoar.10503738.2,https://doi.org/10.1002/essoar.10503738.2,https://essopenarchive.org/users/530351/articl...,posted-content,preprint,Global oceanic iron distribution estimated by ...,"Doi, Toshimasa; Osafune, Satoshi; Masuda, Shuh...",2024.0,2024-07-31,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.1002/essoar.10503738.1,,,,false,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1833910,openalex::W4389470288,HAL,openalex,10.18429/jacow-ipac2023-mopa091,https://doi.org/10.18429/jacow-ipac2023-mopa091,https://hal.science/hal-04328488,preprint,None,The status of the Interaction region design an...,M. Boscolo; Andrea Ciarma; F. Fransesini; S. L...,2023.0,2023-12-08T00:00:00,None,None,None,None,None,None,None,None,None,None,None
1849753,openalex::W4389499288,HAL,openalex,10.18429/jacow-ipac2023-thpa161,https://doi.org/10.18429/jacow-ipac2023-thpa161,https://hal.science/hal-04328489,preprint,None,DYVACS (DYnamic VACuum Simulation) code: gas d...,Suheyla Bilgen; B. Mercier; G. Sattonnay; V. B...,2023.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None
278940,openalex::W4389499288,CERN document server,openalex,10.18429/jacow-ipac2023-thpa161,https://doi.org/10.18429/jacow-ipac2023-thpa161,http://cds.cern.ch/record/2886620,preprint,None,DYVACS (DYnamic VACuum Simulation) code: gas d...,"Suheyla Bilgen; Sattonnay,Gaël; Mercier,Bruno;...",2023.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None
2043292,openalex::W4394007746,Munich Personal RePEc Archive,openalex,None,None,http://www.theses.fr/2022UPASP049/document,preprint,None,Measurement of the N-jettiness variables in th...,J. Mijuskovic,2022.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None


In [27]:
dupes['server_name'].value_counts()

server_name
Authorea Inc.                           3432
Earth and Space Science Open Archive    3430
SSRN                                     572
CERN document server                      15
HAL                                       14
Advance                                    2
Munich Personal RePEc Archive              1
Name: count, dtype: int64

In [28]:
data[data['record_id']=='crossref::10.31124/advance.24454624.v1']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json
15734,crossref::10.31124/advance.24454624.v1,Advance,crossref,10.31124/advance.24454624.v1,https://doi.org/10.31124/advance.24454624.v1,https://advance.sagepub.com/doi/full/10.31124/...,posted-content,preprint,Ancient Social Motive Simulation Hypothesis of...,"Thomas, Frederick",2024.0,2024-02-05,None,None,,,,,false,None,None,None,None
275576,crossref::10.31124/advance.24454624.v1,Authorea Inc.,crossref,10.31124/advance.24454624.v1,https://doi.org/10.31124/advance.24454624.v1,https://advance.sagepub.com/doi/full/10.31124/...,posted-content,preprint,Ancient Social Motive Simulation Hypothesis of...,"Thomas, Frederick",2024.0,2024-02-05,None,None,,,,,false,None,None,None,None


In [29]:
data[data['record_id']=='crossref::10.22541/essoar.170923255.57545328/v1']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json
252630,crossref::10.22541/essoar.170923255.57545328/v1,Authorea Inc.,crossref,10.22541/essoar.170923255.57545328/v1,https://doi.org/10.22541/essoar.170923255.5754...,https://essopenarchive.org/users/528524/articl...,posted-content,preprint,Data Drought in the Humid Tropics: How to Over...,"Frankenberg, Christian; Bar-On, Yinon Moise; Y...",2024.0,2024-02-29,None,None,,,,,false,None,None,None,None
405969,crossref::10.22541/essoar.170923255.57545328/v1,Earth and Space Science Open Archive,crossref,10.22541/essoar.170923255.57545328/v1,https://doi.org/10.22541/essoar.170923255.5754...,https://essopenarchive.org/doi/full/10.22541/e...,posted-content,preprint,Data Drought in the Humid Tropics: How to Over...,"Frankenberg, Christian; Bar-On, Yinon Moise; Y...",2024.0,2024-02-29,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.1029/2024gl108791,,,true,None,None,None,None


#### Resolution

In [30]:
import pandas as pd

# ----------------------------------
# 1) Define server priority
#    Lower value = higher priority (kept first)
# ----------------------------------
server_priority = {
    "Earth and Space Science Open Archive": 1,
    "Advance": 2,
    "Authorea Inc.": 3
}

# Work on a copy to avoid side effects
data = data.copy()

# ----------------------------------
# 2) Add priority column
#    Unknown servers get lowest priority
# ----------------------------------
data['server_priority'] = (
    data['server_name']
    .map(server_priority)
    .fillna(99)
    .astype(int)
)

# ----------------------------------
# 3) Deduplicate STRICTLY on record_id
#    - Sort so preferred server comes first
#    - Keep only the best row per record_id
# ----------------------------------
data_clean = (
    data
    .sort_values(by=['record_id', 'server_priority'])
    .drop_duplicates(subset=['record_id'], keep='first')
    .drop(columns=['server_priority'])
)

# ----------------------------------
# 4) (Optional but recommended) Inspect removals
# ----------------------------------
removed = data.loc[~data.index.isin(data_clean.index)]

print("Removed rows by server_name:")
print(removed['server_name'].value_counts(dropna=False))

print("\nRows before:", len(data))
print("Rows after :", len(data_clean))
print("Rows removed:", len(removed))

# ----------------------------------
# data_clean is the final deduplicated dataframe
# ----------------------------------


Removed rows by server_name:
server_name
Authorea Inc.           3432
SSRN                     286
HAL                        9
CERN document server       6
Name: count, dtype: int64

Rows before: 7970014
Rows after : 7966281
Rows removed: 3733


In [31]:
removed

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,server_priority
275551,crossref::10.1002/essoar.10503117.2,Authorea Inc.,crossref,10.1002/essoar.10503117.2,https://doi.org/10.1002/essoar.10503117.2,https://essopenarchive.org/users/545221/articl...,posted-content,preprint,Carbon Dioxide Removal Estimation Method to Re...,"Fiume, Shannon",2023.0,2023-09-20,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.1002/essoar.10503117.1,,,,false,None,None,None,None,3
275529,crossref::10.1002/essoar.10503623.3,Authorea Inc.,crossref,10.1002/essoar.10503623.3,https://doi.org/10.1002/essoar.10503623.3,https://essopenarchive.org/users/574499/articl...,posted-content,preprint,A Bayesian model for quantifying errors in cit...,"Eisma, Jessica A; Schoups, Gerrit; Davids, Jef...",2023.0,2023-04-04,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.1002/essoar.10503623.1,,,,false,None,None,None,None,3
275562,crossref::10.1002/essoar.10503738.2,Authorea Inc.,crossref,10.1002/essoar.10503738.2,https://doi.org/10.1002/essoar.10503738.2,https://essopenarchive.org/users/530351/articl...,posted-content,preprint,Global oceanic iron distribution estimated by ...,"Doi, Toshimasa; Osafune, Satoshi; Masuda, Shuh...",2024.0,2024-07-31,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.1002/essoar.10503738.1,,,,false,None,None,None,None,3
275503,crossref::10.1002/essoar.10505190.2,Authorea Inc.,crossref,10.1002/essoar.10505190.2,https://doi.org/10.1002/essoar.10505190.2,https://essopenarchive.org/users/544442/articl...,posted-content,preprint,Soil profile stratigraphy detected by ground p...,"Wang, Ping; Li, Xinju; Min, Xiangyu; Xu, Shuo",2023.0,2023-02-01,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.1002/essoar.10505190.1,,,,false,None,None,None,None,3
275548,crossref::10.1002/essoar.10505545.4,Authorea Inc.,crossref,10.1002/essoar.10505545.4,https://doi.org/10.1002/essoar.10505545.4,https://essopenarchive.org/users/543175/articl...,posted-content,preprint,Internal vs Forced Variability Metrics for Geo...,"Sane, Aakash; Fox-Kemper, Baylor; Ullman, David",2023.0,2023-08-10,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.1002/essoar.10505545.1,,,,false,None,None,None,None,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1780531,openalex::W4387594595,HAL,openalex,None,None,https://hal.science/hal-04239157,preprint,None,Measurement of the inclusive semileptonic $B$ ...,F. Abudinén; I. Adachi; R. Adak; K. Adamczyk; ...,2021.0,2023-10-13T00:00:00,None,None,None,None,None,None,None,None,None,None,None,99
1849390,openalex::W4389327243,HAL,openalex,10.18429/jacow-ipac2023-mopa025,https://doi.org/10.18429/jacow-ipac2023-mopa025,https://hal.science/hal-04321508,preprint,None,First coaxial HOM coupler prototypes and RF me...,C. Barbagallo; Patricia Duchesne; W. Kaabi; G....,2023.0,2023-12-06T00:00:00,None,None,None,None,None,None,None,None,None,None,None,99
1833910,openalex::W4389470288,HAL,openalex,10.18429/jacow-ipac2023-mopa091,https://doi.org/10.18429/jacow-ipac2023-mopa091,https://hal.science/hal-04328488,preprint,None,The status of the Interaction region design an...,M. Boscolo; Andrea Ciarma; F. Fransesini; S. L...,2023.0,2023-12-08T00:00:00,None,None,None,None,None,None,None,None,None,None,None,99
278940,openalex::W4389499288,CERN document server,openalex,10.18429/jacow-ipac2023-thpa161,https://doi.org/10.18429/jacow-ipac2023-thpa161,http://cds.cern.ch/record/2886620,preprint,None,DYVACS (DYnamic VACuum Simulation) code: gas d...,"Suheyla Bilgen; Sattonnay,Gaël; Mercier,Bruno;...",2023.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None

In [32]:
removed['server_name'].value_counts()

server_name
Authorea Inc.           3432
SSRN                     286
HAL                        9
CERN document server       6
Name: count, dtype: int64

In [33]:
data_clean.count()

record_id                     7966281
server_name                   7966281
backend                       7966281
doi                           6340449
doi_url                       6340449
landing_page_url              7886835
type_backend_raw              7964986
subtype_backend_raw           5460294
title                         7966272
authors_flat                  7945664
publication_year              7963124
date_created                  7965379
relations_json                4093744
version_label                 2974512
is_version_of                 6274822
is_preprint_of                6274822
has_preprint                  6274822
has_review                    6274822
has_published_version         6274822
published_version_ids_json          0
version_of_ids_json                 0
update_to_json                   8915
raw_relationships_json        3522057
dtype: int64

### Clean columns

In [34]:
data_clean["title"] = (
    data_clean["title"]
    .astype(str)
    .str.strip()
    .replace({"": pd.NA, "None": pd.NA, "null": pd.NA, "nan": pd.NA,
              "N/A": pd.NA, "[]": pd.NA, "{}": pd.NA})
)


In [35]:
data_clean["doi"] = (
    data_clean["doi"]
    .astype(str)
    .str.strip()
    .replace({"": pd.NA, "None": pd.NA, "null": pd.NA, "nan": pd.NA,
              "N/A": pd.NA, "[]": pd.NA, "{}": pd.NA})
)


In [36]:
data_clean["authors_flat"] = (
    data_clean["authors_flat"]
    .astype(str)
    .str.strip()
    .replace({"": pd.NA, "None": pd.NA, "null": pd.NA, "nan": pd.NA,
              "N/A": pd.NA, "[]": pd.NA, "{}": pd.NA})
)


In [37]:
data_clean["landing_page_url"] = (
    data_clean["landing_page_url"]
    .astype(str)
    .str.strip()
    .replace({"": pd.NA, "None": pd.NA, "null": pd.NA, "nan": pd.NA,
              "N/A": pd.NA, "[]": pd.NA, "{}": pd.NA})
)


In [38]:
data_clean.shape

(7966281, 23)

In [39]:
data_clean.count()

record_id                     7966281
server_name                   7966281
backend                       7966281
doi                           6340449
doi_url                       6340449
landing_page_url              7886835
type_backend_raw              7964986
subtype_backend_raw           5460294
title                         7964681
authors_flat                  7945651
publication_year              7963124
date_created                  7965379
relations_json                4093744
version_label                 2974512
is_version_of                 6274822
is_preprint_of                6274822
has_preprint                  6274822
has_review                    6274822
has_published_version         6274822
published_version_ids_json          0
version_of_ids_json                 0
update_to_json                   8915
raw_relationships_json        3522057
dtype: int64

### Summary

In [40]:
import pandas as pd
import numpy as np

data = data_clean.copy()

# -------------------------------------------------
# Configuration
# -------------------------------------------------
GROUP_COL = 'server_name'

MISSINGNESS_COLUMNS = [
    'doi',
    'landing_page_url'
]

COMPLETENESS_COLUMNS = [
    'version_label',
    'relations_json',
    'is_version_of',
    'is_preprint_of',
    'has_review',
    'has_preprint',
    'has_published_version',
    'version_of_ids_json',
    'update_to_json',
    'published_version_ids_json',
    'raw_relationships_json'
]

# -------------------------------------------------
# Helper functions
# -------------------------------------------------
def is_missing(series):
    """Strict missingness (NaN / None)"""
    return series.isna()

def is_incomplete(series):
    """
    Missing OR empty content:
    - NaN
    - empty string
    - empty list
    - empty dict
    """
    return (
        series.isna()
        | (series.astype(str).str.strip() == '')
        | (series.astype(str).isin(['[]', '{}']))
    )

# -------------------------------------------------
# Base counts per server
# -------------------------------------------------
base = (
    data
    .groupby(GROUP_COL, dropna=False)
    .size()
    .rename('total_records')
    .to_frame()
)

# -------------------------------------------------
# Missingness metrics
# -------------------------------------------------
for col in MISSINGNESS_COLUMNS:
    missing_count = (
        data.loc[is_missing(data[col])]
        .groupby(GROUP_COL, dropna=False)
        .size()
        .rename(f'{col}_missing_count')
    )

    base = base.join(missing_count, how='left').fillna(0)
    base[f'{col}_missing_count'] = base[f'{col}_missing_count'].astype(int)
    base[f'{col}_missing_percent'] = (
        base[f'{col}_missing_count'] / base['total_records'] * 100
    ).round(2)

# -------------------------------------------------
# Completeness metrics
# -------------------------------------------------
for col in COMPLETENESS_COLUMNS:
    incomplete_count = (
        data.loc[is_incomplete(data[col])]
        .groupby(GROUP_COL, dropna=False)
        .size()
        .rename(f'{col}_incomplete_count')
    )

    base = base.join(incomplete_count, how='left').fillna(0)
    base[f'{col}_incomplete_count'] = base[f'{col}_incomplete_count'].astype(int)
    base[f'{col}_incomplete_percent'] = (
        base[f'{col}_incomplete_count'] / base['total_records'] * 100
    ).round(2)

# -------------------------------------------------
# Final table
# -------------------------------------------------
summary = base.sort_values('total_records', ascending=False)

# summary


In [41]:
summary.head(60)

,total_records,doi_missing_count,doi_missing_percent,landing_page_url_missing_count,landing_page_url_missing_percent,version_label_incomplete_count,version_label_incomplete_percent,relations_json_incomplete_count,relations_json_incomplete_percent,is_version_of_incomplete_count,is_version_of_incomplete_percent,is_preprint_of_incomplete_count,is_preprint_of_incomplete_percent,has_review_incomplete_count,has_review_incomplete_percent,has_preprint_incomplete_count,has_preprint_incomplete_percent,has_published_version_incomplete_count,has_published_version_incomplete_percent,version_of_ids_json_incomplete_count,version_of_ids_json_incomplete_percent,update_to_json_incomplete_count,update_to_json_incomplete_percent,published_version_ids_json_incomplete_count,published_version_ids_json_incomplete_percent,raw_relationships_json_incomplete_count,raw_relationships_json_incomplete_percent
server_name,,,,,,,,,,,,,,,,,,,,,,,,,,,
arXiv,2920797,0,0.00,0,0.00,0,0.00,1638047,56.08,2920797,100.00,1638164,56.09,2920797,100.00,2920797,100.00,0,0.00,2920797,100.0,2920797,100.00,2920797,100.0,0,0.00
SSRN,1259256,0,0.00,0,0.00,1259236,100.00,1258886,99.97,1259253,100.00,1259001,99.98,1259243,100.00,1259165,99.99,0,0.00,1259256,100.0,1259236,100.00,1259256,100.0,1259256,100.00
HAL,1056415,1024805,97.01,103,0.01,1056415,100.00,1056415,100.00,1056415,100.00,1056415,100.00,1056415,100.00,1056415,100.00,1056415,100.00,1056415,100.0,1056415,100.00,1056415,100.0,1056415,100.00
Research Square,450818,0,0.00,0,0.00,450813,100.00,283964,62.99,435544,96.61,295396,65.52,450818,100.00,450818,100.00,0,0.00,450818,100.0,450813,100.00,450818,100.0,450818,100.00
RePEc: Research Papers in Economics,389398,364045,93.49,39749,10.21,389398,100.00,389398,100.00,389398,100.00,389398,100.00,389398,100.00,389398,100.00,389398,100.00,389398,100.0,389398,100.00,389398,100.0,389398,100.00
bioRxiv,306948,0,0.00,0,0.00,306937,100.00,196496,64.02,306947,100.00,196497,64.02,306948,100.00,306948,100.00,0,0.00,306948,100.0,306937,100.00,306948,100.0,306948,100.00
AgEcon Search,188173,0,0.00,0,0.00,188173,100.00,188173,100.00,188173,100.00,188173,100.00,188173,100.00,188173,100.00,0,0.00,188173,100.0,188173,100.00,188173,100.0,0,0.00
ResearchGate,181231,0,0.00,0,0.00,174949,96.53,174741,96.42,174741,96.42,181231,100.00,181231,100.00,181231,100.00,0,0.00,181231,100.0,181231,100.00,181231,100.0,0,0.00
Zenodo,166786,0,0.00,0,0.00,128664,77.14,179,0.11,163848,98.24,67331,40.37,166706,99.95,166786,100.00,0,0.00,166786,100.0,166786,100.00,166786,100.0,0,0.00


In [42]:
summary.tail(52)

,total_records,doi_missing_count,doi_missing_percent,landing_page_url_missing_count,landing_page_url_missing_percent,version_label_incomplete_count,version_label_incomplete_percent,relations_json_incomplete_count,relations_json_incomplete_percent,is_version_of_incomplete_count,is_version_of_incomplete_percent,is_preprint_of_incomplete_count,is_preprint_of_incomplete_percent,has_review_incomplete_count,has_review_incomplete_percent,has_preprint_incomplete_count,has_preprint_incomplete_percent,has_published_version_incomplete_count,has_published_version_incomplete_percent,version_of_ids_json_incomplete_count,version_of_ids_json_incomplete_percent,update_to_json_incomplete_count,update_to_json_incomplete_percent,published_version_ids_json_incomplete_count,published_version_ids_json_incomplete_percent,raw_relationships_json_incomplete_count,raw_relationships_json_incomplete_percent
server_name,,,,,,,,,,,,,,,,,,,,,,,,,,,
APSA Preprints,1470,0,0.00,0,0.0,1470,100.00,991,67.41,1103,75.03,1275,86.73,1470,100.00,1470,100.00,0,0.0,1470,100.0,1470,100.00,1470,100.0,1470,100.00
PREPRINTS.RU,1415,0,0.00,0,0.0,1415,100.00,1412,99.79,1415,100.00,1413,99.86,1415,100.00,1415,100.00,0,0.0,1415,100.0,1415,100.00,1415,100.0,1415,100.00
Keldysh Institute Preprints,1258,0,0.00,0,0.0,1258,100.00,1221,97.06,1258,100.00,1258,100.00,1258,100.00,1258,100.00,0,0.0,1258,100.0,1258,100.00,1258,100.0,1258,100.00
searchRxiv,1234,0,0.00,0,0.0,1234,100.00,1234,100.00,1234,100.00,1234,100.00,1234,100.00,1234,100.00,0,0.0,1234,100.0,1234,100.00,1234,100.0,1234,100.00
HRB Open Research,1012,0,0.00,0,0.0,651,64.33,101,9.98,867,85.67,1012,100.00,132,13.04,1012,100.00,0,0.0,1012,100.0,651,64.33,1012,100.0,1012,100.00
Jxiv,902,0,0.00,0,0.0,902,100.00,0,0.00,902,100.00,902,100.00,902,100.00,902,100.00,902,100.0,902,100.0,902,100.00,902,100.0,902,100.00
ARPHA Preprints,890,0,0.00,0,0.0,890,100.00,578,64.94,890,100.00,578,64.94,890,100.00,890,100.00,0,0.0,890,100.0,890,100.00,890,100.0,890,100.00
MetaArXiv,880,0,0.00,0,0.0,880,100.00,681,77.39,813,92.39,742,84.32,880,100.00,880,100.00,0,0.0,880,100.0,880,100.00,880,100.0,880,100.00
SportRxiv,878,0,0.00,0,0.0,878,100.00,794,90.43,877,99.89,795,90.55,878,100.00,878,100.00,0,0.0,878,100.0,878,100.00,878,100.0,878,100.00


In [43]:
data_clean

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json
399051,crossref::10.1002/essoar.10500000.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10500000.1,https://doi.org/10.1002/essoar.10500000.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Long-term monitoring of land surface phenologi...,"Tsutsumida, Narumasa",2018.0,2019-11-13,None,None,,,,,false,None,None,None,None
399052,crossref::10.1002/essoar.10500002.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10500002.1,https://doi.org/10.1002/essoar.10500002.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Impact of spatial scale for phenological indic...,"Tsutsumida, Narumasa; Kaduk, Jörg",2018.0,2019-11-13,None,None,,,,,false,None,None,None,None
399047,crossref::10.1002/essoar.10500004.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10500004.1,https://doi.org/10.1002/essoar.10500004.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Observations of Low Latitude Red Aurora in Mex...,"Gonzalez-Esparza, J. Americo; Cuevas-Cardona, ...",2018.0,2019-11-13,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.1029/2018sw001995,,,true,None,None,None,None
399073,crossref::10.1002/essoar.10500007.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10500007.1,https://doi.org/10.1002/essoar.10500007.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Pipeline oil fire detection with MODIS active ...,"Ogungbuyi, Michael Gbenga; Martinez, Peter; Ec...",2018.0,2019-11-13,None,None,,,,,false,None,None,None,None
399068,crossref::10.1002/essoar.10500009.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10500009.1,https://doi.org/10.1002/essoar.10500009.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Land Product Validation of MODIS Derived FPAR ...,"Sharp, Iain; Sanchez-Azofeifa, Arturo; Musilek...",2018.0,2019-12-03,None,None,,,,,false,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2598815,openalex::W999921877,RePEc: Research Papers in Economics,openalex,<NA>,None,https://steconomice.uoradea.ro/anale/volume/20...,preprint,None,IDE sous l'influence du degré de l'intégration...,Simona-Gabriela Serbu Masca,2008.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None
974376,openalex::W999947037,HAL,openalex,<NA>,None,https://hal.science/hal-01922484,preprint,None,Building realistic potential patients queries ...,Lorraine Goeuriot; Wendy W. Chapman; Gareth Jf...,2014.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None
2505522,openalex::W999974616,RePEc: Research Papers in Economics,openalex,<NA>,None,https://EconPapers.repec.org/RePEc:cde:cdewps:19,preprint,None,Debt Financing with Limited Liability and Quan...,Krishnendu Ghosh Dastidar,1994.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None
973276,openalex::W999989114,HAL,openalex,<NA>,None,https://hal.science/hal-01268467,preprint,None,Sustainable orchards' redesign: at the crossro...,Servane Penvern; Sylvaine Simon; Stéphane Bell...,2012.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None


## Manage hierarchy

### check those who have are version

In [44]:
pattern = "ew version"

mask = data_clean['version_label'].str.contains(pattern, regex=False, na=False)
result = data_clean[mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json
2080826,crossref::10.12688/aasopenres.12825.2,Open Research Africa,crossref,10.12688/aasopenres.12825.2,https://doi.org/10.12688/aasopenres.12825.2,https://aasopenresearch.org/articles/1-1/v2,journal-article,None,Prevalence of chloroquine and antifolate drug ...,"Abugri, James; Ansah, Felix; Asante, Kwaku P.;...",2018.0,2018-12-03,"{""has-review"": [{""asserted-by"": ""subject"", ""id...",New version,,,,10.21956/aasopenres.13998.r26676,false,None,None,"[{""DOI"": ""10.12688/aasopenres.12825.1"", ""label...",None
2080871,crossref::10.12688/aasopenres.12832.2,Open Research Africa,crossref,10.12688/aasopenres.12832.2,https://doi.org/10.12688/aasopenres.12832.2,https://aasopenresearch.org/articles/1-3/v2,journal-article,None,The Collaborative African Genomics Network (CA...,"Mboowa, Gerald; Mwesigwa, Savannah; Katagirya,...",2018.0,2018-06-21,"{""has-review"": [{""asserted-by"": ""subject"", ""id...",New version,,,,10.21956/aasopenres.13951.r26487,false,None,None,"[{""DOI"": ""10.12688/aasopenres.12832.1"", ""label...",None
2080890,crossref::10.12688/aasopenres.12837.2,Open Research Africa,crossref,10.12688/aasopenres.12837.2,https://doi.org/10.12688/aasopenres.12837.2,https://aasopenresearch.org/articles/1-12/v2,journal-article,None,Microbiological assessment of sachet water “pu...,"Mosi, Lydia; Adadey, Samuel Mawuli; Sowah, San...",2019.0,2019-01-24,"{""has-review"": [{""asserted-by"": ""subject"", ""id...",New version,,,,10.21956/aasopenres.14017.r26741;10.21956/aaso...,false,None,None,"[{""DOI"": ""10.12688/aasopenres.12837.1"", ""label...",None
2080987,crossref::10.12688/aasopenres.12841.2,Open Research Africa,crossref,10.12688/aasopenres.12841.2,https://doi.org/10.12688/aasopenres.12841.2,https://aasopenresearch.org/articles/1-2/v2,journal-article,None,Ethical and scientific considerations on the e...,"Elliott, Alison M.; Roestenberg, Meta; Wajja, ...",2018.0,2018-08-06,"{""has-review"": [{""asserted-by"": ""subject"", ""id...",New version,,,,10.21956/aasopenres.13967.r26567,false,None,None,"[{""DOI"": ""10.12688/aasopenres.12841.1"", ""label...",None
2080978,crossref::10.12688/aasopenres.12844.2,Open Research Africa,crossref,10.12688/aasopenres.12844.2,https://doi.org/10.12688/aasopenres.12844.2,https://aasopenresearch.org/articles/1-13/v2,journal-article,None,Model framework for governance of genomic rese...,"Yakubu, Aminu; Tindana, Paulina; Matimba, Alic...",2018.0,2018-12-12,None,New version,,,,,false,None,None,"[{""DOI"": ""10.12688/aasopenres.12844.1"", ""label...",None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
530523,crossref::10.3410/f1000research.1-6.v2,F1000Research,crossref,10.3410/f1000research.1-6.v2,https://doi.org/10.3410/f1000research.1-6.v2,http://f1000research.com/articles/1-6/v2,journal-article,None,Sirenomelia in a Cameroonian woman: a case rep...,"Morfaw, Frederick LI; Nana, Philip N",2012.0,2015-06-25,None,New version,,,,,false,None,None,"[{""DOI"": ""10.3410/f1000research.1-6.v1"", ""labe...",None
6332369,datacite::10.5281/zenodo.16753537,Zenodo,datacite,10.5281/zenodo.16753537,https://doi.org/10.5281/zenodo.16753537,https://zenodo.org/doi/10.5281/zenodo.16753537,Preprint,,Temperature-driven tunability of a vanadium di...,"Francesco, Scotognella",2025.0,2025-08-06,"[{""relatedIdentifier"": ""10.5281/zenodo.1743480...",2 [New version of the manuscript uploaded on A...,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""cern.zenodo"", ""typ..."
6368163,datacite::10.5281/zenodo.17434801,Zenodo,datacite,10.5281/zenodo.17434801,https://doi.org/10.5281/zenodo.17434801,https://zenodo.org/doi/10.5281/zenodo.17434801,Pre

In [45]:
result['server_name'].value_counts()

server_name
F1000Research             5651
Wellcome Open Research    1299
Open Research Europe       698
HRB Open Research          361
Gates Open Research        324
Open Research Africa       100
AMRC Open Research          30
MNI Open Research            6
Zenodo                       4
VeriXiv                      2
Name: count, dtype: int64

In [46]:
df_version = data_clean.copy()

# Normalize version_label once
vl = df_version["version_label"].astype(str).str.strip().str.lower()

# CHILD rules
child_mask = (
    # mask_remain &
    vl.isin(["new version", 'new version; retraction'])
)
df_version.loc[child_mask, "records_hierarchy"] = "version"

# correction rules
correction_mask = (
    # mask_remain &
    vl.isin(["correction"])
)
df_version.loc[correction_mask, "records_hierarchy"] = "correction"

print(df_version["records_hierarchy"].value_counts(dropna=False))

records_hierarchy
NaN           7957454
version          8473
correction        354
Name: count, dtype: int64


In [47]:
df_label = df_version[df_version['records_hierarchy'].notna()]
df_label['server_name'].value_counts()

server_name
F1000Research             5651
Wellcome Open Research    1299
Open Research Europe       698
HRB Open Research          361
eLife                      354
Gates Open Research        324
Open Research Africa       100
AMRC Open Research          30
MNI Open Research            6
VeriXiv                      2
Zenodo                       2
Name: count, dtype: int64

In [48]:
pattern = "is-version-of"

mask = df_version['relations_json'].str.contains(pattern, regex=False, na=False)
result = df_version[mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy
388127,crossref::10.1002/essoar.10503623.3,Earth and Space Science Open Archive,crossref,10.1002/essoar.10503623.3,https://doi.org/10.1002/essoar.10503623.3,https://essopenarchive.org/users/574499/articl...,posted-content,preprint,A Bayesian model for quantifying errors in cit...,"Eisma, Jessica A; Schoups, Gerrit; Davids, Jef...",2023.0,2023-04-04,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.1002/essoar.10503623.1,,,,false,None,None,None,None,NaN
388156,crossref::10.1002/essoar.10503738.2,Earth and Space Science Open Archive,crossref,10.1002/essoar.10503738.2,https://doi.org/10.1002/essoar.10503738.2,https://essopenarchive.org/users/530351/articl...,posted-content,preprint,Global oceanic iron distribution estimated by ...,"Doi, Toshimasa; Osafune, Satoshi; Masuda, Shuh...",2024.0,2024-07-31,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.1002/essoar.10503738.1,,,,false,None,None,None,None,NaN
388108,crossref::10.1002/essoar.10505190.2,Earth and Space Science Open Archive,crossref,10.1002/essoar.10505190.2,https://doi.org/10.1002/essoar.10505190.2,https://essopenarchive.org/users/544442/articl...,posted-content,preprint,Soil profile stratigraphy detected by ground p...,"Wang, Ping; Li, Xinju; Min, Xiangyu; Xu, Shuo",2023.0,2023-02-01,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.1002/essoar.10505190.1,,,,false,None,None,None,None,NaN
388140,crossref::10.1002/essoar.10505545.4,Earth and Space Science Open Archive,crossref,10.1002/essoar.10505545.4,https://doi.org/10.1002/essoar.10505545.4,https://essopenarchive.org/users/543175/articl...,posted-content,preprint,Internal vs Forced Variability Metrics for Geo...,"Sane, Aakash; Fox-Kemper, Baylor; Ullman, David",2023.0,2023-08-10,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.1002/essoar.10505545.1,,,,false,None,None,None,None,NaN
388171,crossref::10.1002/essoar.10505617.2,Earth and Space Science Open Archive,crossref,10.1002/essoar.10505617.2,https://doi.org/10.1002/essoar.10505617.2,https://essopenarchive.org/users/545576/articl...,posted-content,preprint,COVID-19 and Social Vulnerabilities in Virgini...,"Patel, Parthay; Patel, Bhaumik",2025.0,2025-10-23,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",None,10.1002/essoar.10505617.1,,,,false,None,None,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9676752,crossref::10.7554/elife.99997.3,eLife,crossref,10.7554/elife.99997.3,https://doi.org/10.7554/elife.99997.3,https://elifesciences.org/articles/99997,journal-article,None,Tripartite organization of brain state dynamic...,"Liu, Lanfang; Jiang, Jiahao; Li, Hehui; Ding, ...",2025.0,2025-01-21,"{""has-preprint"": [{""asserted-by"": ""subject"", ""...",None,10.7554/elife.99997.1;10.7554/elife.99997.2,,10.1101/2024.06.13.598625,10.7554/elife.99997.3.sa3;10.7554/elife.99997....,false,None,None,None,None,NaN
9678717,crossref::10.7554/elife.99999,eLife,crossref,10.7554/elife.99999,https://doi.org/10.7554/elife.99999,https://elifesciences.org/articles/99999,journal-article,None,Glia-mediated gut–brain cytokine signaling cou...,"Malita, Alina; Skakkebaek, Anne H; Kubrak, Olg...",2025.0,2024-09-18,"{""has-preprint"": [{""asserted-by"": ""subject"", ""...",None,10.7554/elife.99999.1;10.7554/elife.99999.2,,10.1101/2024.06.25.600726,,false,None,None,None,None,NaN
9683347,crossref::10.7554/elife.99999.1,eLife,crossref,10.7554/elife.99999.1,https://doi.org/10.7554/elife.99999.1,https://elifesciences.org/reviewed-preprints/9...,posted-content,preprint,Glia-mediated gut-brain cytokine signaling cou...

In [49]:
result['server_name'].value_counts()

server_name
Research Square                         15274
eLife                                   13877
Preprints.org                           13462
ChemRxiv                                12927
PsyArXiv                                 5525
Open Science Framework                   2288
Authorea Inc.                            1931
Qeios                                    1837
CrimRxiv                                 1516
TechRxiv                                 1313
SocArXiv                                 1021
Cambridge Open Engage                     996
APSA Preprints                            367
EdArXiv                                   131
Earth and Space Science Open Archive      125
MetaArXiv                                  67
Advance                                    42
Thesis Commons                             42
PaleorXiv                                  17
EGUsphere                                  13
Encyclopedia                               11
LIS Scholarship Archiv

In [50]:
df_version = df_version.copy()

# Normalize relations_json once
relations_norm = df_version["relations_json"].astype(str).str.lower()

# CHILD rules
child_mask = (
    # mask_remain &
    relations_norm.str.contains("is-version-of", na=False)
)
df_version.loc[child_mask, "records_hierarchy"] = "version"

print(df_version["records_hierarchy"].value_counts(dropna=False))

records_hierarchy
NaN           7884626
version         81301
correction        354
Name: count, dtype: int64


In [51]:
df_label = df_version[df_version['records_hierarchy'].notna()]
df_label['server_name'].value_counts()

server_name
Research Square                         15274
eLife                                   14231
Preprints.org                           13462
ChemRxiv                                12927
F1000Research                            5651
PsyArXiv                                 5525
Open Science Framework                   2288
Authorea Inc.                            1931
Qeios                                    1837
CrimRxiv                                 1516
TechRxiv                                 1313
Wellcome Open Research                   1299
SocArXiv                                 1021
Cambridge Open Engage                     996
Open Research Europe                      698
APSA Preprints                            367
HRB Open Research                         361
Gates Open Research                       324
EdArXiv                                   131
Earth and Space Science Open Archive      125
Open Research Africa                      100
MetaArXiv             

### check those who are review

In [52]:
pattern = "is-review-of"

mask = df_version['relations_json'].str.contains(pattern, regex=False, na=False)
result = df_version[mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy
333250,crossref::10.21428/cb6ab371.2b7a7a53,CrimRxiv,crossref,10.21428/cb6ab371.2b7a7a53,https://doi.org/10.21428/cb6ab371.2b7a7a53,https://www.crimrxiv.com/pub/zkdkie05,peer-review,None,"Review of ""Relative Openness in Peer Review: C...","Jacques, Scott",2025.0,2025-01-01,"{""is-review-of"": [{""asserted-by"": ""subject"", ""...",None,,,,,false,None,None,None,None,NaN
332004,crossref::10.21428/cb6ab371.33c9e126,CrimRxiv,crossref,10.21428/cb6ab371.33c9e126,https://doi.org/10.21428/cb6ab371.33c9e126,https://www.crimrxiv.com/pub/s4okkv64,peer-review,None,"Review 1 of ""Treating Criminal Justice-Involve...","Windle, James",2022.0,2022-05-24,"{""is-review-of"": [{""asserted-by"": ""subject"", ""...",None,,,,,false,None,None,None,None,NaN
332005,crossref::10.21428/cb6ab371.c4d02d0a,CrimRxiv,crossref,10.21428/cb6ab371.c4d02d0a,https://doi.org/10.21428/cb6ab371.c4d02d0a,https://www.crimrxiv.com/pub/frlqbegb,peer-review,None,"Review 2 of ""Doing death work: A mixed method ...","Brondolo, Elizabeth",2022.0,2022-05-24,"{""is-review-of"": [{""asserted-by"": ""subject"", ""...",None,,,,,false,None,None,None,None,NaN
332320,crossref::10.21428/cb6ab371.e2488a45,CrimRxiv,crossref,10.21428/cb6ab371.e2488a45,https://doi.org/10.21428/cb6ab371.e2488a45,https://www.crimrxiv.com/pub/iumal68d,peer-review,None,"Review of ""Ranking the openness of criminology...","Burt, Callie",2022.0,2022-10-10,"{""is-review-of"": [{""asserted-by"": ""subject"", ""...",None,,,,,false,None,None,None,None,NaN
333245,crossref::10.21428/cb6ab371.e42226cb,CrimRxiv,crossref,10.21428/cb6ab371.e42226cb,https://doi.org/10.21428/cb6ab371.e42226cb,https://www.crimrxiv.com/pub/8qb8nsfj,peer-review,None,"Review of ""A plea for open access to qualitati...","Dickinson, Timothy",2024.0,2024-12-18,"{""is-review-of"": [{""asserted-by"": ""subject"", ""...",None,,,,,false,None,None,None,None,NaN
332311,crossref::10.21428/cb6ab371.f514a7ba,CrimRxiv,crossref,10.21428/cb6ab371.f514a7ba,https://doi.org/10.21428/cb6ab371.f514a7ba,https://www.crimrxiv.com/pub/19r39l3l,peer-review,None,"Review of ""Ranking the openness of criminology...","Wheeler, Andrew",2022.0,2022-10-06,"{""is-review-of"": [{""asserted-by"": ""subject"", ""...",None,,,,,false,None,None,None,None,NaN
375884,crossref::10.5194/egusphere-2025-1476-ac1,EGUsphere,crossref,10.5194/egusphere-2025-1476-ac1,https://doi.org/10.5194/egusphere-2025-1476-ac1,https://egusphere.copernicus.org/preprints/202...,posted-content,other,Reply on RC1,"de Jong, Jasper",2025.0,2025-06-17,"{""is-review-of"": [{""asserted-by"": ""object"", ""i...",None,,,,,false,None,None,None,None,NaN
375888,crossref::10.5194/egusphere-2025-1476-ac2,EGUsphere,crossref,10.5194/egusphere-2025-1476-ac2,https://doi.org/10.5194/egusphere-2025-1476-ac2,https://egusphere.copernicus.org/preprints/202...,posted-content,other,Reply on RC2,"de Jong, Jasper",2025.0,2025-06-17,"{""is-review-of"": [{""asserted-by"": ""object"", ""i...",None,,,,,false,None,None,None,None,NaN
375892,crossref::10.5194/egusphere-2025-1476-ac3,EGUsphere,crossref,10.5194/egusphere-2025-1476-ac3,https://doi.org/10.5194/egusphere-2025-1476-ac3,https://egusphere.copernicus.org/preprints/202...,posted-content,other,Reply on CEC1,"de Jong, Jasper",2025.0,2025-06-17,"{""is-review-of"": [{""asserted-by"": ""object"", ""i...",None,,,,,false,None,None,None,None,NaN
375897,crossref::10.5194/egusphere-2025-1476-ac4,EGUsphere,crossref,10.5194/egusphere-2025-1476-ac4,https://doi.org/10.5194/egusphere-2025-1476-ac4,https://egusphere.copernicus.org/preprints/202...,posted-content,other,Reply on CEC1,"de Jong, Jasper",2025.0,2025-06-22,"{""is-review-of"": [{""asserted-b

In [53]:
result['server_name'].value_counts()

server_name
EGUsphere    21
CrimRxiv      6
Name: count, dtype: int64

In [54]:
df_version_review = df_version.copy()

# Normalize relations_json once
relations_norm = df_version_review["relations_json"].astype(str).str.lower()

# Review signals
review_mask = (
    # mask_remain &
    (
        # relations_norm.str.contains("has-review", na=False) |
        relations_norm.str.contains("is-review-of", na=False) 
    )
)

df_version_review.loc[review_mask, "records_hierarchy"] = "review"
print(df_version_review["records_hierarchy"].value_counts(dropna=False))

records_hierarchy
NaN           7884599
version         81301
correction        354
review             27
Name: count, dtype: int64


In [55]:
df_label = df_version_review[df_version_review['records_hierarchy'].notna()]
df_label['server_name'].value_counts()

server_name
Research Square                         15274
eLife                                   14231
Preprints.org                           13462
ChemRxiv                                12927
F1000Research                            5651
PsyArXiv                                 5525
Open Science Framework                   2288
Authorea Inc.                            1931
Qeios                                    1837
CrimRxiv                                 1522
TechRxiv                                 1313
Wellcome Open Research                   1299
SocArXiv                                 1021
Cambridge Open Engage                     996
Open Research Europe                      698
APSA Preprints                            367
HRB Open Research                         361
Gates Open Research                       324
EdArXiv                                   131
Earth and Space Science Open Archive      125
Open Research Africa                      100
MetaArXiv             

### use regular expression to extract version and others

In [56]:
import re
import numpy as np
import pandas as pd

df = df_version_review.copy()

# ------------------------------------------------------------
# 0) Ensure target column exists
# ------------------------------------------------------------

# normalize strings once (safe)
df["landing_norm"] = df.get("landing_page_url", "").astype(str).str.lower()
df["doi_norm"] = df.get("doi", "").astype(str).str.lower()

# ------------------------------------------------------------
# 1) Your regex (good: avoids /v284p and .v50i7)
# ------------------------------------------------------------
VERSION_RX = re.compile(
    r'(?P<token>('
    r'/v\d+(?![a-z0-9])|'       # /v1 but not /v284p
    r'\.v\d+(?![a-z0-9])|'      # .v1 but not .v50i7
    r'_v\d+(?![a-z0-9])|'       # _v1
    r'-v\d+(?![a-z0-9])|'       # -v2 but not -v8018x
    r'-rc\d+(?![a-z0-9])|'      # -rc1
    r'-cc\d+(?![a-z0-9])|'      # -cc1
    r'-supplement|'             # -supplement
    r'\.sa\d+(?![a-z0-9])'      # .sa10
    r'))',
    re.IGNORECASE
)

# helpers: classify token family + get numeric v
VNUM_RX = re.compile(r'(?:^|[._/\-])v(\d+)$', re.IGNORECASE)  # matches v1 at end of token
def extract_token(text: str):
    if not isinstance(text, str) or text.strip() == "" or text.lower() == "nan":
        return None
    m = VERSION_RX.search(text)
    return m.group("token") if m else None

def token_kind(token: str):
    if not isinstance(token, str):
        return None
    t = token.lower()
    if t.startswith(("/v", ".v", "_v", "-v")):
        return "explicit_version"
    if t.startswith("-rc"):
        return "rc"
    if t.startswith("-cc"):
        return "cc"
    if t == "-supplement":
        return "supplement"
    if t.startswith(".sa"):
        return "supplementary_asset"
    return "other"

def token_vnum(token: str):
    if not isinstance(token, str):
        return np.nan
    t = token.lower()
    # extract trailing v number for explicit version tokens only
    m = re.search(r'v(\d+)$', t)
    return float(m.group(1)) if m else np.nan

# ------------------------------------------------------------
# 2) Apply extraction: landing_page_url first, then doi
# ------------------------------------------------------------
remain = df["records_hierarchy"].fillna("other").eq("other")

# --- landing page tokens ---
df.loc[remain, "version_token_lp"] = df.loc[remain, "landing_norm"].map(extract_token)
df.loc[remain, "token_kind_lp"] = df.loc[remain, "version_token_lp"].map(token_kind)
df.loc[remain, "vnum_lp"] = df.loc[remain, "version_token_lp"].map(token_vnum)

# Label rules (landing_page_url)
# - Only use explicit vN tokens for parent/child
# - parent if v0 or v1, child if v>=2
lp_parent = remain & (df["token_kind_lp"] == "explicit_version") & (df["vnum_lp"].isin([0.0, 1.0]))
lp_child  = remain & (df["token_kind_lp"] == "explicit_version") & (df["vnum_lp"] >= 2)

df.loc[lp_parent, "records_hierarchy"] = "parent"
df.loc[lp_child,  "records_hierarchy"] = "version"

# Everything else from landing tokens (rc/cc/supplement/.sa) keep as other
# If you prefer to label them separately, do it here:
df.loc[remain & df["token_kind_lp"].isin(["rc","cc"]), "records_hierarchy"] = "comment"
df.loc[remain & df["token_kind_lp"].isin(["supplement","supplementary_asset"]), "records_hierarchy"] = "part_of"

# --- doi tokens (only for still-unlabeled) ---
remain2 = df["records_hierarchy"].fillna("other").eq("other")

df.loc[remain2, "version_token_doi"] = df.loc[remain2, "doi_norm"].map(extract_token)
df.loc[remain2, "token_kind_doi"] = df.loc[remain2, "version_token_doi"].map(token_kind)
df.loc[remain2, "vnum_doi"] = df.loc[remain2, "version_token_doi"].map(token_vnum)

doi_parent = remain2 & (df["token_kind_doi"] == "explicit_version") & (df["vnum_doi"].isin([0.0, 1.0]))
doi_child  = remain2 & (df["token_kind_doi"] == "explicit_version") & (df["vnum_doi"] >= 2)

df.loc[doi_parent, "records_hierarchy"] = "parent"
df.loc[doi_child,  "records_hierarchy"] = "version"

# ------------------------------------------------------------
# 3) Optional: quick diagnostics
# ------------------------------------------------------------
# How many got labeled via landing vs doi?
print(df["records_hierarchy"].value_counts(dropna=False))
# print(df.loc[df["records_hierarchy"].isin(["parent","child"]),
#              ["server_name","landing_page_url","doi","version_token_lp","version_token_doi","records_hierarchy"]].head(20))


records_hierarchy
NaN           7061392
parent         780499
version        119314
part_of          4677
correction        354
review             27
comment            18
Name: count, dtype: int64


In [57]:
df_label = df[df['records_hierarchy'].notna()]
df_label['server_name'].value_counts()

server_name
Research Square                         450818
Preprints.org                           115815
Open Science Framework                   80649
Authorea Inc.                            55926
PsyArXiv                                 50234
                                         ...  
Prepublicaciones OpenCiencia                 1
CERN document server                         1
PhilSci-Archive                              1
Munich Personal RePEc Archive                1
National Bureau of Economic Research         1
Name: count, Length: 76, dtype: int64

In [58]:
df = df.copy()

# Ensure column exists
# if "records_hierarchy" not in df.columns:
#     df["records_hierarchy"] = "other"

# Normalize relations_json once
relations_norm = df["landing_page_url"].astype(str).str.lower()

# Only touch rows not already parent/child
mask_remain = df["records_hierarchy"].isin(["other", None, np.nan])

# Part signals
part_mask = (
    mask_remain &
    (
        relations_norm.str.contains("#fig", na=False) |
        relations_norm.str.contains("#digest", na=False) |
        relations_norm.str.contains("#supp", na=False) |
        relations_norm.str.contains("#video", na=False) |
        relations_norm.str.contains("#media", na=False) |
        relations_norm.str.contains("#tbl", na=False) |
        relations_norm.str.contains("#table", na=False) |
        relations_norm.str.contains("#sd", na=False) |
        relations_norm.str.contains("#transrepform", na=False) |
        relations_norm.str.contains("/figures#", na=False) |
        relations_norm.str.contains("#box", na=False) |
        relations_norm.str.contains("#app", na=False) |
        relations_norm.str.contains("#resp", na=False) |
        relations_norm.str.contains("#a", na=False) |
        relations_norm.str.contains("#b", na=False) |
        relations_norm.str.contains("#c", na=False) |
        relations_norm.str.contains("#s", na=False) |
        relations_norm.str.contains("#atbl", na=False) |
        relations_norm.str.contains("#sa", na=False) |
        relations_norm.str.contains("#none", na=False) |
        relations_norm.str.contains("#desfig", na=False) |
        relations_norm.str.contains("#keyresource", na=False) |
        relations_norm.str.contains("#abstract", na=False) 
    )
)
df.loc[part_mask, "records_hierarchy"] = "part_of"
#
# Review signals
comment_mask = (
    mask_remain &
    (
        relations_norm.str.contains("#ac", na=False) |
        relations_norm.str.contains("#rc", na=False) |
        relations_norm.str.contains("#cc", na=False) |
        relations_norm.str.contains("#ec", na=False) |
        relations_norm.str.contains("/peer-reviews", na=False) |
        relations_norm.str.contains("#decision-letter", na=False) |
        relations_norm.str.contains("#cec", na=False) 
    )
)

df.loc[comment_mask, "records_hierarchy"] = "comment"


print(df["records_hierarchy"].value_counts(dropna=False))

records_hierarchy
NaN           7059923
parent         780499
version        119314
part_of          5922
correction        354
comment           242
review             27
Name: count, dtype: int64


In [59]:
df_label = df[df['records_hierarchy'].notna()]
df_label['server_name'].value_counts()

server_name
Research Square                         450818
Preprints.org                           115815
Open Science Framework                   80649
Authorea Inc.                            55926
PsyArXiv                                 50234
                                         ...  
Prepublicaciones OpenCiencia                 1
CERN document server                         1
PhilSci-Archive                              1
Munich Personal RePEc Archive                1
National Bureau of Economic Research         1
Name: count, Length: 78, dtype: int64

### identify publish versions

In [60]:
pattern = "has-preprint"

mask = df['relations_json'].str.contains(pattern, regex=False, na=False)
result = df[mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
396330,crossref::10.1002/essoar.10502762.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10502762.1,https://doi.org/10.1002/essoar.10502762.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Extant mat world analog microbes synchronize m...,"Biddanda, Bopaiah A; Weinke, Anthony D",2020.0,2020-04-28,"{""has-preprint"": [{""asserted-by"": ""subject"", ""...",None,,10.1002/essoar.10502762.1,10.1002/essoar.10502762.1,,true,None,None,None,None,NaN,https://essopenarchive.org/doi/full/10.1002/es...,10.1002/essoar.10502762.1,None,None,NaN,None,None,NaN
396597,crossref::10.1002/essoar.10503174.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10503174.1,https://doi.org/10.1002/essoar.10503174.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Constraints on the upper mantle structure bene...,"Kendall, Elodie; Ferreira, Ana M. G.; Chang, S...",2020.0,2020-05-23,"{""has-preprint"": [{""asserted-by"": ""subject"", ""...",None,,,10.1002/essoar.10503631.1,,false,None,None,None,None,NaN,https://essopenarchive.org/doi/full/10.1002/es...,10.1002/essoar.10503174.1,None,None,NaN,None,None,NaN
397462,crossref::10.1002/essoar.10503991.2,Earth and Space Science Open Archive,crossref,10.1002/essoar.10503991.2,https://doi.org/10.1002/essoar.10503991.2,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Sinking CO2 in supercritical reservoirs,"Vilarrasa, Victor; Parisio, Francesco",2020.0,2020-08-20,"{""has-preprint"": [{""asserted-by"": ""subject"", ""...",None,,10.1002/essoar.10503991.2,10.1002/essoar.10503991.3;10.1002/essoar.10503...,,true,None,None,None,None,NaN,https://essopenarchive.org/doi/full/10.1002/es...,10.1002/essoar.10503991.2,None,None,NaN,None,None,NaN
396122,crossref::10.1002/essoar.10504386.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10504386.1,https://doi.org/10.1002/essoar.10504386.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Assessing Level of Awareness about Water Gover...,"Zakaria, Abdul-Razak; Matsui, Kenichi",2020.0,2020-10-09,"{""has-preprint"": [{""asserted-by"": ""object"", ""i...",None,,10.1002/essoar.10504386.1,10.1002/essoar.10504386.1,,true,None,None,None,None,NaN,https://essopenarchive.org/doi/full/10.1002/es...,10.1002/essoar.10504386.1,None,None,NaN,None,None,NaN
395900,crossref::10.1002/essoar.10504797.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10504797.1,https://doi.org/10.1002/essoar.10504797.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Variability of the Atlantic Ocean North Equato...,"Dimoune, Djoirka Minto; Hernandez, Fabrice; Ar...",2020.0,2020-11-18,"{""has-preprint"": [{""asserted-by"": ""subject"", ""...",None,,10.1002/essoar.10504797.1,10.1002/essoar.10504797.1,,true,None,None,None,None,NaN,https://essopenarchive.org/doi/full/10.1002/es...,10.1002/essoar.10504797.1,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9679039,crossref::10.7554/elife.99989.3,eLife,crossref,10.7554/elife.99989.3,https://doi.org/10.7554/elife.99989.3,https://elifesciences.org/articles/99989,journal-article,None,Tonotopy is not preserved in a descending stag...,"Gu, Miaoqing; Liang, Shanshan; Zhu, Jiahui; Li...",2025.0,2025-10-14,"{""has-preprint"": [{""asserted-by"": ""subject"", ""...",None,10.7554/elife.99989.1;10.7554/elife.99989.2,,10.1101/2024.05.25.595883,10.7554/elife.9

In [61]:
result['server_name'].value_counts()

server_name
eLife                                   12710
SSRN                                       91
Earth and Space Science Open Archive       31
Gates Open Research                        28
Qeios                                      24
UCL Open Environment                        3
EGUsphere                                   1
Name: count, dtype: int64

In [62]:
df = df.copy()

# Normalize relations_json once
relations_norm = df["relations_json"].astype(str).str.lower()

# Review signals
review_mask = (
    (
        relations_norm.str.contains("has-preprint", na=False) 
    )
)

df.loc[review_mask, "records_hierarchy"] = "publish_version"
print(df["records_hierarchy"].value_counts(dropna=False))

records_hierarchy
NaN                7052494
parent              780463
version             113891
publish_version      12888
part_of               5922
correction             354
comment                242
review                  27
Name: count, dtype: int64


In [63]:
df_label = df[df['records_hierarchy'].notna()]
df_label['server_name'].value_counts()

server_name
Research Square                         450818
Preprints.org                           115815
Open Science Framework                   80649
Authorea Inc.                            55926
PsyArXiv                                 50234
                                         ...  
Prepublicaciones OpenCiencia                 1
CERN document server                         1
PhilSci-Archive                              1
Munich Personal RePEc Archive                1
National Bureau of Economic Research         1
Name: count, Length: 79, dtype: int64

#### remaining

In [64]:
df_remain = df[df['records_hierarchy'].isna()]
df_remain['server_name'].value_counts()

server_name
arXiv                                  2920797
SSRN                                   1259165
HAL                                    1056207
RePEc: Research Papers in Economics     388814
bioRxiv                                 306948
                                        ...   
AMRC Open Research                           3
NewAddictionsX                               2
Gates Open Research                          1
MNI Open Research                            1
Open Research Africa                         1
Name: count, Length: 99, dtype: int64

### Label server who assingn unique doi for all version as parent

In [65]:
import pandas as pd
import numpy as np
import re

# ============================================================
# 1) Read the Google Sheet (rules tab) as CSV
#    - Works for public / shared-to-anyone sheets
# ============================================================

SHEET_ID = "10_7FdcpZjntqFsEHIii7bAM72uF__of_iUohSD5w8w4"
GID = "1230415212"  # the gid you shared for the 'rules' tab

rules_csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}"
rules = pd.read_csv(rules_csv_url)

# Normalize column names (strip spaces/newlines)
rules.columns = (
    rules.columns.astype(str)
    .str.replace(r"\s+", " ", regex=True)  # collapse whitespace/newlines
    .str.strip()
)

# ============================================================
# 2) Extract servers where "versionning - doi" == "unique"
# ============================================================

# These are the column names seen in your screenshot:
SERVER_COL = "Field_server_name"
VERS_COL = "versionning - doi"  # after normalization it should match like this

# Safety: show close matches if something is off
if SERVER_COL not in rules.columns or VERS_COL not in rules.columns:
    print("Columns available:", rules.columns.tolist())
    raise KeyError(f"Expected columns not found. Need: {SERVER_COL!r} and {VERS_COL!r}")

# Build "unique versioning" server list
unique_servers = (
    rules.loc[
        rules[VERS_COL].astype(str).str.strip().str.lower().eq("unique"),
        SERVER_COL
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

print(f"Unique-versioning servers found: {len(unique_servers)}")
print(unique_servers[:30])  # preview

# ============================================================
# 3) Apply mapping to your dataframe df -> records_hierarchy = "parent"
# ============================================================

df = df.copy()

# Normalize server_name in df for matching
df["server_name_norm"] = df["server_name"].astype(str).str.strip()

# Normalize unique list too (strip)
unique_servers_norm = [s.strip() for s in unique_servers]

# Create / overwrite records_hierarchy
# Default: keep existing if present; else "other"
# df["records_hierarchy"] = df["records_hierarchy"] if "records_hierarchy" in df.columns else "other"

mask_unique = df["server_name_norm"].isin(unique_servers_norm)
# df.loc[mask_unique, "records_hierarchy"] = "parent"
# only fill missing values (uncomment instead of overwrite)
df.loc[mask_unique & df["records_hierarchy"].isna(), "records_hierarchy"] = "parent"

# Optional: drop helper col
df.drop(columns=["server_name_norm"], inplace=True)

# Quick check
print(df["records_hierarchy"].value_counts(dropna=False).head(10))
# print(df.loc[df["server_name"].isin(unique_servers_norm), ["server_name","records_hierarchy"]].drop_duplicates().head(20))


Unique-versioning servers found: 60
['AgEcon Search', 'AIJR Preprints', 'ARPHA Preprints', 'ART-Dok', 'arXiv', 'Bepress Legal Repository', 'bioRxiv', 'CERN document server', 'CogPrints', 'Covid-19 Preprints', 'CrimRxiv', 'CrossAsia-Repository', 'Digital Access to Scholarship at Harvard (DASH) (Harvard University)', 'DSpace@MIT', 'E-LIS Repository', 'EasyChair preprint', 'EcoEvoRxiv', 'EconStor Preprints', 'Electron Colloquium Comput Complex', 'ELPUB (Universitat Wuppertal)', 'EmeRI', 'EnerarXiv', 'HAL', 'HANS Publication PrePrints', 'Humanities Commons CORE', 'IACR Cryptology ePrint Archive', 'IndiaRxiv', 'JMIR Preprints', 'Keldysh Institute Preprints', 'LatArXiv']
records_hierarchy
parent             7715975
NaN                 116982
version             113891
publish_version      12888
part_of               5922
correction             354
comment                242
review                  27
Name: count, dtype: int64


#### remaining

In [66]:
df_remain = df[df['records_hierarchy'].isna()]
df_remain['server_name'].value_counts()

server_name
ChemRxiv                                26513
Earth and Space Science Open Archive    12875
EGUsphere                               10411
eLife                                    8366
TechRxiv                                 8320
PsyArXiv                                 7959
EarthArXiv                               6687
PeerJ Preprints                          6446
Authorea Inc.                            6092
engrXiv                                  5062
SocArXiv                                 3072
INA-Rxiv                                 2830
Cambridge Open Engage                    2089
AfricArXiv                               1730
Thesis Commons                           1723
Advance                                  1717
APSA Preprints                           1101
Jxiv                                      902
AgriRxiv                                  827
EdArXiv                                   550
Arabixiv                                  291
Law Archive           

### manage server-by-server

#### eLife

In [67]:
df_remain[df_remain['server_name']=='eLife']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
9658075,crossref::10.7554/elife,eLife,crossref,10.7554/elife,https://doi.org/10.7554/elife,https://elifesciences.org/,journal,None,eLife,<NA>,None,2017-07-25,None,None,,,,,false,None,None,None,None,NaN,https://elifesciences.org/,10.7554/elife,None,None,NaN,None,None,NaN
9658086,crossref::10.7554/elife.00003,eLife,crossref,10.7554/elife.00003,https://doi.org/10.7554/elife.00003,https://elifesciences.org/articles/00003,journal-article,None,A novel role for lipid droplets in the organis...,"Anand, Preetha; Cermelli, Silvia; Li, Zhihuan;...",2012.0,2012-11-13,"{""has-review"": [{""asserted-by"": ""object"", ""id""...",None,,,,10.7554/elife.00003.013;10.3410/f.717964024.79...,false,None,None,None,None,NaN,https://elifesciences.org/articles/00003,10.7554/elife.00003,None,None,NaN,None,None,NaN
9658085,crossref::10.7554/elife.00005,eLife,crossref,10.7554/elife.00005,https://doi.org/10.7554/elife.00005,https://elifesciences.org/articles/00005,journal-article,None,Molecular architecture of human polycomb repre...,"Ciferri, Claudio; Lander, Gabriel C; Maiolica,...",2012.0,2012-10-30,"{""has-review"": [{""asserted-by"": ""object"", ""id""...",None,,,,10.7554/elife.00005.021;10.7554/elife.00005.02...,false,None,None,None,None,NaN,https://elifesciences.org/articles/00005,10.7554/elife.00005,None,None,NaN,None,None,NaN
9658084,crossref::10.7554/elife.00007,eLife,crossref,10.7554/elife.00007,https://doi.org/10.7554/elife.00007,https://elifesciences.org/articles/00007,journal-article,None,Herbivory-induced volatiles function as defens...,"Schuman, Meredith C; Barthel, Kathleen; Baldwi...",2012.0,2012-10-15,"{""has-review"": [{""asserted-by"": ""object"", ""id""...",None,,,,10.7554/elife.00007.022;10.7554/elife.00007.021,false,None,None,None,None,NaN,https://elifesciences.org/articles/00007,10.7554/elife.00007,None,None,NaN,None,None,NaN
9658087,crossref::10.7554/elife.00011,eLife,crossref,10.7554/elife.00011,https://doi.org/10.7554/elife.00011,https://elifesciences.org/articles/00011,journal-article,None,Nascent-Seq reveals novel features of mouse ci...,"Menet, Jerome S; Rodriguez, Joseph; Abruzzi, K...",2012.0,2012-11-13,"{""has-review"": [{""asserted-by"": ""object"", ""id""...",None,,,,10.7554/elife.00011.026;10.7554/elife.00011.025,false,None,None,None,None,NaN,https://elifesciences.org/articles/00011,10.7554/elife.00011,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9674638,crossref::10.7554/elife.99560,eLife,crossref,10.7554/elife.99560,https://doi.org/10.7554/elife.99560,https://elifesciences.org/articles/99560,journal-article,None,Paying attention,"Poth, Christian H",2024.0,2024-06-10,None,None,,,,,false,None,None,None,None,NaN,https://elifesciences.org/articles/99560,10.7554/elife.99560,None,None,NaN,None,None,NaN
9674777,crossref::10.7554/elife.99765,eLife,crossref,10.7554/elife.99765,https://doi.org/10.7554/elife.99765,https://elifesciences.org/articles/99765,journal-article,None,Redox takes control,"Plaza-Menacho, Iván",2024.0,2024-06-20,None,None,,,,,false,None,None,None,None,NaN,https://elifesciences.org/articles/99765,10.7554/elife.99765,None,None,NaN,None,None,NaN
9675585,crossref::10.7554/elife.99770,eLife,crossref,10.7554/elife.99770,https://doi.org/10.7554/elife.99770,https://elifesciences.org/articles/99770,journal-article,None,Exploring protein structural ensembles: Integr...,"Belyaeva, Julia; Elgeti, Matthias",2024.0,2024-09-16,None,None,,,,,false,None,None,None,

In [68]:
pattern = "10.7554/elife.12523"

mask = df['doi'].str.contains(pattern, regex=False, na=False)
result = df[mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
9671715,crossref::10.7554/elife.12523,eLife,crossref,10.7554/elife.12523,https://doi.org/10.7554/elife.12523,https://elifesciences.org/articles/12523,journal-article,None,Hold your breath!,"Lust, Katharina; Wittbrodt, Joachim",2015.0,2015-12-11,None,None,,,,,false,None,None,None,None,NaN,https://elifesciences.org/articles/12523,10.7554/elife.12523,None,None,NaN,None,None,NaN


In [69]:
pattern = "10.7554/elife.100000"

mask = df['doi'].str.contains(pattern, regex=False, na=False)
result = df[mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
9678129,crossref::10.7554/elife.100000,eLife,crossref,10.7554/elife.100000,https://doi.org/10.7554/elife.100000,https://elifesciences.org/articles/100000,journal-article,None,Group identification drives brain integration ...,"Xie, Enhui; Zha, Shuyi; Xu, Yiyang; Li, Xianchun",2025.0,2024-10-14,"{""has-preprint"": [{""asserted-by"": ""subject"", ""...",None,10.7554/elife.100000.1;10.7554/elife.100000.2;...,,10.1101/2024.06.03.597223,,false,None,None,None,None,publish_version,https://elifesciences.org/articles/100000,10.7554/elife.100000,NaN,NaN,NaN,NaN,NaN,NaN
9683629,crossref::10.7554/elife.100000.1,eLife,crossref,10.7554/elife.100000.1,https://doi.org/10.7554/elife.100000.1,https://elifesciences.org/reviewed-preprints/1...,posted-content,preprint,Group identification drives brain integration ...,"Xie, Enhui; Zha, Shuyi; Xu, Yiyang; Li, Xianchun",2024.0,2024-10-14,"{""has-review"": [{""asserted-by"": ""object"", ""id""...",None,10.1101/2024.06.03.597223;10.7554/elife.100000...,,,10.7554/elife.100000.1.sa3;10.7554/elife.10000...,false,None,None,None,None,version,https://elifesciences.org/reviewed-preprints/1...,10.7554/elife.100000.1,NaN,NaN,NaN,NaN,NaN,NaN
9685245,crossref::10.7554/elife.100000.2,eLife,crossref,10.7554/elife.100000.2,https://doi.org/10.7554/elife.100000.2,https://elifesciences.org/reviewed-preprints/1...,posted-content,preprint,Group identification drives brain integration ...,"Xie, Enhui; Zha, Shuyi; Xu, Yiyang; Li, Xianchun",2025.0,2025-04-01,"{""has-review"": [{""asserted-by"": ""object"", ""id""...",None,10.1101/2024.06.03.597223;10.7554/elife.100000...,,,10.7554/elife.100000.2.sa0;10.7554/elife.10000...,false,None,None,None,None,version,https://elifesciences.org/reviewed-preprints/1...,10.7554/elife.100000.2,NaN,NaN,NaN,NaN,NaN,NaN
9685885,crossref::10.7554/elife.100000.3,eLife,crossref,10.7554/elife.100000.3,https://doi.org/10.7554/elife.100000.3,https://elifesciences.org/reviewed-preprints/1...,posted-content,preprint,Group identification drives brain integration ...,"Xie, Enhui; Zha, Shuyi; Xu, Yiyang; Li, Xianchun",2025.0,2025-06-04,"{""has-review"": [{""asserted-by"": ""object"", ""id""...",None,10.1101/2024.06.03.597223;10.7554/elife.100000...,,,10.7554/elife.100000.3.sa2;10.7554/elife.10000...,false,None,None,None,None,version,https://elifesciences.org/reviewed-preprints/1...,10.7554/elife.100000.3,NaN,NaN,NaN,NaN,NaN,NaN
9678124,crossref::10.7554/elife.100000.4,eLife,crossref,10.7554/elife.100000.4,https://doi.org/10.7554/elife.100000.4,https://elifesciences.org/articles/100000,journal-article,None,Group identification drives brain integration ...,"Xie, Enhui; Zha, Shuyi; Xu, Yiyang; Li, Xianchun",2025.0,2025-06-24,"{""has-preprint"": [{""asserted-by"": ""subject"", ""...",None,10.7554/elife.100000.1;10.7554/elife.100000.2;...,,10.1101/2024.06.03.597223,10.7554/elife.100000.4.sa1;10.7554/elife.10000...,false,None,None,None,None,publish_version,https://elifesciences.org/articles/100000,10.7554/elife.100000.4,NaN,NaN,NaN,NaN,NaN,NaN


In [70]:
pattern = ".1"

mask = df[df['server_name']=='eLife']['doi'].str.contains(pattern, regex=False, na=False)
result = df[df['server_name']=='eLife'][mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
9678129,crossref::10.7554/elife.100000,eLife,crossref,10.7554/elife.100000,https://doi.org/10.7554/elife.100000,https://elifesciences.org/articles/100000,journal-article,None,Group identification drives brain integration ...,"Xie, Enhui; Zha, Shuyi; Xu, Yiyang; Li, Xianchun",2025.0,2024-10-14,"{""has-preprint"": [{""asserted-by"": ""subject"", ""...",None,10.7554/elife.100000.1;10.7554/elife.100000.2;...,,10.1101/2024.06.03.597223,,false,None,None,None,None,publish_version,https://elifesciences.org/articles/100000,10.7554/elife.100000,NaN,NaN,NaN,NaN,NaN,NaN
9683629,crossref::10.7554/elife.100000.1,eLife,crossref,10.7554/elife.100000.1,https://doi.org/10.7554/elife.100000.1,https://elifesciences.org/reviewed-preprints/1...,posted-content,preprint,Group identification drives brain integration ...,"Xie, Enhui; Zha, Shuyi; Xu, Yiyang; Li, Xianchun",2024.0,2024-10-14,"{""has-review"": [{""asserted-by"": ""object"", ""id""...",None,10.1101/2024.06.03.597223;10.7554/elife.100000...,,,10.7554/elife.100000.1.sa3;10.7554/elife.10000...,false,None,None,None,None,version,https://elifesciences.org/reviewed-preprints/1...,10.7554/elife.100000.1,NaN,NaN,NaN,NaN,NaN,NaN
9685245,crossref::10.7554/elife.100000.2,eLife,crossref,10.7554/elife.100000.2,https://doi.org/10.7554/elife.100000.2,https://elifesciences.org/reviewed-preprints/1...,posted-content,preprint,Group identification drives brain integration ...,"Xie, Enhui; Zha, Shuyi; Xu, Yiyang; Li, Xianchun",2025.0,2025-04-01,"{""has-review"": [{""asserted-by"": ""object"", ""id""...",None,10.1101/2024.06.03.597223;10.7554/elife.100000...,,,10.7554/elife.100000.2.sa0;10.7554/elife.10000...,false,None,None,None,None,version,https://elifesciences.org/reviewed-preprints/1...,10.7554/elife.100000.2,NaN,NaN,NaN,NaN,NaN,NaN
9685885,crossref::10.7554/elife.100000.3,eLife,crossref,10.7554/elife.100000.3,https://doi.org/10.7554/elife.100000.3,https://elifesciences.org/reviewed-preprints/1...,posted-content,preprint,Group identification drives brain integration ...,"Xie, Enhui; Zha, Shuyi; Xu, Yiyang; Li, Xianchun",2025.0,2025-06-04,"{""has-review"": [{""asserted-by"": ""object"", ""id""...",None,10.1101/2024.06.03.597223;10.7554/elife.100000...,,,10.7554/elife.100000.3.sa2;10.7554/elife.10000...,false,None,None,None,None,version,https://elifesciences.org/reviewed-preprints/1...,10.7554/elife.100000.3,NaN,NaN,NaN,NaN,NaN,NaN
9678124,crossref::10.7554/elife.100000.4,eLife,crossref,10.7554/elife.100000.4,https://doi.org/10.7554/elife.100000.4,https://elifesciences.org/articles/100000,journal-article,None,Group identification drives brain integration ...,"Xie, Enhui; Zha, Shuyi; Xu, Yiyang; Li, Xianchun",2025.0,2025-06-24,"{""has-preprint"": [{""asserted-by"": ""subject"", ""...",None,10.7554/elife.100000.1;10.7554/elife.100000.2;...,,10.1101/2024.06.03.597223,10.7554/elife.100000.4.sa1;10.7554/elife.10000...,false,None,None,None,None,publish_version,https://elifesciences.org/articles/100000,10.7554/elife.100000.4,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9682910,crossref::10.7554/elife.99990.1,eLife,crossref,10.7554/elife.99990.1,https://doi.org/10.7554/elife.99990.1,https://elifesciences.org/reviewed-preprints/9...,posted-content,preprint,The integrated WF-Haldane (WFH) model of genet...,"Ruan, Yongsen; Wang, Xiaopei; Hou, Mei; Diao, ...",2024.0,2024-07-31,"{""has-review"": [{""asserted-by"": ""object"", ""id""...",None,10.1101/2024.02.19.58108

In [71]:
df[df['server_name']=='eLife']['doi']

9658075            10.7554/elife
9658086      10.7554/elife.00003
9658085      10.7554/elife.00005
9658084      10.7554/elife.00007
9658087      10.7554/elife.00011
                   ...          
9676752    10.7554/elife.99997.3
9678717      10.7554/elife.99999
9683347    10.7554/elife.99999.1
9686114    10.7554/elife.99999.2
9678716    10.7554/elife.99999.3
Name: doi, Length: 29901, dtype: object

In [72]:
import re
import numpy as np
import pandas as pd

df = df.copy()

# Ensure column exists
# if "records_hierarchy" not in df.columns:
#     df["records_hierarchy"] = np.nan

# Work ONLY on remaining rows
remaining = df["records_hierarchy"].isna()

is_elife = df["server_name"].astype(str).str.lower().eq("elife")
doi_l = df["doi"].astype(str).str.lower()

# ---------- Regex patterns ----------

# Base parent: 10.7554/elife.12523
RX_PARENT_BASE = re.compile(    r"^10\.7554/elife\.\d+$",    re.IGNORECASE)

# Explicit parent v1 (NO padding)
RX_PARENT_V1 = re.compile(    r"^10\.7554/elife\.\d+\.1$",    re.IGNORECASE)

# Any dotted numeric suffix (captures padding too)
RX_ANY_SUFFIX = re.compile(r"^10\.7554/elife\.\d+\.(\d+)$", re.IGNORECASE)

# ---------- Apply rules ----------

# Parent: base DOI
mask_parent_base = remaining & is_elife & doi_l.str.match(RX_PARENT_BASE, na=False)
df.loc[mask_parent_base, "records_hierarchy"] = "parent"

# Parent: explicit ".1" ONLY
mask_parent_v1 = remaining & is_elife & doi_l.str.match(RX_PARENT_V1, na=False)
df.loc[mask_parent_v1, "records_hierarchy"] = "parent_duplicate"

# Child: any numeric suffix EXCEPT exact ".1"
suffix = doi_l.str.extract(RX_ANY_SUFFIX)[0]

mask_child = (
    remaining
    & is_elife
    & suffix.notna()
    & (suffix != "1")   # excludes .1 but keeps .001, .002, .2, etc.
)

df.loc[mask_child, "records_hierarchy"] = "version"
print(df["records_hierarchy"].value_counts(dropna=False))

records_hierarchy
parent              7724332
version              113895
NaN                  108617
publish_version       12888
part_of                5922
correction              354
comment                 242
review                   27
parent_duplicate          4
Name: count, dtype: int64


In [73]:
df_remain = df[df['records_hierarchy'].isna()]
df_remain['server_name'].value_counts()

server_name
ChemRxiv                                26513
Earth and Space Science Open Archive    12875
EGUsphere                               10411
TechRxiv                                 8320
PsyArXiv                                 7959
EarthArXiv                               6687
PeerJ Preprints                          6446
Authorea Inc.                            6092
engrXiv                                  5062
SocArXiv                                 3072
INA-Rxiv                                 2830
Cambridge Open Engage                    2089
AfricArXiv                               1730
Thesis Commons                           1723
Advance                                  1717
APSA Preprints                           1101
Jxiv                                      902
AgriRxiv                                  827
EdArXiv                                   550
Arabixiv                                  291
Law Archive                               287
MarXiv                

In [74]:
df_remain[df_remain['server_name']=='eLife']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
9658075,crossref::10.7554/elife,eLife,crossref,10.7554/elife,https://doi.org/10.7554/elife,https://elifesciences.org/,journal,None,eLife,<NA>,None,2017-07-25,None,None,,,,,false,None,None,None,None,NaN,https://elifesciences.org/,10.7554/elife,None,None,NaN,None,None,NaN


In [75]:
# pattern = "10.7554/elife.99997"

# mask = df['doi'].str.contains(pattern, regex=False, na=False)
# result = df[mask]
# result

# df = df[df.records_hiearchy.isin(['publish_version', 'version'])]



#### ChemRxiv

In [76]:
df_remain[df_remain['server_name']=='ChemRxiv']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
289639,crossref::10.26434/chemrxiv-2021-00kkd,ChemRxiv,crossref,10.26434/chemrxiv-2021-00kkd,https://doi.org/10.26434/chemrxiv-2021-00kkd,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Understanding MOF nucleation from solution wit...,"Kollias, Loukas; Rousseau, Roger; Glezakou, Va...",2021.0,2021-12-23,None,None,,,,,false,None,None,None,None,NaN,https://chemrxiv.org/engage/chemrxiv/article-d...,10.26434/chemrxiv-2021-00kkd,None,None,NaN,None,None,NaN
315925,crossref::10.26434/chemrxiv-2021-00rj4,ChemRxiv,crossref,10.26434/chemrxiv-2021-00rj4,https://doi.org/10.26434/chemrxiv-2021-00rj4,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Electrochemical Ozone Generation Using Compact...,"Wood, Georgia; Terrero Rodriguez, Irina; Tully...",2021.0,2022-01-25,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.1149/1945-7111/ac3ff4,,,true,None,None,None,None,NaN,https://chemrxiv.org/engage/chemrxiv/article-d...,10.26434/chemrxiv-2021-00rj4,None,None,NaN,None,None,NaN
315869,crossref::10.26434/chemrxiv-2021-012c7,ChemRxiv,crossref,10.26434/chemrxiv-2021-012c7,https://doi.org/10.26434/chemrxiv-2021-012c7,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Surface modification of carbon dots with tetra...,"Sviridova, Elizaveta; Barras, Alexandre; Plotn...",2021.0,2022-01-25,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.1016/j.msec.2022.112697,,,true,None,None,None,None,NaN,https://chemrxiv.org/engage/chemrxiv/article-d...,10.26434/chemrxiv-2021-012c7,None,None,NaN,None,None,NaN
289634,crossref::10.26434/chemrxiv-2021-01dfq,ChemRxiv,crossref,10.26434/chemrxiv-2021-01dfq,https://doi.org/10.26434/chemrxiv-2021-01dfq,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,On the Role of Solvent in the Formation of Vac...,"Marinova, Veselina; Wood, Geoffrey P. F.; Marz...",2021.0,2021-12-23,None,None,,,,,false,None,None,None,None,NaN,https://chemrxiv.org/engage/chemrxiv/article-d...,10.26434/chemrxiv-2021-01dfq,None,None,NaN,None,None,NaN
315974,crossref::10.26434/chemrxiv-2021-01hrg,ChemRxiv,crossref,10.26434/chemrxiv-2021-01hrg,https://doi.org/10.26434/chemrxiv-2021-01hrg,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Visible Light-driven Metal-free C–H Functional...,"Kersting, Lena; Kuhn, Leah; Anokhin, Maksim; S...",2021.0,2021-12-21,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.1002/cptc.202200109,,,true,None,None,None,None,NaN,https://chemrxiv.org/engage/chemrxiv/article-d...,10.26434/chemrxiv-2021-01hrg,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289489,crossref::10.26434/chemrxiv.5917351,ChemRxiv,crossref,10.26434/chemrxiv.5917351,https://doi.org/10.26434/chemrxiv.5917351,https://chemrxiv.org/articles/CO2_Activation_o...,posted-content,preprint,CO2 Activation on Heterostructures of Bi2O3-Na...,"Nolan, Michael",2018.0,2018-02-23,None,None,,,,,false,None,None,None,None,NaN,https://chemrxiv.org/articles/co2_activation_o...,10.26434/chemrxiv.5917351,None,None,NaN,None,None,NaN
290907,crossref::10.26434/chemrxiv.6483989,ChemRxiv,crossref,10.26434/chemrxiv.6483989,https://doi.org/10.26434/chemrxiv.6483989,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Highly sensitive gating in pH-responsive nanoc...,"Lopez, Luis G.; Nap, Rikkert J.",2018.0,2018-06-12,None,None,,,,,false,None,Non

In [77]:
pattern = "v"

mask = df_remain[df_remain['server_name']=='ChemRxiv']['doi'].str.contains(pattern, regex=False, na=False)
result = df_remain[df_remain['server_name']=='ChemRxiv'][mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
289639,crossref::10.26434/chemrxiv-2021-00kkd,ChemRxiv,crossref,10.26434/chemrxiv-2021-00kkd,https://doi.org/10.26434/chemrxiv-2021-00kkd,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Understanding MOF nucleation from solution wit...,"Kollias, Loukas; Rousseau, Roger; Glezakou, Va...",2021.0,2021-12-23,None,None,,,,,false,None,None,None,None,NaN,https://chemrxiv.org/engage/chemrxiv/article-d...,10.26434/chemrxiv-2021-00kkd,None,None,NaN,None,None,NaN
315925,crossref::10.26434/chemrxiv-2021-00rj4,ChemRxiv,crossref,10.26434/chemrxiv-2021-00rj4,https://doi.org/10.26434/chemrxiv-2021-00rj4,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Electrochemical Ozone Generation Using Compact...,"Wood, Georgia; Terrero Rodriguez, Irina; Tully...",2021.0,2022-01-25,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.1149/1945-7111/ac3ff4,,,true,None,None,None,None,NaN,https://chemrxiv.org/engage/chemrxiv/article-d...,10.26434/chemrxiv-2021-00rj4,None,None,NaN,None,None,NaN
315869,crossref::10.26434/chemrxiv-2021-012c7,ChemRxiv,crossref,10.26434/chemrxiv-2021-012c7,https://doi.org/10.26434/chemrxiv-2021-012c7,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Surface modification of carbon dots with tetra...,"Sviridova, Elizaveta; Barras, Alexandre; Plotn...",2021.0,2022-01-25,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.1016/j.msec.2022.112697,,,true,None,None,None,None,NaN,https://chemrxiv.org/engage/chemrxiv/article-d...,10.26434/chemrxiv-2021-012c7,None,None,NaN,None,None,NaN
289634,crossref::10.26434/chemrxiv-2021-01dfq,ChemRxiv,crossref,10.26434/chemrxiv-2021-01dfq,https://doi.org/10.26434/chemrxiv-2021-01dfq,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,On the Role of Solvent in the Formation of Vac...,"Marinova, Veselina; Wood, Geoffrey P. F.; Marz...",2021.0,2021-12-23,None,None,,,,,false,None,None,None,None,NaN,https://chemrxiv.org/engage/chemrxiv/article-d...,10.26434/chemrxiv-2021-01dfq,None,None,NaN,None,None,NaN
315974,crossref::10.26434/chemrxiv-2021-01hrg,ChemRxiv,crossref,10.26434/chemrxiv-2021-01hrg,https://doi.org/10.26434/chemrxiv-2021-01hrg,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Visible Light-driven Metal-free C–H Functional...,"Kersting, Lena; Kuhn, Leah; Anokhin, Maksim; S...",2021.0,2021-12-21,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.1002/cptc.202200109,,,true,None,None,None,None,NaN,https://chemrxiv.org/engage/chemrxiv/article-d...,10.26434/chemrxiv-2021-01hrg,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289489,crossref::10.26434/chemrxiv.5917351,ChemRxiv,crossref,10.26434/chemrxiv.5917351,https://doi.org/10.26434/chemrxiv.5917351,https://chemrxiv.org/articles/CO2_Activation_o...,posted-content,preprint,CO2 Activation on Heterostructures of Bi2O3-Na...,"Nolan, Michael",2018.0,2018-02-23,None,None,,,,,false,None,None,None,None,NaN,https://chemrxiv.org/articles/co2_activation_o...,10.26434/chemrxiv.5917351,None,None,NaN,None,None,NaN
290907,crossref::10.26434/chemrxiv.6483989,ChemRxiv,crossref,10.26434/chemrxiv.6483989,https://doi.org/10.26434/chemrxiv.6483989,https://chemrxiv.org/engage/chemrxiv/article-d...,posted-content,preprint,Highly sensitive gating in pH-responsive nanoc...,"Lopez, Luis G.; Nap, Rikkert J.",2018.0,2018-06-12,None,None,,,,,false,None,Non

In [78]:
df = df.copy()

# Work only on remaining (not already forced to parent)
mask_remain = df["records_hierarchy"] != "parent"

# Normalize version_label once
server_name = df["server_name"].astype(str).str.strip()#.str.lower()

# CHILD rules
child_mask = (
    mask_remain &
    server_name.isin(["ChemRxiv"])
)
df.loc[child_mask, "records_hierarchy"] = "parent"

print(df["records_hierarchy"].value_counts(dropna=False))

records_hierarchy
parent              7763789
version              100951
NaN                   82104
publish_version       12888
part_of                5922
correction              354
comment                 242
review                   27
parent_duplicate          4
Name: count, dtype: int64


In [79]:
df_remain = df[df['records_hierarchy'].isna()]
df_remain['server_name'].value_counts()

server_name
Earth and Space Science Open Archive    12875
EGUsphere                               10411
TechRxiv                                 8320
PsyArXiv                                 7959
EarthArXiv                               6687
PeerJ Preprints                          6446
Authorea Inc.                            6092
engrXiv                                  5062
SocArXiv                                 3072
INA-Rxiv                                 2830
Cambridge Open Engage                    2089
AfricArXiv                               1730
Thesis Commons                           1723
Advance                                  1717
APSA Preprints                           1101
Jxiv                                      902
AgriRxiv                                  827
EdArXiv                                   550
Arabixiv                                  291
Law Archive                               287
MarXiv                                    212
MetaArXiv             

#### OSF-based servers

In [80]:
df_remain[df_remain['server_name']=='TechRxiv']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
6224433,crossref::10.36227/techrxiv.10002041,TechRxiv,crossref,10.36227/techrxiv.10002041,https://doi.org/10.36227/techrxiv.10002041,https://www.techrxiv.org/articles/Comment_on_C...,posted-content,preprint,Comment on ‘ Comment on ‘Enhancing the securit...,"Ding, Yuan",2019.0,2019-10-18,None,None,,,,,false,None,None,None,None,NaN,https://www.techrxiv.org/articles/comment_on_c...,10.36227/techrxiv.10002041,None,None,NaN,None,None,NaN
6224439,crossref::10.36227/techrxiv.10002782,TechRxiv,crossref,10.36227/techrxiv.10002782,https://doi.org/10.36227/techrxiv.10002782,https://www.techrxiv.org/articles/Novel_Energi...,posted-content,preprint,Novel Energisation Method for Offshore Wind Fa...,"Saborío-Romano, Oscar; Bidadfar, Ali; Sakamuri...",2019.0,2019-10-20,None,None,,,,,false,None,None,None,None,NaN,https://www.techrxiv.org/articles/novel_energi...,10.36227/techrxiv.10002782,None,None,NaN,None,None,NaN
6224434,crossref::10.36227/techrxiv.10005770,TechRxiv,crossref,10.36227/techrxiv.10005770,https://doi.org/10.36227/techrxiv.10005770,https://www.techrxiv.org/articles/Modified_SHE...,posted-content,preprint,Modified SHE for Grid Connection.pdf,"Santra, Subhendu Bikash",2019.0,2019-10-20,None,None,,,,,false,None,None,None,None,NaN,https://www.techrxiv.org/articles/modified_she...,10.36227/techrxiv.10005770,None,None,NaN,None,None,NaN
6224437,crossref::10.36227/techrxiv.10007051,TechRxiv,crossref,10.36227/techrxiv.10007051,https://doi.org/10.36227/techrxiv.10007051,https://www.techrxiv.org/articles/Can_Frequenc...,posted-content,preprint,Can Frequency Diverse Array Prevent Wireless E...,"Ding, Yuan; Narbudowicz, Adam",2019.0,2019-10-21,None,None,,,,,false,None,None,None,None,NaN,https://www.techrxiv.org/articles/can_frequenc...,10.36227/techrxiv.10007051,None,None,NaN,None,None,NaN
6224436,crossref::10.36227/techrxiv.10008968,TechRxiv,crossref,10.36227/techrxiv.10008968,https://doi.org/10.36227/techrxiv.10008968,https://www.techrxiv.org/articles/Online_param...,posted-content,preprint,Online parameter identification of synchronous...,"Alves, Erick; Noeland, Jonas; Marafioti, Gianc...",2019.0,2019-10-21,None,None,,,,,false,None,None,None,None,NaN,https://www.techrxiv.org/articles/online_param...,10.36227/techrxiv.10008968,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6232628,crossref::10.36227/techrxiv.24750039,TechRxiv,crossref,10.36227/techrxiv.24750039,https://doi.org/10.36227/techrxiv.24750039,https://www.techrxiv.org/articles/preprint/Opt...,posted-content,preprint,Optimizing Complex CPQ Software Systems for Qu...,"Alexander, Thomson",2023.0,2023-12-07,None,None,,,,,false,None,None,None,None,NaN,https://www.techrxiv.org/articles/preprint/opt...,10.36227/techrxiv.24750039,None,None,NaN,None,None,NaN
6232629,crossref::10.36227/techrxiv.24751587,TechRxiv,crossref,10.36227/techrxiv.24751587,https://doi.org/10.36227/techrxiv.24751587,https://www.techrxiv.org/articles/preprint/Fre...,posted-content,preprint,Frequency Diverse Array With Discrete Fourier ...,"Wang, Kai; Yu, Zichuan; Jin, Zhiyuan; Zhong, F...",2023.0,2023-12-07,None,None,,,,,false,None,None,None,None,NaN,https://www.techrxiv.org/articles/preprint/fre...,10.36227/techrxiv.24751587,None,None,NaN,None,None,NaN
6232630,crossref::10.36227/techrxiv.24751989,TechRxiv,crossref,10.36227/techrxiv.24751989,https://doi.org/10.36227/techrxiv.24751989,https://www.techrxiv.org/articles/preprint/Qua...,posted-content,preprint,Qu

In [81]:
pattern = "10.31234/osf.io/zypk9"

mask = df['doi'].str.contains(pattern, regex=False, na=False)
result = df[mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
2351069,crossref::10.31234/osf.io/zypk9,PsyArXiv,crossref,10.31234/osf.io/zypk9,https://doi.org/10.31234/osf.io/zypk9,https://osf.io/zypk9,posted-content,preprint,The Opposition of Surprisal and Semantic Simil...,"Sun, Kun; Nixon, Jessie S.",2020.0,2020-12-10,None,None,,,,,false,None,None,None,None,NaN,https://osf.io/zypk9,10.31234/osf.io/zypk9,None,None,NaN,None,None,NaN
2357828,crossref::10.31234/osf.io/zypk9_v1,PsyArXiv,crossref,10.31234/osf.io/zypk9_v1,https://doi.org/10.31234/osf.io/zypk9_v1,https://osf.io/zypk9_v1,posted-content,preprint,WITHDRAWN,<NA>,2020.0,2025-05-19,None,None,,,,,false,None,None,None,None,parent,https://osf.io/zypk9_v1,10.31234/osf.io/zypk9_v1,_v1,explicit_version,1.0,NaN,NaN,NaN


In [82]:
pattern = "10.36227/techrxiv.24750039"

mask = df['doi'].str.contains(pattern, regex=False, na=False)
result = df[mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
6232628,crossref::10.36227/techrxiv.24750039,TechRxiv,crossref,10.36227/techrxiv.24750039,https://doi.org/10.36227/techrxiv.24750039,https://www.techrxiv.org/articles/preprint/Opt...,posted-content,preprint,Optimizing Complex CPQ Software Systems for Qu...,"Alexander, Thomson",2023.0,2023-12-07,None,None,,,,,false,None,None,None,None,NaN,https://www.techrxiv.org/articles/preprint/opt...,10.36227/techrxiv.24750039,None,None,NaN,None,None,NaN
6239847,crossref::10.36227/techrxiv.24750039.v1,TechRxiv,crossref,10.36227/techrxiv.24750039.v1,https://doi.org/10.36227/techrxiv.24750039.v1,https://www.techrxiv.org/doi/full/10.36227/tec...,posted-content,preprint,Optimizing Complex CPQ Software Systems for Qu...,"Alexander, Thomson",2023.0,2023-12-07,None,None,,,,,false,None,None,None,None,parent,https://www.techrxiv.org/doi/full/10.36227/tec...,10.36227/techrxiv.24750039.v1,.v1,explicit_version,1.0,NaN,NaN,NaN


In [83]:
import re
import numpy as np

# Ensure column exists
if 'records_hierarchy' not in df.columns:
    df['records_hierarchy'] = pd.NA

# Work only on remaining (unlabeled)
remain = df['records_hierarchy'].isna()

# ----------------------------
# 1) OSF-based servers: parent if DOI is exactly osf.io/<5chars>
#    Examples:
#      10.31234/osf.io/zypk9  -> parent
#      10.31234/osf.io/zypk9_v1  -> NOT parent by this rule
# ----------------------------
OSF_SERVERS = {
    'PsyArXiv',
    'Thesis Commons',
    'SocArXiv',
    'OSF Preprints',
    'Open Science Framework',
    'MindRxiv',
    'MetaArXiv',
    'SportRxiv',
    'LawArXiv',
    'EarthArXiv',
    'EngrXiv',
    'MarXiv',
    'INA-Rxiv',
    'AfricArXiv',
    'AgriXiv',
    'Arabixiv',
    # add/remove servers you want to include
}

# matches "...osf.io/ABCDE" at end of DOI string
OSF_PARENT_RX = re.compile(r'osf\.io/[a-z0-9]{5}$', re.IGNORECASE)

mask_osf_parent = (
    remain
    # & df['server_name'].isin(OSF_SERVERS)
    & df['doi'].astype(str).str.lower().str.contains('osf.io/', na=False)
    & df['doi'].astype(str).str.match(r'.*osf\.io/[a-z0-9]{5}$', na=False)
)

df.loc[mask_osf_parent, 'records_hierarchy'] = 'parent'


# ----------------------------
# 2) TechRxiv: parent if DOI is exactly "techrxiv.<digits>"
#    Examples:
#      10.36227/techrxiv.24750039 -> parent
#      10.36227/techrxiv.24750039.v1 -> NOT parent by this rule
# ----------------------------
TECHRXIV_PARENT_RX = re.compile(r'techrxiv\.\d+$', re.IGNORECASE)

mask_techrxiv_parent = (
    df['records_hierarchy'].isna()
    & (df['server_name'] == 'TechRxiv')
    & df['doi'].astype(str).str.match(r'.*techrxiv\.\d+$', na=False)
)

df.loc[mask_techrxiv_parent, 'records_hierarchy'] = 'parent'


# ----------------------------
# Optional: if you want to label obvious "child" for OSF when suffix exists
# (only do this if you are confident the suffix means versioning)
# Example: osf.io/xxxxx_v2 or osf.io/xxxxx-v2 or osf.io/xxxxx.v2
# ----------------------------
OSF_CHILD_RX = re.compile(r'osf\.io/[a-z0-9]{5}([._-]?v\d+)$', re.IGNORECASE)

mask_osf_child = (
    df['records_hierarchy'].isna()
    & df['server_name'].isin(OSF_SERVERS)
    & df['doi'].astype(str).str.match(r'.*osf\.io/[a-z0-9]{5}([._-]?v\d+)$', na=False)
)

df.loc[mask_osf_child, 'records_hierarchy'] = 'version'


# ----------------------------
# Optional: TechRxiv child rule if you see explicit versioning later
# Example: techrxiv.24750039.v2 or techrxiv.24750039-v2
# ----------------------------
mask_techrxiv_child = (
    df['records_hierarchy'].isna()
    & (df['server_name'] == 'TechRxiv')
    & df['doi'].astype(str).str.match(r'.*techrxiv\.\d+([._-]?v\d+)$', na=False)
)

df.loc[mask_techrxiv_child, 'records_hierarchy'] = 'version'


# Quick check
print(df['records_hierarchy'].value_counts(dropna=False))


records_hierarchy
parent              7794595
version              100951
NaN                   51298
publish_version       12888
part_of                5922
correction              354
comment                 242
review                   27
parent_duplicate          4
Name: count, dtype: int64


In [84]:
df_remain = df[df['records_hierarchy'].isna()]
df_remain['server_name'].value_counts()

server_name
Earth and Space Science Open Archive    12875
EGUsphere                               10411
PeerJ Preprints                          6446
Authorea Inc.                            6092
EarthArXiv                               4693
engrXiv                                  2733
Cambridge Open Engage                    2089
Advance                                  1717
AfricArXiv                               1689
APSA Preprints                           1101
Jxiv                                      902
AgriRxiv                                  380
PoolText                                   79
Oroboros Instruments                       70
F1000Research                              14
AMRC Open Research                          3
Open Research Africa                        1
Gates Open Research                         1
MNI Open Research                           1
eLife                                       1
Name: count, dtype: int64

In [85]:
pattern = "10.31234/osf.io/zypk9"

mask = df['doi'].str.contains(pattern, regex=False, na=False)
result = df[mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
2351069,crossref::10.31234/osf.io/zypk9,PsyArXiv,crossref,10.31234/osf.io/zypk9,https://doi.org/10.31234/osf.io/zypk9,https://osf.io/zypk9,posted-content,preprint,The Opposition of Surprisal and Semantic Simil...,"Sun, Kun; Nixon, Jessie S.",2020.0,2020-12-10,None,None,,,,,false,None,None,None,None,parent,https://osf.io/zypk9,10.31234/osf.io/zypk9,None,None,NaN,None,None,NaN
2357828,crossref::10.31234/osf.io/zypk9_v1,PsyArXiv,crossref,10.31234/osf.io/zypk9_v1,https://doi.org/10.31234/osf.io/zypk9_v1,https://osf.io/zypk9_v1,posted-content,preprint,WITHDRAWN,<NA>,2020.0,2025-05-19,None,None,,,,,false,None,None,None,None,parent,https://osf.io/zypk9_v1,10.31234/osf.io/zypk9_v1,_v1,explicit_version,1.0,NaN,NaN,NaN


In [86]:
pattern = "10.36227/techrxiv.24750039"

mask = df['doi'].str.contains(pattern, regex=False, na=False)
result = df[mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
6232628,crossref::10.36227/techrxiv.24750039,TechRxiv,crossref,10.36227/techrxiv.24750039,https://doi.org/10.36227/techrxiv.24750039,https://www.techrxiv.org/articles/preprint/Opt...,posted-content,preprint,Optimizing Complex CPQ Software Systems for Qu...,"Alexander, Thomson",2023.0,2023-12-07,None,None,,,,,false,None,None,None,None,parent,https://www.techrxiv.org/articles/preprint/opt...,10.36227/techrxiv.24750039,None,None,NaN,None,None,NaN
6239847,crossref::10.36227/techrxiv.24750039.v1,TechRxiv,crossref,10.36227/techrxiv.24750039.v1,https://doi.org/10.36227/techrxiv.24750039.v1,https://www.techrxiv.org/doi/full/10.36227/tec...,posted-content,preprint,Optimizing Complex CPQ Software Systems for Qu...,"Alexander, Thomson",2023.0,2023-12-07,None,None,,,,,false,None,None,None,None,parent,https://www.techrxiv.org/doi/full/10.36227/tec...,10.36227/techrxiv.24750039.v1,.v1,explicit_version,1.0,NaN,NaN,NaN


#### EarthArXiv, Authorea Inc., Cambridge Open Engage

In [87]:
import re
import pandas as pd

# Ensure target column exists
# if 'records_hierarchy' not in df.columns:
#     df['records_hierarchy'] = pd.NA

remain = df['records_hierarchy'].isna()

doi_s = df['doi'].astype(str).str.strip().str.lower()

# -------------------------
# EarthArXiv: 10.15697/<token>
# Examples: 10.15697/fk20022, 10.15697/fk2v59g
# -------------------------
mask_eartharxiv_parent = (
    remain
    & (df['server_name'] == 'EarthArXiv')
    & doi_s.str.match(r'^10\.15697/[a-z0-9]+$', na=False)
)
df.loc[mask_eartharxiv_parent, 'records_hierarchy'] = 'parent'

# -------------------------
# Authorea: 10.22541/au.<digits>.<digits>
# Examples: 10.22541/au.148928287.78038962
# -------------------------
mask_authorea_parent = (
    df['records_hierarchy'].isna()
    & (df['server_name'] == 'Authorea Inc.')
    & doi_s.str.match(r'^10\.22541/au\.\d+\.\d+$', na=False)
)
df.loc[mask_authorea_parent, 'records_hierarchy'] = 'parent'

# -------------------------
# The Winnower: 10.15200/winn.<digits>.<digits>
# Examples: 10.15200/winn.143644.45920
# -------------------------
mask_winn_parent = (
    df['records_hierarchy'].isna()
    & doi_s.str.match(r'^10\.15200/winn\.\d+\.\d+$', na=False)
)
df.loc[mask_winn_parent, 'records_hierarchy'] = 'parent'

# -------------------------
# engrXiv: 10.31224/<digits>
# Examples: 10.31224/2109
# -------------------------
mask_engrxiv_parent = (
    df['records_hierarchy'].isna()
    & (df['server_name'] == 'engrXiv')
    & doi_s.str.match(r'^10\.31224/\d+$', na=False)
)
df.loc[mask_engrxiv_parent, 'records_hierarchy'] = 'parent'

# -------------------------
# Cambridge Open Engage: 10.33774/coe-<year or token>-<token>
# Examples: 10.33774/coe-2025-zz7tp, 10.33774/coe-2020-03prm, 10.33774/coe-xxxx-xxxxx
# -------------------------
mask_coe_parent = (
    df['records_hierarchy'].isna()
    & (df['server_name'] == 'Cambridge Open Engage')
    & doi_s.str.match(r'^10\.33774/coe-[a-z0-9]{4}-[a-z0-9]{4,}$', na=False)
)
df.loc[mask_coe_parent, 'records_hierarchy'] = 'parent'

# Optional quick check
print(df['records_hierarchy'].value_counts(dropna=False))


records_hierarchy
parent              7805511
version              100951
NaN                   40382
publish_version       12888
part_of                5922
correction              354
comment                 242
review                   27
parent_duplicate          4
Name: count, dtype: int64


#### Advance, APSA Preprints, AgriRxiv, EGUsphere

In [88]:
import pandas as pd

# Ensure column exists
# if 'records_hierarchy' not in df.columns:
#     df['records_hierarchy'] = pd.NA

remain = df['records_hierarchy'].isna()
doi_s = df['doi'].astype(str).str.strip().str.lower()

# -------------------------
# Advance: 10.31124/advance.<digits>
# Examples: 10.31124/advance.10005662, 10.31124/advance.9978302
# -------------------------
mask_advance_parent = (
    remain
    & (df['server_name'] == 'Advance')
    & doi_s.str.match(r'^10\.31124/advance\.\d+$', na=False)
)
df.loc[mask_advance_parent, 'records_hierarchy'] = 'parent'

# -------------------------
# APSA Preprints: 10.33774/apsa-<yyyy>-<token>
# Examples: 10.33774/apsa-2025-zqggn, 10.33774/apsa-2019-0b2dl
# -------------------------
mask_apsa_parent = (
    df['records_hierarchy'].isna()
    & (df['server_name'] == 'APSA Preprints')
    & doi_s.str.match(r'^10\.33774/apsa-\d{4}-[a-z0-9]+$', na=False)
)
df.loc[mask_apsa_parent, 'records_hierarchy'] = 'parent'

# -------------------------
# AgriRxiv: 10.31220/agrirxiv.<yyyy>.<5digits>
# Examples: 10.31220/agrirxiv.2020.00001, 10.31220/agrirxiv.2025.00384
# -------------------------
mask_agrirxiv_parent = (
    df['records_hierarchy'].isna()
    & (df['server_name'] == 'AgriRxiv')
    & doi_s.str.match(r'^10\.31220/agrirxiv\.\d{4}\.\d{5}$', na=False)
)
df.loc[mask_agrirxiv_parent, 'records_hierarchy'] = 'parent'

# -------------------------
# EGUsphere (and EGUsphere-style journal DOIs): 10.5194/<slug>-<yyyy>-<number>
# Examples:
#   10.5194/egusphere-2022-1
#   10.5194/egusphere-2022-1000
#   10.5194/amt-2022-295
#   10.5194/hess-2024-3989
# -------------------------
mask_egusphere_parent = (
    df['records_hierarchy'].isna()
    & (df['server_name'] == 'EGUsphere')
    & doi_s.str.match(r'^10\.5194/[a-z0-9]+-\d{4}-\d+$', na=False)
)
df.loc[mask_egusphere_parent, 'records_hierarchy'] = 'parent'

# Optional quick check
print(df['records_hierarchy'].value_counts(dropna=False))

records_hierarchy
parent              7819120
version              100951
NaN                   26773
publish_version       12888
part_of                5922
correction              354
comment                 242
review                   27
parent_duplicate          4
Name: count, dtype: int64


#### Oroboros Instruments, PoolText, ScienceOpen Preprints, EarthArXiv

In [89]:
import pandas as pd

# Ensure column exists
# if 'records_hierarchy' not in df.columns:
#     df['records_hierarchy'] = pd.NA

remain = df['records_hierarchy'].isna()
doi_s = df['doi'].astype(str).str.strip().str.lower()

# ------------------------------------------------------------
# Oroboros Instruments (examples)
# - 10.26124/bec.2024-0007
# - 10.26124/mitofit:ea19.mipschool.0005
# Parent if EXACT pattern with nothing after.
# ------------------------------------------------------------
mask_oroboros_parent = (
    remain
    & (df['server_name'] == 'Oroboros Instruments')
    & doi_s.str.match(
        r'^10\.26124/('
        r'bec\.\d{4}-\d{4}'                       # bec.2024-0007
        r'|mitofit:[a-z0-9]+\.[a-z0-9]+\.\d{4}'   # mitofit:ea19.mipschool.0005
        r')$',
        na=False
    )
)
df.loc[mask_oroboros_parent, 'records_hierarchy'] = 'parent'

# ------------------------------------------------------------
# Oroboros Instruments — Parent DOI patterns
# Examples (parents):
# - 10.26124/bec.2025-0005ar
# - 10.26124/bec.2025-0005it
# - 10.26124/bec:2024-0001
# - 10.26124/becprep.2025-0006.ed2
# - 10.26124/becprep.2025-0005
# - 10.26124/mitofit:190001
# - 10.26124/mitofit:2021-0005
# ------------------------------------------------------------
OROBOROS_PARENT_RX = r'^10\.26124/(' \
    r'bec[.:]\d{4}-\d{4}[a-z]{0,3}' \
    r'|' \
    r'becprep\.\d{4}-\d{4}(?:\.[a-z0-9]{1,6})?' \
    r'|' \
    r'mitofit:\d{6}' \
    r'|' \
    r'mitofit:\d{4}-\d{4}' \
    r')$'

mask_oroboros_parent = (
    remain
    & (df['server_name'] == 'Oroboros Instruments')
    & doi_s.str.match(OROBOROS_PARENT_RX, na=False)
)

df.loc[mask_oroboros_parent, 'records_hierarchy'] = 'parent'

# ------------------------------------------------------------
# PoolText (examples)
# - 10.31923/5547-4288-0095
# - 10.31923/pooltext-preprint-0067-3907-0053
# Parent if EXACT pattern with nothing after.
# ------------------------------------------------------------
mask_pooltext_parent = (
    df['records_hierarchy'].isna()
    & (df['server_name'] == 'PoolText')
    & doi_s.str.match(
        r'^10\.31923/('
        r'\d{4}-\d{4}-\d{4}'                                  # 5547-4288-0095
        r'|pooltext-preprint-\d{4}-\d{4}-\d{4}'               # pooltext-preprint-0067-3907-0053
        r')$',
        na=False
    )
)
df.loc[mask_pooltext_parent, 'records_hierarchy'] = 'parent'


# ------------------------------------------------------------
# ScienceOpen Preprints — Parent DOI patterns (STRICT / exact)
# Parent if EXACT pattern with nothing after.
# ------------------------------------------------------------

SCIENCEOPEN_PARENT_RX = (
    r'^10\.14293/('
    # 1) Old pattern like: s2199-1006.1.sor-.sdg.01
    r's2199-1006\.1\.sor-\.[a-z0-9]+\.\d{2}'
    r'|'
    # 2) New s2199-r2om patterns:
    #    - s2199-r2om-0001
    #    - s2199-r2om-abs-0003
    r's2199-r2om-(?:abs-)?\d{4}'
    r'|'
    # 3) New "rexpo" compact pattern:
    #    - s2199-rexpo22011v1
    r's2199-rexpo\d{5}v\d+'
    r'|'
    # 4) New ssp-am patterns:
    #    - s2199-ssp-am22-0001
    #    - s2199-ssp-am23-01001
    #    - s2199-ssp-am25-01015
    r's2199-ssp-am\d{2}-\d{4,5}'
    r'|'
    # 5) sblunisa patterns:
    #    - sblunisa.2023a024.mm (old)
    #    - sblunisa.2023a002.vnm (new)
    #    - sblunisa.2023a017.ojjt (new)
    r'sblunisa\.\d{4}a\d{3}\.[a-z0-9]{2,4}'
    r')$'
)

mask_scienceopen_parent = (
    df['records_hierarchy'].isna()
    & (df['server_name'] == 'ScienceOpen Preprints')
    & doi_s.str.match(SCIENCEOPEN_PARENT_RX, na=False)
)

df.loc[mask_scienceopen_parent, 'records_hierarchy'] = 'parent'


# ------------------------------------------------------------
# EarthArXiv
# Examples:
# - 10.31223/x50025  (5)
# - 10.31223/x5003j  (6)
# - 10.31223/x5zr0p  (6)
# Parent if EXACT pattern with nothing after.
# ------------------------------------------------------------
mask_eartharxiv_parent = (
    df['records_hierarchy'].isna()
    & (df['server_name'] == 'EarthArXiv')
    & doi_s.str.match(r'^10\.31223/[a-z0-9]{5,6}$', na=False)   # 5–6 chars after slash
)
df.loc[mask_eartharxiv_parent, 'records_hierarchy'] = 'parent'


# Optional quick check
print(df['records_hierarchy'].value_counts(dropna=False))

records_hierarchy
parent              7823960
version              100951
NaN                   21933
publish_version       12888
part_of                5922
correction              354
comment                 242
review                   27
parent_duplicate          4
Name: count, dtype: int64


#### PeerJ Preprints

In [90]:
import re

# ------------------------------------------------------------
# PeerJ Preprints
# Parent:
#  - no trailing vN: 10.7287/peerj.preprints.1001
#  - trailing v1:    10.7287/peerj.preprints.1001v1
# Child:
#  - trailing v2+:   10.7287/peerj.preprints.1001v2, v3, ...
# ------------------------------------------------------------

doi_s = df['doi'].astype(str).str.strip().str.lower()

mask_peerj = (
    df['records_hierarchy'].isna()
    & (df['server_name'] == 'PeerJ Preprints')
    & doi_s.str.startswith('10.7287/peerj.preprints.', na=False)
)

# Extract trailing version number if present (v1, v2, ...)
peerj_v_str = doi_s.where(mask_peerj).str.extract(r'v(?P<v>\d+)$')['v']

# Convert safely to numeric (NaN stays NaN)
peerj_v_num = pd.to_numeric(peerj_v_str, errors='coerce')

# Parent: no version suffix OR v1
mask_peerj_parent = mask_peerj & (peerj_v_num.isna() | (peerj_v_num == 1))
df.loc[mask_peerj_parent, 'records_hierarchy'] = 'parent'

# Child: v2+
mask_peerj_child = mask_peerj & (peerj_v_num >= 2)
df.loc[mask_peerj_child, 'records_hierarchy'] = 'version'

# Optional quick check
print(df['records_hierarchy'].value_counts(dropna=False))

records_hierarchy
parent              7829031
version              102326
NaN                   15487
publish_version       12888
part_of                5922
correction              354
comment                 242
review                   27
parent_duplicate          4
Name: count, dtype: int64


#### Earth and Space Science Open Archive

In [91]:
import pandas as pd

# -------------------------------------------------------------------
# Normalize DOI strings:
# - ensure everything is string
# - remove leading/trailing spaces
# - lowercase for consistent matching
# -------------------------------------------------------------------
doi_s = df['doi'].astype(str).str.strip().str.lower()

# -------------------------------------------------------------------
# Identify records to process:
# - records_hierarchy not yet assigned
# - server is Earth and Space Science Open Archive (ESSOAr)
# -------------------------------------------------------------------
mask_essoar = (
    df['records_hierarchy'].isna()
    & (df['server_name'] == 'Earth and Space Science Open Archive')
)

# -------------------------------------------------------------------
# Extract the final numeric version from the DOI
#
# Examples:
#   10.1002/essoar.10500074.1  → n = 1
#   10.1002/essoar.10500061.2  → n = 2
#   10.22541/essoar.xxx.xxx.3  → n = 3
#
# Regex explanation:
#   \.        → literal dot
#   (?P<n>)  → capture group named "n"
#   \d+      → one or more digits
#   $        → end of string (must be the last segment)
# -------------------------------------------------------------------
essoar_n_str = (
    doi_s
    .where(mask_essoar)                # only evaluate ESSOAr rows
    .str.extract(r'\.(?P<n>\d+)$')['n']  # extract trailing version number
)

# Convert extracted version to numeric:
# - invalid or missing values become NaN (safe for comparisons)
essoar_n = pd.to_numeric(essoar_n_str, errors='coerce')

# -------------------------------------------------------------------
# Label parent records:
# - ESSOAr records
# - version suffix == ".1"
# -------------------------------------------------------------------
mask_essoar_parent = mask_essoar & (essoar_n == 1)
df.loc[mask_essoar_parent, 'records_hierarchy'] = 'parent'

# -------------------------------------------------------------------
# Label child records:
# - ESSOAr records
# - version suffix >= ".2"
# -------------------------------------------------------------------
mask_essoar_child = mask_essoar & (essoar_n >= 2)
df.loc[mask_essoar_child, 'records_hierarchy'] = 'child'

# -------------------------------------------------------------------
# Optional sanity check:
# Show distribution of hierarchy labels
# -------------------------------------------------------------------
print(df['records_hierarchy'].value_counts(dropna=False))


records_hierarchy
parent              7839845
version              102326
publish_version       12888
part_of                5922
NaN                    2612
child                  2061
correction              354
comment                 242
review                   27
parent_duplicate          4
Name: count, dtype: int64


In [92]:
df.loc[df['server_name'].eq('Earth and Space Science Open Archive'), 'records_hierarchy'].value_counts(dropna=False)

records_hierarchy
parent             19866
child               2061
version              789
publish_version       31
part_of                1
Name: count, dtype: int64

#### F1000Research

In [93]:
df.loc[df['server_name'].eq('F1000Research'), 'records_hierarchy'].value_counts(dropna=False)


records_hierarchy
parent     11155
version     5704
NaN           14
Name: count, dtype: int64

In [94]:
df_remain = df[df['records_hierarchy'].isna()]
df_remain['server_name'].value_counts()

server_name
AfricArXiv              1689
Jxiv                     902
F1000Research             14
AMRC Open Research         3
Gates Open Research        1
Open Research Africa       1
eLife                      1
MNI Open Research          1
Name: count, dtype: int64

In [95]:
df_remain[df_remain['server_name']=='F1000Research']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
545556,crossref::10.12688/f1000research,F1000Research,crossref,10.12688/f1000research,https://doi.org/10.12688/f1000research,http://www.f1000research.com,journal,None,F1000Research,<NA>,None,2025-07-24,None,None,,,,,false,None,None,None,None,NaN,http://www.f1000research.com,10.12688/f1000research,None,None,NaN,None,None,NaN
546827,crossref::10.12688/f1000research.11198.1,F1000Research,crossref,10.12688/f1000research.11198.1,https://doi.org/10.12688/f1000research.11198.1,https://f1000research.com/articles/6-1014,journal-article,None,New perspectives on the regulation of type II ...,"Becerra-Diaz, Mireya",2017.0,2025-11-28,None,None,,,,,false,None,None,None,None,NaN,https://f1000research.com/articles/6-1014,10.12688/f1000research.11198.1,None,None,NaN,None,None,NaN
538713,crossref::10.12688/f1000research.124059.1,F1000Research,crossref,10.12688/f1000research.124059.1,https://doi.org/10.12688/f1000research.124059.1,https://f1000research.com/articles/11-1230/,journal-article,None,LiftoffTools: a toolkit for comparing gene ann...,"Shumate, Alaina; Salzberg, Steven",2022.0,2022-10-28,None,None,,,,,false,None,None,None,None,NaN,https://f1000research.com/articles/11-1230/,10.12688/f1000research.124059.1,None,None,NaN,None,None,NaN
546826,crossref::10.12688/f1000research.14416.1,F1000Research,crossref,10.12688/f1000research.14416.1,https://doi.org/10.12688/f1000research.14416.1,https://f1000research.com/articles/7-1362,journal-article,None,Recent advances in the understanding and manag...,"Hay, Ashley",2018.0,2025-11-28,None,None,,,,,false,None,None,None,None,NaN,https://f1000research.com/articles/7-1362,10.12688/f1000research.14416.1,None,None,NaN,None,None,NaN
546908,crossref::10.12688/f1000research.163729.1,F1000Research,crossref,10.12688/f1000research.163729.1,https://doi.org/10.12688/f1000research.163729.1,https://f1000research.com/articles/14-656,journal-article,None,Autoimmune Patient Health Through a Flourishin...,"Pasca Rina, Amherstia",2025.0,2025-12-01,None,None,,,,,false,None,None,None,None,NaN,https://f1000research.com/articles/14-656,10.12688/f1000research.163729.1,None,None,NaN,None,None,NaN
546913,crossref::10.12688/f1000research.166247.1,F1000Research,crossref,10.12688/f1000research.166247.1,https://doi.org/10.12688/f1000research.166247.1,https://f1000research.com/articles/14-598,journal-article,None,Anticipated Growth in Healthcare Spending: The...,"Rugchatjaroen, Krish",2025.0,2025-12-01,None,None,,,,,false,None,None,None,None,NaN,https://f1000research.com/articles/14-598,10.12688/f1000research.166247.1,None,None,NaN,None,None,NaN
546919,crossref::10.12688/f1000research.168206.1,F1000Research,crossref,10.12688/f1000research.168206.1,https://doi.org/10.12688/f1000research.168206.1,https://f1000research.com/articles/14-975,journal-article,None,Scientific Productivity and Population Health:...,"Altamimi, Omar",2025.0,2025-12-01,None,None,,,,,false,None,None,None,None,NaN,https://f1000research.com/articles/14-975,10.12688/f1000research.168206.1,None,None,NaN,None,None,NaN
530386,crossref::10.3410/10.3410/f1000devtestarticley,F1000Research,crossref,10.3410/10.3410/f1000devtestarticley,https://doi.org/10.3410/10.3410/f1000devtestar...,http://www.xy.net/article,journal-article,None,someTitle,"abcde, XXXX",2012.0,2012-06-11,None,None,,,,,false,None,None,None,None,NaN,http://www.xy.net/article,10.3410/10.3410/f1000devtestarticley,None,None,NaN,None,None,NaN
530674,crossref::10.3410/123.430,F1000Research,crossref,10.3410/123.430,https://doi.org/10.3410/123.430,http://www.someURl.

In [96]:
df_remain[df_remain['server_name']=='Open Research Africa']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
2080972,crossref::10.12688/aasopenres,Open Research Africa,crossref,10.12688/aasopenres,https://doi.org/10.12688/aasopenres,http://www.aasopenresearch.org,journal,None,AAS Open Research,<NA>,None,2022-03-25,None,None,,,,,false,None,None,None,None,NaN,http://www.aasopenresearch.org,10.12688/aasopenres,None,None,NaN,None,None,NaN


In [97]:
df_remain[df_remain['server_name']=='AMRC Open Research']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
179,crossref::10.12688/amrcopenres,AMRC Open Research,crossref,10.12688/amrcopenres,https://doi.org/10.12688/amrcopenres,http://www.amrcopenresearch.org,journal,None,AMRC Open Research,<NA>,None,2019-02-19,None,None,,,,,false,None,None,None,None,NaN,http://www.amrcopenresearch.org,10.12688/amrcopenres,None,None,NaN,None,None,NaN
244,crossref::10.12688/amrcopenres.crossmark-policy,AMRC Open Research,crossref,10.12688/amrcopenres.crossmark-policy,https://doi.org/10.12688/amrcopenres.crossmark...,https://amrcopenresearch.org/about/policies,dataset,None,<NA>,<NA>,None,2018-11-14,None,None,,,,,false,None,None,None,None,NaN,https://amrcopenresearch.org/about/policies,10.12688/amrcopenres.crossmark-policy,None,None,NaN,None,None,NaN
240,crossref::10.12688/healthopenres,AMRC Open Research,crossref,10.12688/healthopenres,https://doi.org/10.12688/healthopenres,http://www.healthopenresearch.org,journal,None,Health Open Research,<NA>,None,2022-11-23,None,None,,,,,false,None,None,None,None,NaN,http://www.healthopenresearch.org,10.12688/healthopenres,None,None,NaN,None,None,NaN


In [98]:
df_remain[df_remain['server_name']=='MNI Open Research']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
1972382,crossref::10.12688/mniopenres,MNI Open Research,crossref,10.12688/mniopenres,https://doi.org/10.12688/mniopenres,http://www.mniopenresearch.org,journal,None,MNI Open Research,<NA>,None,2021-06-21,None,None,,,,,false,None,None,None,None,NaN,http://www.mniopenresearch.org,10.12688/mniopenres,None,None,NaN,None,None,NaN


In [99]:
df_remain[df_remain['server_name']=='Gates Open Research']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
548380,crossref::10.12688/gatesopenres,Gates Open Research,crossref,10.12688/gatesopenres,https://doi.org/10.12688/gatesopenres,http://www.gatesopenresearch.org,journal,None,Gates Open Research,<NA>,None,2017-11-06,None,None,,,,,false,None,None,None,None,NaN,http://www.gatesopenresearch.org,10.12688/gatesopenres,None,None,NaN,None,None,NaN


In [100]:
df_remain[df_remain['server_name']=='eLife']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
9658075,crossref::10.7554/elife,eLife,crossref,10.7554/elife,https://doi.org/10.7554/elife,https://elifesciences.org/,journal,None,eLife,<NA>,None,2017-07-25,None,None,,,,,false,None,None,None,None,NaN,https://elifesciences.org/,10.7554/elife,None,None,NaN,None,None,NaN


In [101]:
import pandas as pd

# Normalize DOI strings:
# - lower case
# - strip spaces
# - collapse accidental double slashes after the prefix (10.3410// -> 10.3410/)
doi_s = (
    df["doi"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"^10\.3410//", "10.3410/", regex=True)
)

# ------------------------------------------------------------
# F1000Research
# We support TWO DOI "families" that exist in your data:
#
# A) 10.12688/f1000research[.<id>.<N>]
#    - root: 10.12688/f1000research            -> parent
#    - versioned: 10.12688/f1000research.11198.1 -> parent
#                 10.12688/f1000research.11198.2 -> child
#
# B) 10.3410/f1000research.<something>.vN
#    - 10.3410/f1000research.1-29.v1 -> parent
#    - 10.3410/f1000research.1-29.v2 -> child
#    - 10.3410/f1000research.2-72.v1 -> parent
# ------------------------------------------------------------
mask_f1000 = (
    df["records_hierarchy"].isna()
    & (df["server_name"] == "F1000Research")
)

# -------------------------
# Case A: 10.12688 root DOI
# -------------------------
mask_f1000_root = mask_f1000 & doi_s.eq("10.12688/f1000research")
df.loc[mask_f1000_root, "records_hierarchy"] = "parent"

# -----------------------------------------------
# Case A2: 10.12688/f1000research.<digits>.<N>
# -----------------------------------------------
f1000_v12688_str = (
    doi_s.where(mask_f1000)
         .str.extract(r"^10\.12688/f1000research\.\d+\.(?P<v>\d+)$")["v"]
)
f1000_v12688 = pd.to_numeric(f1000_v12688_str, errors="coerce")

mask_f1000_12688_parent = mask_f1000 & (f1000_v12688 == 1)
df.loc[mask_f1000_12688_parent, "records_hierarchy"] = "parent"

mask_f1000_12688_child = mask_f1000 & (f1000_v12688 >= 2)
df.loc[mask_f1000_12688_child, "records_hierarchy"] = "child"

# -----------------------------------------
# Case B: 10.3410/f1000research.<...>.vN
# - accept things like: 1-29, 2-72, 1-50, etc.
# - also accept possible extra dots inside the middle part
# -----------------------------------------
f1000_v3410_str = (
    doi_s.where(mask_f1000)
         .str.extract(r"^10\.3410/f1000research\.[a-z0-9.\-]+\.v(?P<v>\d+)$")["v"]
)
f1000_v3410 = pd.to_numeric(f1000_v3410_str, errors="coerce")

mask_f1000_3410_parent = mask_f1000 & (f1000_v3410 == 1)
df.loc[mask_f1000_3410_parent, "records_hierarchy"] = "parent"

mask_f1000_3410_child = mask_f1000 & (f1000_v3410 >= 2)
df.loc[mask_f1000_3410_child, "records_hierarchy"] = "child"

# -------------------------
# Optional: remaining -> others
# -------------------------
mask_f1000_left = mask_f1000 & df["records_hierarchy"].isna()
df.loc[mask_f1000_left, "records_hierarchy"] = "others"

# Quick check
print(df.loc[df["server_name"] == "F1000Research", "records_hierarchy"].value_counts(dropna=False))


records_hierarchy
parent     11162
version     5704
others         7
Name: count, dtype: int64


In [102]:
# Normalize DOI (same style you already use)
doi_s = df["doi"].astype(str).str.strip().str.lower()

# Only touch rows not yet labeled
mask_unlabeled = df["records_hierarchy"].isna()

# Root DOIs you want to classify as "others"
ROOT_OTHERS_DOIS = {
    "10.7554/elife",
    "10.12688/gatesopenres",
    "10.12688/mniopenres",
    "10.12688/amrcopenres",
    "10.12688/aasopenres",
}

mask_root_others = mask_unlabeled & doi_s.isin(ROOT_OTHERS_DOIS)
df.loc[mask_root_others, "records_hierarchy"] = "others"


In [103]:

# -------------------------------------------------------------------
# Optional sanity check:
# Show distribution of hierarchy labels
# -------------------------------------------------------------------
print(df['records_hierarchy'].value_counts(dropna=False))

records_hierarchy
parent              7839852
version              102326
publish_version       12888
part_of                5922
NaN                    2593
child                  2061
correction              354
comment                 242
review                   27
others                   12
parent_duplicate          4
Name: count, dtype: int64


In [104]:
pattern = "others"


mask = df['records_hierarchy'].str.contains(pattern, regex=False, na=False)
result = df[mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
2080972,crossref::10.12688/aasopenres,Open Research Africa,crossref,10.12688/aasopenres,https://doi.org/10.12688/aasopenres,http://www.aasopenresearch.org,journal,None,AAS Open Research,<NA>,None,2022-03-25,None,None,,,,,false,None,None,None,None,others,http://www.aasopenresearch.org,10.12688/aasopenres,None,None,NaN,None,None,NaN
179,crossref::10.12688/amrcopenres,AMRC Open Research,crossref,10.12688/amrcopenres,https://doi.org/10.12688/amrcopenres,http://www.amrcopenresearch.org,journal,None,AMRC Open Research,<NA>,None,2019-02-19,None,None,,,,,false,None,None,None,None,others,http://www.amrcopenresearch.org,10.12688/amrcopenres,None,None,NaN,None,None,NaN
548380,crossref::10.12688/gatesopenres,Gates Open Research,crossref,10.12688/gatesopenres,https://doi.org/10.12688/gatesopenres,http://www.gatesopenresearch.org,journal,None,Gates Open Research,<NA>,None,2017-11-06,None,None,,,,,false,None,None,None,None,others,http://www.gatesopenresearch.org,10.12688/gatesopenres,None,None,NaN,None,None,NaN
1972382,crossref::10.12688/mniopenres,MNI Open Research,crossref,10.12688/mniopenres,https://doi.org/10.12688/mniopenres,http://www.mniopenresearch.org,journal,None,MNI Open Research,<NA>,None,2021-06-21,None,None,,,,,false,None,None,None,None,others,http://www.mniopenresearch.org,10.12688/mniopenres,None,None,NaN,None,None,NaN
530386,crossref::10.3410/10.3410/f1000devtestarticley,F1000Research,crossref,10.3410/10.3410/f1000devtestarticley,https://doi.org/10.3410/10.3410/f1000devtestar...,http://www.xy.net/article,journal-article,None,someTitle,"abcde, XXXX",2012.0,2012-06-11,None,None,,,,,false,None,None,None,None,others,http://www.xy.net/article,10.3410/10.3410/f1000devtestarticley,None,None,NaN,None,None,NaN
530674,crossref::10.3410/123.430,F1000Research,crossref,10.3410/123.430,https://doi.org/10.3410/123.430,http://www.someURl.com,journal-article,None,someTitle,"someName, someName",2009.0,2012-09-13,None,None,,,,,false,None,None,None,None,others,http://www.someurl.com,10.3410/123.430,None,None,NaN,None,None,NaN
530385,crossref::10.3410/f1000devtestarticlez,F1000Research,crossref,10.3410/f1000devtestarticlez,https://doi.org/10.3410/f1000devtestarticlez,http://researchdev.f1000.com/articles/F1000Dev...,journal-article,None,F1000DevTestArticleZ,"LNameZ, firstnameZ",2012.0,2012-06-07,None,None,,,,,false,None,None,None,None,others,http://researchdev.f1000.com/articles/f1000dev...,10.3410/f1000devtestarticlez,None,None,NaN,None,None,NaN
530384,crossref::10.3410/f1000devtestcrossmarkpolicy,F1000Research,crossref,10.3410/f1000devtestcrossmarkpolicy,https://doi.org/10.3410/f1000devtestcrossmarkp...,resource,journal-article,None,F1000DevTestCrossMarkPolicy,"Manager, Policy",2012.0,2012-05-31,None,None,,,,,false,None,None,None,None,others,resource,10.3410/f1000devtestcrossmarkpolicy,None,None,NaN,None,None,NaN
530387,crossref::10.3410/f1000res,F1000Research,crossref,10.3410/f1000res,https://doi.org/10.3410/f1000res,2046-1402,journal,None,F1000 Research,<NA>,None,2013-03-05,None,None,,,,,false,None,None,None,None,others,2046-1402,10.3410/f1000res,None,None,NaN,None,None,NaN
530389,crossref::10.3410/f1000research,F1000Research,crossref,10.3410/f1000research,https://doi.org/10.3410/f1000research,http://www.f1000research.com/,journal,None,F1000Research,<NA>,None,2013-05-09,None,None,,,,,false,None,None,None,None,others,http://www.f1000research.com/,10.3410/f1000research,None,None,NaN,None,None,NaN


In [105]:
df_remain = df[df['records_hierarchy'].isna()]
df_remain['server_name'].value_counts()

server_name
AfricArXiv            1689
Jxiv                   902
AMRC Open Research       2
Name: count, dtype: int64

In [106]:
# df_remain[df_remain['server_name']=='AgriRxiv']['landing_page_url'][207416]

In [107]:
# pattern = "others"


# mask = df[df['server_name']=='F1000Research']['records_hierarchy'].str.contains(pattern, regex=False, na=False)
# result = df[df['server_name']=='F1000Research'][mask]
# result

In [108]:
# df_remain['landing_page_url'][286254]

In [109]:
# pattern = ".1"


# mask = df[df['server_name']=='eLife']['doi'].str.contains(pattern, regex=False, na=False)
# result = df[df['server_name']=='eLife'][mask]
# result

In [110]:
# result['landing_page_url'][9366338]

In [111]:
# result['landing_page_url'][9297794] 

In [112]:
# pattern = "10.7287/peerj.preprints.999"
# #video #media 

# mask = df['doi'].str.contains(pattern, regex=False, na=False)
# result = df[mask]
# result

In [113]:
# df_remain['relations_json'].value_counts()

In [114]:
# df_remain[df_remain['relations_json']=='{"is-preprint-of": [{"asserted-by": "subject", "id": "10.31237/osf.io/yr86k", "id-type": "doi"}]}']

## Check duplicates in secondary preprint repositories

### get list of accros server

In [115]:
import pandas as pd

# ============================================================
# Goal
# ============================================================
# Read the "rules" Google Sheet and extract a list of servers
# that are marked as primary sources.
#
# In the sheet:
# - Column "Field_server_name" contains the server name
# - Column "primary_source" contains "yes" for primary servers
#
# Output:
# - unique_servers: Python list of server names marked as primary
# ============================================================


# ============================================================
# 1) Read the Google Sheet tab as CSV
# ============================================================
# Google Sheets can be exported as CSV if the sheet is public
# (or shared with link access).
# You need:
# - SHEET_ID: the spreadsheet ID
# - GID: the tab id (worksheet id)
SHEET_ID = "10_7FdcpZjntqFsEHIii7bAM72uF__of_iUohSD5w8w4"
GID = "1230415212"  # tab gid for the rules sheet

# Build the CSV export URL
rules_csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}"

# Read the rules table into a DataFrame
rules = pd.read_csv(rules_csv_url)


# ============================================================
# 2) Normalize column names
# ============================================================
# Sheets sometimes contain extra spaces/newlines in column headers.
# This normalizes them to reduce "column not found" errors.
rules.columns = (
    rules.columns.astype(str)
    .str.replace(r"\s+", " ", regex=True)  # collapse multiple spaces/newlines into 1 space
    .str.strip()                           # remove leading/trailing spaces
)


# ============================================================
# 3) Define which columns we rely on
# ============================================================
# SERVER_COL: server name column
# FLAG_COL: column marking primary servers ("yes")
SERVER_COL = "Field_server_name"
FLAG_COL = "primary_source"


# ============================================================
# 4) Safety check: make sure expected columns exist
# ============================================================
# If the sheet changes (renamed columns, etc.), we fail early with a helpful message.
if SERVER_COL not in rules.columns or FLAG_COL not in rules.columns:
    print("Columns available in the sheet:", rules.columns.tolist())
    raise KeyError(f"Expected columns not found. Need: {SERVER_COL!r} and {FLAG_COL!r}")


# ============================================================
# 5) Extract primary servers (where primary_source == "yes")
# ============================================================
# Steps:
# - normalize the flag column to text
# - strip spaces, lowercase
# - keep rows where value == "yes"
# - take the server names
# - drop missing names
# - strip spaces
# - keep unique values
primary_servers = (
    rules.loc[
        rules[FLAG_COL].astype(str).str.strip().str.lower().eq("yes"),
        SERVER_COL
    ]
    .dropna()                 # remove missing server names
    .astype(str)
    .str.strip()              # normalize server name text
    .unique()                 # keep distinct values only
    .tolist()                 # convert numpy array to normal Python list
)


# ============================================================
# 6) Quick preview
# ============================================================
print(f"Primary servers found: {len(primary_servers)}")
print(primary_servers[:30])  # show first 30 as a preview


Primary servers found: 73
['Advance', 'AfricArXiv', 'AgEcon Search', 'AgriRxiv', 'AIJR Preprints', 'APSA Preprints', 'Arabixiv', 'ARPHA Preprints', 'ART-Dok', 'arXiv', 'Authorea Inc.', 'Beilstein Archives', 'BioHackrXiv', 'bioRxiv', 'BodoArXiv', 'Cambridge Open Engage', 'CERN document server', 'ChemRxiv', 'CoP', 'Covid-19 Preprints', 'CrimRxiv', 'Earth and Space Science Open Archive', 'EarthArXiv', 'EcoEvoRxiv', 'ECSarXiv', 'EdArXiv', 'eLife', 'Encyclopedia', 'EnerarXiv', 'engrXiv']


In [116]:
set(primary_servers[:5])

{'AIJR Preprints', 'Advance', 'AfricArXiv', 'AgEcon Search', 'AgriRxiv'}

### doi

In [117]:
df_mirror = df.copy()

In [118]:
df_touse = df_mirror[df_mirror['doi'].notna()].copy()
df_touse

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
399051,crossref::10.1002/essoar.10500000.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10500000.1,https://doi.org/10.1002/essoar.10500000.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Long-term monitoring of land surface phenologi...,"Tsutsumida, Narumasa",2018.0,2019-11-13,None,None,,,,,false,None,None,None,None,parent,https://essopenarchive.org/doi/full/10.1002/es...,10.1002/essoar.10500000.1,None,None,NaN,None,None,NaN
399052,crossref::10.1002/essoar.10500002.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10500002.1,https://doi.org/10.1002/essoar.10500002.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Impact of spatial scale for phenological indic...,"Tsutsumida, Narumasa; Kaduk, Jörg",2018.0,2019-11-13,None,None,,,,,false,None,None,None,None,parent,https://essopenarchive.org/doi/full/10.1002/es...,10.1002/essoar.10500002.1,None,None,NaN,None,None,NaN
399047,crossref::10.1002/essoar.10500004.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10500004.1,https://doi.org/10.1002/essoar.10500004.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Observations of Low Latitude Red Aurora in Mex...,"Gonzalez-Esparza, J. Americo; Cuevas-Cardona, ...",2018.0,2019-11-13,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.1029/2018sw001995,,,true,None,None,None,None,parent,https://essopenarchive.org/doi/full/10.1002/es...,10.1002/essoar.10500004.1,None,None,NaN,None,None,NaN
399073,crossref::10.1002/essoar.10500007.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10500007.1,https://doi.org/10.1002/essoar.10500007.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Pipeline oil fire detection with MODIS active ...,"Ogungbuyi, Michael Gbenga; Martinez, Peter; Ec...",2018.0,2019-11-13,None,None,,,,,false,None,None,None,None,parent,https://essopenarchive.org/doi/full/10.1002/es...,10.1002/essoar.10500007.1,None,None,NaN,None,None,NaN
399068,crossref::10.1002/essoar.10500009.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10500009.1,https://doi.org/10.1002/essoar.10500009.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Land Product Validation of MODIS Derived FPAR ...,"Sharp, Iain; Sanchez-Azofeifa, Arturo; Musilek...",2018.0,2019-12-03,None,None,,,,,false,None,None,None,None,parent,https://essopenarchive.org/doi/full/10.1002/es...,10.1002/essoar.10500009.1,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6205812,openalex::W998353979,Social Science Open Access Repository,openalex,10.12759/hsr.35.2010.3.47-78,https://doi.org/10.12759/hsr.35.2010.3.47-78,http://www.ssoar.info/ssoar/handle/document/31068,article,None,"""Gutes Klassenbewusstsein, Parteiverbundenheit...",Sandra Meenzen,2010.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://www.ssoar.info/ssoar/handle/document/31068,10.12759/hsr.35.2010.3.47-78,None,None,NaN,None,None,NaN
6197570,openalex::W999063011,Social Science Open Access Repository,openalex,10.15464/isi.42.2009.1-5,https://doi.org/10.15464/isi.42.2009.1-5,http://www.ssoar.info/ssoar/handle/document/21392,article,None,Jeder fünfte Erwerbstätige ist aus beruflichen...,Silvia Ruppenthal; Detlev Lück,2009.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://www.ssoar.info/s

In [119]:
dupes = df_touse[df_touse.duplicated(subset=['doi'], keep=False)]
dupes

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
9503922,crossref::10.1101/103937,bioRxiv,crossref,10.1101/103937,https://doi.org/10.1101/103937,http://biorxiv.org/lookup/doi/10.1101/103937,posted-content,preprint,LET-99-dependent spatial restriction of active...,"Bouvrais, H.; Chesneau, L.; Pastezeur, S.; Del...",2017.0,2017-01-29,None,None,,,,,false,None,None,None,None,parent,http://biorxiv.org/lookup/doi/10.1101/103937,10.1101/103937,None,None,NaN,None,None,NaN
9601114,crossref::10.1101/2019.12.23.887166,bioRxiv,crossref,10.1101/2019.12.23.887166,https://doi.org/10.1101/2019.12.23.887166,http://biorxiv.org/lookup/doi/10.1101/2019.12....,posted-content,preprint,Model balancing: in search of consistent metab...,"Liebermeister, Wolfram; Noor, Elad",2019.0,2019-12-24,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.3390/metabo11110749,,,true,None,None,None,None,parent,http://biorxiv.org/lookup/doi/10.1101/2019.12....,10.1101/2019.12.23.887166,None,None,NaN,None,None,NaN
9514240,crossref::10.1101/2020.01.22.915215,bioRxiv,crossref,10.1101/2020.01.22.915215,https://doi.org/10.1101/2020.01.22.915215,http://biorxiv.org/lookup/doi/10.1101/2020.01....,posted-content,preprint,Not only compulsivity: The SAPAP3-KO mouse rec...,"Lamothe, H; Schreiweis, C; Lavielle, O; Mallet...",2020.0,2020-01-23,None,None,,,,,false,None,None,None,None,parent,http://biorxiv.org/lookup/doi/10.1101/2020.01....,10.1101/2020.01.22.915215,None,None,NaN,None,None,NaN
9509051,crossref::10.1101/2020.02.03.919597,bioRxiv,crossref,10.1101/2020.02.03.919597,https://doi.org/10.1101/2020.02.03.919597,http://biorxiv.org/lookup/doi/10.1101/2020.02....,posted-content,preprint,Germline inherited small RNAs clear untranslat...,"Quarato, Piergiuseppe; Singh, Meetali; Cornes,...",2020.0,2020-02-04,None,None,,,,,false,None,None,None,None,parent,http://biorxiv.org/lookup/doi/10.1101/2020.02....,10.1101/2020.02.03.919597,None,None,NaN,None,None,NaN
9418678,crossref::10.1101/2020.06.08.118984,bioRxiv,crossref,10.1101/2020.06.08.118984,https://doi.org/10.1101/2020.06.08.118984,http://biorxiv.org/lookup/doi/10.1101/2020.06....,posted-content,preprint,TBPL2/TFIIA complex establishes the maternal t...,"Yu, Changwei; Cvetesic, Nevena; Hisler, Vincen...",2020.0,2020-06-09,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.1038/s41467-020-20239-4,,,true,None,None,None,None,parent,http://biorxiv.org/lookup/doi/10.1101/2020.06....,10.1101/2020.06.08.118984,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2505509,openalex::W974745794,RePEc: Research Papers in Economics,openalex,10.22004/ag.econ.123005,https://doi.org/10.22004/ag.econ.123005,http://publications.dyson.cornell.edu/research...,preprint,None,IDENTIFYING A REDUCED SET OF SALIENT ATTRIBUTE...,Heiko Miles; Steven J. Schwager; John E. Lenz,1994.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://publications.dyson.cornell.edu/research...,10.22004/ag.econ.123005,None,None,NaN,None,None,NaN
2598756,openalex::W975694582,RePEc: Research Papers in Economics,openalex,10.22004/ag.econ.358354,https://doi.org/10.22004/ag.econ.358354,https://doi.org/10.22004/ag.econ.358354,article,None,Strategie rozwoju przedsiębiorstw z polskim ka...,Vitali Naumavets,2001.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://doi.org/10.22004/ag.econ.358354,10.22004/ag.econ.358354,None,None,NaN,None,None,NaN
2602970,openalex::W98271585,ReP

In [120]:
dupes['server_name'].value_counts()

server_name
RePEc: Research Papers in Economics      10472
arXiv                                     7147
AgEcon Search                             6445
HAL                                       4035
ResearchGate                               827
CERN document server                       776
Zenodo                                     191
bioRxiv                                     29
Social Science Open Access Repository       24
Open Science Framework                      21
SSRN                                        17
Humanities Commons CORE                     16
AfricArXiv                                   2
EarthArXiv                                   2
Research Square                              2
Open Research Europe                         1
Wellcome Open Research                       1
AgriRxiv                                     1
medRxiv                                      1
eLife                                        1
TechRxiv                                     1
L

In [121]:
dupes[dupes['server_name']=='ResearchGate']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
3114562,datacite::10.13140/rg.2.1.1007.9848,ResearchGate,datacite,10.13140/rg.2.1.1007.9848,https://doi.org/10.13140/rg.2.1.1007.9848,https://www.researchgate.net/doi/10.13140/RG.2...,Text,Preprint,classEx - an online software for classroom exp...,"Giamattei, Marcus; Lambsdorff, Johann Graf",2015.0,2015-07-20,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""rg.rg"", ""type"": ""c...",parent,https://www.researchgate.net/doi/10.13140/rg.2...,10.13140/rg.2.1.1007.9848,None,None,NaN,None,None,NaN
3114591,datacite::10.13140/rg.2.1.3434.4162,ResearchGate,datacite,10.13140/rg.2.1.3434.4162,https://doi.org/10.13140/rg.2.1.3434.4162,https://www.researchgate.net/doi/10.13140/RG.2...,Text,Preprint,Is the financial sector Luxembourg's engine of...,"Guarda, Paolo Wolf; Abdelaziz Rouabah",2015.0,2015-08-31,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""rg.rg"", ""type"": ""c...",parent,https://www.researchgate.net/doi/10.13140/rg.2...,10.13140/rg.2.1.3434.4162,None,None,NaN,None,None,NaN
3114566,datacite::10.13140/rg.2.1.3572.0802,ResearchGate,datacite,10.13140/rg.2.1.3572.0802,https://doi.org/10.13140/rg.2.1.3572.0802,https://www.researchgate.net/doi/10.13140/RG.2...,Text,Preprint,Labor informality: choice or sign of segmentat...,"Cruz, Gustavo Adolfo García",2015.0,2015-07-21,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""rg.rg"", ""type"": ""c...",parent,https://www.researchgate.net/doi/10.13140/rg.2...,10.13140/rg.2.1.3572.0802,None,None,NaN,None,None,NaN
3114650,datacite::10.13140/rg.2.1.4278.2806,ResearchGate,datacite,10.13140/rg.2.1.4278.2806,https://doi.org/10.13140/rg.2.1.4278.2806,https://www.researchgate.net/doi/10.13140/RG.2...,Text,Preprint,The innovation‐trade nexus: Italy in historica...,"Domini, Giacomo",2016.0,2016-01-05,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""rg.rg"", ""type"": ""c...",parent,https://www.researchgate.net/doi/10.13140/rg.2...,10.13140/rg.2.1.4278.2806,None,None,NaN,None,None,NaN
3114551,datacite::10.13140/rg.2.1.4429.0729/1,ResearchGate,datacite,10.13140/rg.2.1.4429.0729/1,https://doi.org/10.13140/rg.2.1.4429.0729/1,https://www.researchgate.net/doi/10.13140/RG.2...,Text,Preprint,Private Debt Overhang and the Government Spend...,"Bernardini, Marco; Peersman, Gert",2015.0,2015-06-04,"[{""relatedIdentifier"": ""10.13140/rg.2.1.4429.0...",1,10.13140/rg.2.1.4429.0729,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""rg.rg"", ""type"": ""c...",parent,https://www.researchgate.net/doi/10.13140/rg.2...,10.13140/rg.2.1.4429.0729/1,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3209564,datacite::10.13140/rg.2.2.36722.17602,ResearchGate,datacite,10.13140/rg.2.2.36722.17602,https://doi.org/10.13140/rg.2.2.36722.17602,https://www.researchgate.net/doi/10.13140/RG.2...,Text,Preprint,"PSE operators of the paper ""Influence of free-...",T C L Fava; B A Lobo; P A S Nogueira; A P Scha...,2023.0,2023-04-19,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""rg.rg"", ""type"": ""c...",parent,https://www.researchgate.net/doi/10.13140/rg.2...,10.13140/rg.2.2.36722.17602,None,None,NaN,None,None,NaN
3148600,datacite::10.13140/rg.2.2.36726.75848,ResearchGate,datacite,10.13140/rg.2.2.36726.75848,https://doi.org/10.13140/rg.2.2.36726.75848,https://www.researchgate.net/doi/10.13140/RG.2...,Text,Preprint,Generalised Impulse Response Function as a Per...,"Ajevski

In [122]:
df[df['doi']=='10.13140/rg.2.2.36331.69924']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
3142404,datacite::10.13140/rg.2.2.36331.69924,ResearchGate,datacite,10.13140/rg.2.2.36331.69924,https://doi.org/10.13140/rg.2.2.36331.69924,https://www.researchgate.net/doi/10.13140/RG.2...,Text,Preprint,"TGD view about homeopathy, water memory, and e...",M Pitkänen,2014.0,2019-09-19,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""rg.rg"", ""type"": ""c...",parent,https://www.researchgate.net/doi/10.13140/rg.2...,10.13140/rg.2.2.36331.69924,None,None,NaN,None,None,NaN


In [123]:
df[df['doi']=='10.22004/ag.econ.133088']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
104486,datacite::10.22004/ag.econ.133088,AgEcon Search,datacite,10.22004/ag.econ.133088,https://doi.org/10.22004/ag.econ.133088,https://ageconsearch.umn.edu/record/133088,Text,Text,Farmland price bubbles: wavelet-based evidence,"Power, Gabriel J.; Turvey, Calum G.",2006.0,2019-08-30,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""tind.agecon"", ""typ...",parent,https://ageconsearch.umn.edu/record/133088,10.22004/ag.econ.133088,None,None,NaN,None,None,NaN
2598785,openalex::W985302891,RePEc: Research Papers in Economics,openalex,10.22004/ag.econ.133088,https://doi.org/10.22004/ag.econ.133088,https://ageconsearch.umn.edu/record/133088/fil...,preprint,None,Farmland price bubbles: wavelet-based evidence,Gabriel J. Power; Calum G. Turvey,2006.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://ageconsearch.umn.edu/record/133088/fil...,10.22004/ag.econ.133088,None,None,NaN,None,None,NaN


In [124]:
df[df['doi']=='10.1101/2019.12.23.887166']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
9601114,crossref::10.1101/2019.12.23.887166,bioRxiv,crossref,10.1101/2019.12.23.887166,https://doi.org/10.1101/2019.12.23.887166,http://biorxiv.org/lookup/doi/10.1101/2019.12....,posted-content,preprint,Model balancing: in search of consistent metab...,"Liebermeister, Wolfram; Noor, Elad",2019.0,2019-12-24,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.3390/metabo11110749,,,true,None,None,None,None,parent,http://biorxiv.org/lookup/doi/10.1101/2019.12....,10.1101/2019.12.23.887166,None,None,NaN,None,None,NaN
1513184,openalex::W2995005865,HAL,openalex,10.1101/2019.12.23.887166,https://doi.org/10.1101/2019.12.23.887166,https://hal.science/hal-02437604,preprint,None,Model balancing: consistent in-vivo kinetic co...,Wolfram Liebermeister,2019.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://hal.science/hal-02437604,10.1101/2019.12.23.887166,None,None,NaN,None,None,NaN


In [125]:
df[df['doi']=='10.1101/2020.02.03.919597']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
9509051,crossref::10.1101/2020.02.03.919597,bioRxiv,crossref,10.1101/2020.02.03.919597,https://doi.org/10.1101/2020.02.03.919597,http://biorxiv.org/lookup/doi/10.1101/2020.02....,posted-content,preprint,Germline inherited small RNAs clear untranslat...,"Quarato, Piergiuseppe; Singh, Meetali; Cornes,...",2020.0,2020-02-04,None,None,,,,,false,None,None,None,None,parent,http://biorxiv.org/lookup/doi/10.1101/2020.02....,10.1101/2020.02.03.919597,None,None,NaN,None,None,NaN
1689520,openalex::W3004251285,HAL,openalex,10.1101/2020.02.03.919597,https://doi.org/10.1101/2020.02.03.919597,https://pasteur.hal.science/pasteur-02626442,preprint,None,Argonaute catalytic activity is required for m...,Piergiuseppe Quarato; Meetali Singh; Eric Corn...,2020.0,2020-02-07T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://pasteur.hal.science/pasteur-02626442,10.1101/2020.02.03.919597,None,None,NaN,None,None,NaN


In [126]:
df[df['doi']=='10.3220/rep_20_1_2014']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi


In [127]:
df[df['doi']=='10.48550/arxiv.0704.0324']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
7464076,datacite::10.48550/arxiv.0704.0324,arXiv,datacite,10.48550/arxiv.0704.0324,https://doi.org/10.48550/arxiv.0704.0324,https://arxiv.org/abs/0704.0324,Preprint,Article,On the pseudospectrum of elliptic quadratic di...,"Pravda-Starov, Karel",2007.0,2022-03-16,[],1,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""arxiv.content"", ""t...",parent,https://arxiv.org/abs/0704.0324,10.48550/arxiv.0704.0324,None,None,NaN,None,None,NaN
598673,openalex::W2951243300,HAL,openalex,10.48550/arxiv.0704.0324,https://doi.org/10.48550/arxiv.0704.0324,https://hal.science/hal-00139490,preprint,None,On the pseudospectrum of elliptic quadratic di...,Karel Pravda‐Starov,2007.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://hal.science/hal-00139490,10.48550/arxiv.0704.0324,None,None,NaN,None,None,NaN


In [128]:
df[df['doi']=='10.13140/2.1.2910.4001']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
3114494,datacite::10.13140/2.1.2910.4001,ResearchGate,datacite,10.13140/2.1.2910.4001,https://doi.org/10.13140/2.1.2910.4001,https://www.researchgate.net/doi/10.13140/2.1....,Text,Preprint,"Polymer physics, the quantum harmonic oscillat...",P R Silva,2014.0,2014-08-27,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""rg.rg"", ""type"": ""c...",parent,https://www.researchgate.net/doi/10.13140/2.1....,10.13140/2.1.2910.4001,None,None,NaN,None,None,NaN


In [129]:
df['records_hierarchy'].value_counts()

records_hierarchy
parent              7839852
version              102326
publish_version       12888
part_of                5922
child                  2061
correction              354
comment                 242
review                   27
others                   12
parent_duplicate          4
Name: count, dtype: int64

#### function

In [130]:
import pandas as pd

# -----------------------------
# 0) Define primary servers
# -----------------------------
# PRIMARY_SERVERS = {
#     "arXiv",
#     "bioRxiv",
#     "medRxiv",
#     "SSRN",
#     "TechRxiv",
#     "Wellcome Open Research",
#     "AgriRxiv",
#     "EarthArXiv",
#     "Law Archive",
#     "SocArXiv",
#     "Thesis Commons",
#     "Research Square",
#     "Open Research Europe",
#     "Oroboros Instruments",
#     "ResearchGate",
#     "AgEcon Search",
#     "Zenodo",
#     "Open Science Framework",
#     "Humanities Commons CORE",
#     "CERN document server",
#     "eLife",
# }

PRIMARY_SERVERS = set(primary_servers)  
# -----------------------------
# 1) Normalize DOI
# -----------------------------
doi_norm = (
    df_mirror["doi"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({"": pd.NA, "none": pd.NA, "nan": pd.NA, "null": pd.NA})
)

# -----------------------------
# 2) Flags
# -----------------------------
is_primary = df_mirror["server_name"].isin(PRIMARY_SERVERS)
is_dup = doi_norm.notna() & doi_norm.duplicated(keep=False)

# -----------------------------
# 3) Identify DOI → primary server mapping (ONLY ONCE)
# -----------------------------
primary_by_doi = (
    df_mirror.loc[is_primary & is_dup, ["server_name"]]
    .assign(doi=doi_norm[is_primary & is_dup])
    .dropna(subset=["doi"])
    .groupby("doi")["server_name"]
    .first()   # take first primary (fast & deterministic)
)

# -----------------------------
# 4) Mark mirror rows
# -----------------------------
mask_mirror = is_dup & ~is_primary & doi_norm.isin(primary_by_doi.index)

df_mirror.loc[mask_mirror, "records_hierarchy"] = (
    "mirror (" + doi_norm[mask_mirror].map(primary_by_doi) + ")"
)


# primary_origin = doi_norm[mask_mirror].map(primary_by_doi).fillna("unknown").astype(str)

# df_mirror.loc[mask_mirror, "records_hierarchy"] = "mirror (" + primary_origin + ")"

# -----------------------------
# 5) Sanity check
# -----------------------------
print("Mirror rows:", mask_mirror.sum())
print(df_mirror["records_hierarchy"].value_counts(dropna=False).head(20))


Mirror rows: 13936
records_hierarchy
parent                              7825924
version                              102319
publish_version                       12888
mirror (AgEcon Search)                 6446
mirror (arXiv)                         6375
part_of                                5921
NaN                                    2593
child                                  2061
mirror (ResearchGate)                   827
correction                              354
comment                                 242
mirror (Zenodo)                         191
mirror (bioRxiv)                         29
review                                   27
mirror (Open Science Framework)          21
mirror (SSRN)                            17
mirror (Humanities Commons CORE)         16
others                                   12
parent_duplicate                          4
mirror (EarthArXiv)                       2
Name: count, dtype: int64


In [131]:
df_mirror['records_hierarchy'].value_counts()

records_hierarchy
parent                              7825924
version                              102319
publish_version                       12888
mirror (AgEcon Search)                 6446
mirror (arXiv)                         6375
part_of                                5921
child                                  2061
mirror (ResearchGate)                   827
correction                              354
comment                                 242
mirror (Zenodo)                         191
mirror (bioRxiv)                         29
review                                   27
mirror (Open Science Framework)          21
mirror (SSRN)                            17
mirror (Humanities Commons CORE)         16
others                                   12
parent_duplicate                          4
mirror (EarthArXiv)                       2
mirror (AfricArXiv)                       2
mirror (Research Square)                  2
mirror (AgriRxiv)                         1
mirror (SocArX

In [132]:
dupes[dupes['server_name']=='EconStor Preprints']#.tail(60)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi


In [133]:
df_mirror[df_mirror['doi']=='10.1007/s10273-008-0850-2']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi


In [134]:
# pattern = "osf.io"


# mask = ~dupes[dupes['server_name']=='Open Science Framework']['doi'].str.contains(pattern, regex=False, na=False)
# result = dupes[dupes['server_name']=='Open Science Framework'][mask]
# result

In [135]:
df_mirror[df_mirror['doi']=='10.1101/2019.12.23.887166']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
9601114,crossref::10.1101/2019.12.23.887166,bioRxiv,crossref,10.1101/2019.12.23.887166,https://doi.org/10.1101/2019.12.23.887166,http://biorxiv.org/lookup/doi/10.1101/2019.12....,posted-content,preprint,Model balancing: in search of consistent metab...,"Liebermeister, Wolfram; Noor, Elad",2019.0,2019-12-24,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.3390/metabo11110749,,,true,None,None,None,None,parent,http://biorxiv.org/lookup/doi/10.1101/2019.12....,10.1101/2019.12.23.887166,None,None,NaN,None,None,NaN
1513184,openalex::W2995005865,HAL,openalex,10.1101/2019.12.23.887166,https://doi.org/10.1101/2019.12.23.887166,https://hal.science/hal-02437604,preprint,None,Model balancing: consistent in-vivo kinetic co...,Wolfram Liebermeister,2019.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,mirror (bioRxiv),https://hal.science/hal-02437604,10.1101/2019.12.23.887166,None,None,NaN,None,None,NaN


In [136]:
pattern = "mirror"


mask = df_mirror[df_mirror['server_name']=='F1000Research']['records_hierarchy'].str.contains(pattern, regex=False, na=False)
result = df_mirror[df_mirror['server_name']=='F1000Research'][mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi


In [137]:
pattern = "mirror"


mask = df_mirror['records_hierarchy'].str.contains(pattern, regex=False, na=False)
result = df_mirror[mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
2505576,openalex::W100680786,RePEc: Research Papers in Economics,openalex,10.22004/ag.econ.50567,https://doi.org/10.22004/ag.econ.50567,https://ageconsearch.umn.edu/record/50567,preprint,None,The Impact of Oil Prices on the U.S. and Mexic...,Owen Wagner,2009.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,mirror (AgEcon Search),https://ageconsearch.umn.edu/record/50567,10.22004/ag.econ.50567,None,None,NaN,None,None,NaN
2505641,openalex::W101709879,RePEc: Research Papers in Economics,openalex,10.22004/ag.econ.131335,https://doi.org/10.22004/ag.econ.131335,https://ageconsearch.umn.edu/record/131335/fil...,preprint,None,Explaining Farmland Price Dynamics,Madhab R. Khoju; Bruce L. Ahrendsen,1993.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,mirror (AgEcon Search),https://ageconsearch.umn.edu/record/131335/fil...,10.22004/ag.econ.131335,None,None,NaN,None,None,NaN
2748293,openalex::W1018621947,RePEc: Research Papers in Economics,openalex,10.22004/ag.econ.187970,https://doi.org/10.22004/ag.econ.187970,https://econpapers.repec.org/RePEc:ags:cars09:...,preprint,None,Technical efficiency in the agricultural secto...,Carel Ligeon; Curtis M. Jolly,2010.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,mirror (AgEcon Search),https://econpapers.repec.org/repec:ags:cars09:...,10.22004/ag.econ.187970,None,None,NaN,None,None,NaN
2612494,openalex::W102085527,RePEc: Research Papers in Economics,openalex,10.22004/ag.econ.134965,https://doi.org/10.22004/ag.econ.134965,https://ageconsearch.umn.edu/record/134965,preprint,None,Estimating the Elasticity of Demand and the Pr...,James Breen; Daragh Clancy; Trevor Donnellan; ...,2012.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,mirror (AgEcon Search),https://ageconsearch.umn.edu/record/134965,10.22004/ag.econ.134965,None,None,NaN,None,None,NaN
2748434,openalex::W1031117948,RePEc: Research Papers in Economics,openalex,10.22004/ag.econ.150993,https://doi.org/10.22004/ag.econ.150993,https://ageconsearch.umn.edu/bitstream/150993/...,preprint,None,OWN AND CROSS PASS-THROUGH IN A STRUCTURAL FRA...,Vardges Hovhannisyan; Marin Božić,2013.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,mirror (AgEcon Search),https://ageconsearch.umn.edu/bitstream/150993/...,10.22004/ag.econ.150993,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2505509,openalex::W974745794,RePEc: Research Papers in Economics,openalex,10.22004/ag.econ.123005,https://doi.org/10.22004/ag.econ.123005,http://publications.dyson.cornell.edu/research...,preprint,None,IDENTIFYING A REDUCED SET OF SALIENT ATTRIBUTE...,Heiko Miles; Steven J. Schwager; John E. Lenz,1994.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,mirror (AgEcon Search),http://publications.dyson.cornell.edu/research...,10.22004/ag.econ.123005,None,None,NaN,None,None,NaN
2598756,openalex::W975694582,RePEc: Research Papers in Economics,openalex,10.22004/ag.econ.358354,https://doi.org/10.22004/ag.econ.358354,https://doi.org/10.22004/ag.econ.358354,article,None,Strategie rozwoju przedsiębiorstw z polskim ka...,Vitali Naumavets,2001.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,mirror (AgEcon Search),https://doi.org/10.22004/ag.econ.358354,10.22004/ag.econ.358354,None,None,NaN,None,None,NaN
2602970,openalex::W98271585,RePEc: Resear

In [138]:
result['server_name'].value_counts()

server_name
RePEc: Research Papers in Economics      9996
HAL                                      3921
Social Science Open Access Repository      19
Name: count, dtype: int64

In [139]:
dupes_mirror_df = df_mirror[df_mirror['doi'].notna()]
dupes_mirror_df

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
399051,crossref::10.1002/essoar.10500000.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10500000.1,https://doi.org/10.1002/essoar.10500000.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Long-term monitoring of land surface phenologi...,"Tsutsumida, Narumasa",2018.0,2019-11-13,None,None,,,,,false,None,None,None,None,parent,https://essopenarchive.org/doi/full/10.1002/es...,10.1002/essoar.10500000.1,None,None,NaN,None,None,NaN
399052,crossref::10.1002/essoar.10500002.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10500002.1,https://doi.org/10.1002/essoar.10500002.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Impact of spatial scale for phenological indic...,"Tsutsumida, Narumasa; Kaduk, Jörg",2018.0,2019-11-13,None,None,,,,,false,None,None,None,None,parent,https://essopenarchive.org/doi/full/10.1002/es...,10.1002/essoar.10500002.1,None,None,NaN,None,None,NaN
399047,crossref::10.1002/essoar.10500004.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10500004.1,https://doi.org/10.1002/essoar.10500004.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Observations of Low Latitude Red Aurora in Mex...,"Gonzalez-Esparza, J. Americo; Cuevas-Cardona, ...",2018.0,2019-11-13,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.1029/2018sw001995,,,true,None,None,None,None,parent,https://essopenarchive.org/doi/full/10.1002/es...,10.1002/essoar.10500004.1,None,None,NaN,None,None,NaN
399073,crossref::10.1002/essoar.10500007.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10500007.1,https://doi.org/10.1002/essoar.10500007.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Pipeline oil fire detection with MODIS active ...,"Ogungbuyi, Michael Gbenga; Martinez, Peter; Ec...",2018.0,2019-11-13,None,None,,,,,false,None,None,None,None,parent,https://essopenarchive.org/doi/full/10.1002/es...,10.1002/essoar.10500007.1,None,None,NaN,None,None,NaN
399068,crossref::10.1002/essoar.10500009.1,Earth and Space Science Open Archive,crossref,10.1002/essoar.10500009.1,https://doi.org/10.1002/essoar.10500009.1,https://essopenarchive.org/doi/full/10.1002/es...,posted-content,preprint,Land Product Validation of MODIS Derived FPAR ...,"Sharp, Iain; Sanchez-Azofeifa, Arturo; Musilek...",2018.0,2019-12-03,None,None,,,,,false,None,None,None,None,parent,https://essopenarchive.org/doi/full/10.1002/es...,10.1002/essoar.10500009.1,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6205812,openalex::W998353979,Social Science Open Access Repository,openalex,10.12759/hsr.35.2010.3.47-78,https://doi.org/10.12759/hsr.35.2010.3.47-78,http://www.ssoar.info/ssoar/handle/document/31068,article,None,"""Gutes Klassenbewusstsein, Parteiverbundenheit...",Sandra Meenzen,2010.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://www.ssoar.info/ssoar/handle/document/31068,10.12759/hsr.35.2010.3.47-78,None,None,NaN,None,None,NaN
6197570,openalex::W999063011,Social Science Open Access Repository,openalex,10.15464/isi.42.2009.1-5,https://doi.org/10.15464/isi.42.2009.1-5,http://www.ssoar.info/ssoar/handle/document/21392,article,None,Jeder fünfte Erwerbstätige ist aus beruflichen...,Silvia Ruppenthal; Detlev Lück,2009.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://www.ssoar.info/s

In [140]:
dupes_mirror = dupes_mirror_df[dupes_mirror_df.duplicated(subset=['doi'], keep=False)]
dupes_mirror

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
9503922,crossref::10.1101/103937,bioRxiv,crossref,10.1101/103937,https://doi.org/10.1101/103937,http://biorxiv.org/lookup/doi/10.1101/103937,posted-content,preprint,LET-99-dependent spatial restriction of active...,"Bouvrais, H.; Chesneau, L.; Pastezeur, S.; Del...",2017.0,2017-01-29,None,None,,,,,false,None,None,None,None,parent,http://biorxiv.org/lookup/doi/10.1101/103937,10.1101/103937,None,None,NaN,None,None,NaN
9601114,crossref::10.1101/2019.12.23.887166,bioRxiv,crossref,10.1101/2019.12.23.887166,https://doi.org/10.1101/2019.12.23.887166,http://biorxiv.org/lookup/doi/10.1101/2019.12....,posted-content,preprint,Model balancing: in search of consistent metab...,"Liebermeister, Wolfram; Noor, Elad",2019.0,2019-12-24,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.3390/metabo11110749,,,true,None,None,None,None,parent,http://biorxiv.org/lookup/doi/10.1101/2019.12....,10.1101/2019.12.23.887166,None,None,NaN,None,None,NaN
9514240,crossref::10.1101/2020.01.22.915215,bioRxiv,crossref,10.1101/2020.01.22.915215,https://doi.org/10.1101/2020.01.22.915215,http://biorxiv.org/lookup/doi/10.1101/2020.01....,posted-content,preprint,Not only compulsivity: The SAPAP3-KO mouse rec...,"Lamothe, H; Schreiweis, C; Lavielle, O; Mallet...",2020.0,2020-01-23,None,None,,,,,false,None,None,None,None,parent,http://biorxiv.org/lookup/doi/10.1101/2020.01....,10.1101/2020.01.22.915215,None,None,NaN,None,None,NaN
9509051,crossref::10.1101/2020.02.03.919597,bioRxiv,crossref,10.1101/2020.02.03.919597,https://doi.org/10.1101/2020.02.03.919597,http://biorxiv.org/lookup/doi/10.1101/2020.02....,posted-content,preprint,Germline inherited small RNAs clear untranslat...,"Quarato, Piergiuseppe; Singh, Meetali; Cornes,...",2020.0,2020-02-04,None,None,,,,,false,None,None,None,None,parent,http://biorxiv.org/lookup/doi/10.1101/2020.02....,10.1101/2020.02.03.919597,None,None,NaN,None,None,NaN
9418678,crossref::10.1101/2020.06.08.118984,bioRxiv,crossref,10.1101/2020.06.08.118984,https://doi.org/10.1101/2020.06.08.118984,http://biorxiv.org/lookup/doi/10.1101/2020.06....,posted-content,preprint,TBPL2/TFIIA complex establishes the maternal t...,"Yu, Changwei; Cvetesic, Nevena; Hisler, Vincen...",2020.0,2020-06-09,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.1038/s41467-020-20239-4,,,true,None,None,None,None,parent,http://biorxiv.org/lookup/doi/10.1101/2020.06....,10.1101/2020.06.08.118984,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2505509,openalex::W974745794,RePEc: Research Papers in Economics,openalex,10.22004/ag.econ.123005,https://doi.org/10.22004/ag.econ.123005,http://publications.dyson.cornell.edu/research...,preprint,None,IDENTIFYING A REDUCED SET OF SALIENT ATTRIBUTE...,Heiko Miles; Steven J. Schwager; John E. Lenz,1994.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,mirror (AgEcon Search),http://publications.dyson.cornell.edu/research...,10.22004/ag.econ.123005,None,None,NaN,None,None,NaN
2598756,openalex::W975694582,RePEc: Research Papers in Economics,openalex,10.22004/ag.econ.358354,https://doi.org/10.22004/ag.econ.358354,https://doi.org/10.22004/ag.econ.358354,article,None,Strategie rozwoju przedsiębiorstw z polskim ka...,Vitali Naumavets,2001.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,mirror (AgEcon Search),https://doi.org/10.22004/ag.econ.358354,10.22004/ag.econ.358354,None,None,NaN,None,None,NaN

In [141]:
dupes_mirror['server_name'].value_counts()

server_name
RePEc: Research Papers in Economics      10472
arXiv                                     7147
AgEcon Search                             6445
HAL                                       4035
ResearchGate                               827
CERN document server                       776
Zenodo                                     191
bioRxiv                                     29
Social Science Open Access Repository       24
Open Science Framework                      21
SSRN                                        17
Humanities Commons CORE                     16
AfricArXiv                                   2
EarthArXiv                                   2
Research Square                              2
Open Research Europe                         1
Wellcome Open Research                       1
AgriRxiv                                     1
medRxiv                                      1
eLife                                        1
TechRxiv                                     1
L

In [142]:
dupes_mirror['records_hierarchy'].value_counts()

records_hierarchy
parent                              16074
mirror (AgEcon Search)               6446
mirror (arXiv)                       6375
mirror (ResearchGate)                 827
mirror (Zenodo)                       191
mirror (bioRxiv)                       29
mirror (Open Science Framework)        21
mirror (SSRN)                          17
mirror (Humanities Commons CORE)       16
version                                 3
mirror (AfricArXiv)                     2
mirror (EarthArXiv)                     2
mirror (Research Square)                2
publish_version                         1
mirror (AgriRxiv)                       1
mirror (SocArXiv)                       1
mirror (Law Archive)                    1
mirror (eLife)                          1
mirror (Thesis Commons)                 1
mirror (Oroboros Instruments)           1
mirror (TechRxiv)                       1
mirror (medRxiv)                        1
Name: count, dtype: int64

In [143]:
df_mirror['records_hierarchy'].value_counts()

records_hierarchy
parent                              7825924
version                              102319
publish_version                       12888
mirror (AgEcon Search)                 6446
mirror (arXiv)                         6375
part_of                                5921
child                                  2061
mirror (ResearchGate)                   827
correction                              354
comment                                 242
mirror (Zenodo)                         191
mirror (bioRxiv)                         29
review                                   27
mirror (Open Science Framework)          21
mirror (SSRN)                            17
mirror (Humanities Commons CORE)         16
others                                   12
parent_duplicate                          4
mirror (EarthArXiv)                       2
mirror (AfricArXiv)                       2
mirror (Research Square)                  2
mirror (AgriRxiv)                         1
mirror (SocArX

In [144]:
data_clean['server_name'].value_counts().head(60)

server_name
arXiv                                                                   2920797
SSRN                                                                    1259256
HAL                                                                     1056415
Research Square                                                          450818
RePEc: Research Papers in Economics                                      389398
bioRxiv                                                                  306948
AgEcon Search                                                            188173
ResearchGate                                                             181231
Zenodo                                                                   166786
Open Science Framework                                                   117171
Preprints.org                                                            115815
medRxiv                                                                   75743
Munich Personal RePEc Archiv

In [145]:
df[df['records_hierarchy']=='parent']['server_name'].value_counts().head(60)

server_name
arXiv                                                                   2920797
SSRN                                                                    1259165
HAL                                                                     1056320
Research Square                                                          401982
RePEc: Research Papers in Economics                                      389112
bioRxiv                                                                  306948
AgEcon Search                                                            188173
ResearchGate                                                             181231
Zenodo                                                                   166784
Open Science Framework                                                   114877
Preprints.org                                                            102231
medRxiv                                                                   75743
Munich Personal RePEc Archiv

In [146]:
df_mirror[df_mirror['records_hierarchy']=='parent']['server_name'].value_counts().head(60)

server_name
arXiv                                                                   2920797
SSRN                                                                    1259165
HAL                                                                     1052400
Research Square                                                          401982
RePEc: Research Papers in Economics                                      379122
bioRxiv                                                                  306948
AgEcon Search                                                            188173
ResearchGate                                                             181231
Zenodo                                                                   166784
Open Science Framework                                                   114877
Preprints.org                                                            102231
medRxiv                                                                   75743
Munich Personal RePEc Archiv

### landing_page_url

In [147]:
df_landing_page_url = df_mirror[df_mirror['landing_page_url'].notna()]
df_landing_page_url = df_landing_page_url[df_landing_page_url['records_hierarchy']=='parent']

In [148]:
dupes_landing_page_url = df_landing_page_url[df_landing_page_url.duplicated(subset=['landing_page_url'], keep=False)]
dupes_landing_page_url

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
530474,crossref::10.12688/f1000research.1-10.v1,F1000Research,crossref,10.12688/f1000research.1-10.v1,https://doi.org/10.12688/f1000research.1-10.v1,http://f1000research.com/articles/1-10/v1,journal-article,None,Murine Tim-1 is excluded from the immunologica...,"Lin, Jean; Chen, Leo; Kane, Lawrence P",2012.0,2013-05-09,None,None,,,,,false,None,None,None,None,parent,http://f1000research.com/articles/1-10/v1,10.12688/f1000research.1-10.v1,/v1,explicit_version,1.0,NaN,NaN,NaN
530450,crossref::10.12688/f1000research.1-12.v1,F1000Research,crossref,10.12688/f1000research.1-12.v1,https://doi.org/10.12688/f1000research.1-12.v1,http://f1000research.com/articles/1-12/v1,journal-article,None,Diversion at the ER: How Plasmodium falciparum...,"Römisch, Karin",2012.0,2013-05-09,None,None,,,,,false,None,None,None,None,parent,http://f1000research.com/articles/1-12/v1,10.12688/f1000research.1-12.v1,/v1,explicit_version,1.0,NaN,NaN,NaN
530451,crossref::10.12688/f1000research.1-2.v1,F1000Research,crossref,10.12688/f1000research.1-2.v1,https://doi.org/10.12688/f1000research.1-2.v1,http://f1000research.com/articles/1-2/v1,journal-article,None,Considerations for clinical read alignment and...,"Oliver, Gavin R",2012.0,2013-05-09,None,None,,,,,false,None,None,None,None,parent,http://f1000research.com/articles/1-2/v1,10.12688/f1000research.1-2.v1,/v1,explicit_version,1.0,NaN,NaN,NaN
530538,crossref::10.12688/f1000research.1-23.v1,F1000Research,crossref,10.12688/f1000research.1-23.v1,https://doi.org/10.12688/f1000research.1-23.v1,http://f1000research.com/articles/1-23/v1,journal-article,None,Female circumcision: Limiting the harm,"Kandil, Mohamed",2012.0,2013-05-09,None,None,,,,,false,None,None,None,None,parent,http://f1000research.com/articles/1-23/v1,10.12688/f1000research.1-23.v1,/v1,explicit_version,1.0,NaN,NaN,NaN
530671,crossref::10.12688/f1000research.1-36.v1,F1000Research,crossref,10.12688/f1000research.1-36.v1,https://doi.org/10.12688/f1000research.1-36.v1,http://f1000research.com/articles/1-36/v1,journal-article,None,Termination of mid-trimester pregnancies: miso...,"Shabana, Ayman; Salah, Hesham; Kandil, Mohamed...",2012.0,2013-05-09,None,None,,,,,false,None,None,None,None,parent,http://f1000research.com/articles/1-36/v1,10.12688/f1000research.1-36.v1,/v1,explicit_version,1.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2988192,openalex::W96404234,RePEc: Research Papers in Economics,openalex,<NA>,None,http://asers.eu/journals/jemt/jemt-issues.html,article,None,DEMAND OF REGIONAL TOURISTS VISITING LAO PEOPL...,Sakkarin Nonthapot; Thanet Wattanakul,2016.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://asers.eu/journals/jemt/jemt-issues.html,<na>,None,None,NaN,None,None,NaN
880252,openalex::W975457731,HAL,openalex,<NA>,None,https://hal.inrae.fr/hal-02774168,preprint,None,Ivorian and Malaysian cocoa supply : a compara...,Françoise Jarrige,1993.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://hal.inrae.fr/hal-02774168,<na>,None,None,NaN,None,None,NaN
2941407,openalex::W985669677,RePEc: Research Papers in Economics,openalex,<NA>,None,https://foresight.forecasters.org/shop/,preprint,None,Nate SilverÕs The Signal and the Noise: Why So...,David Orrell,2013.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://foresight.forecasters.org/shop/,<na>,None,None,NaN,None,None,NaN
2633124,open

In [149]:
dupes_landing_page_url['server_name'].value_counts()

server_name
HAL                                                                     3602
RePEc: Research Papers in Economics                                     1588
ResearchHub                                                              171
Munich Personal RePEc Archive                                            157
AgEcon Search                                                            122
PsyArXiv                                                                 119
TechRxiv                                                                  65
F1000Research                                                             62
arXiv                                                                     44
DSpace@MIT                                                                44
Authorea Inc.                                                             44
Digital Access to Scholarship at Harvard (DASH) (Harvard University)      32
EarthArXiv                                                      

In [150]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='eLife']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi


In [151]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='HAL'].sort_values(by='landing_page_url', ascending=False)


,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
1856332,openalex::W4391556995,HAL,openalex,<NA>,None,https://uphf.hal.science/hal-04427814,preprint,None,Recherche &amp; Conception Centrées sur l’Huma...,Bako Rajaonah,2024.0,2024-02-06T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://uphf.hal.science/hal-04427814,<na>,None,None,NaN,None,None,NaN
1857612,openalex::W4402466855,HAL,openalex,<NA>,None,https://uphf.hal.science/hal-04427814,preprint,None,Recherche et Conception Centrées sur l’Humain ...,Bako Rajaonah,2024.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://uphf.hal.science/hal-04427814,<na>,None,None,NaN,None,None,NaN
1856331,openalex::W4391556994,HAL,openalex,<NA>,None,https://uphf.hal.science/hal-04427807,preprint,None,Human-Centred Research &amp; Design for Inclus...,Bako Rajaonah,2024.0,2024-02-06T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://uphf.hal.science/hal-04427807,<na>,None,None,NaN,None,None,NaN
1857610,openalex::W4402466852,HAL,openalex,<NA>,None,https://uphf.hal.science/hal-04427807,preprint,None,Human-Centred Research and Design for Inclusiv...,Bako Rajaonah,2024.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://uphf.hal.science/hal-04427807,<na>,None,None,NaN,None,None,NaN
1456102,openalex::W4300758384,HAL,openalex,<NA>,None,https://uphf.hal.science/hal-03402279,preprint,None,L’aide à la décision comme cadre de gouvernanc...,Igor Crévits; Laurence Bonnafous; Saïd Hanafi,2015.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://uphf.hal.science/hal-03402279,<na>,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
570623,openalex::W2267354779,HAL,openalex,<NA>,None,http://www.utc.fr/,preprint,None,HNLS : une approche constructiviste de connais...,Nasreddine Bouhaï; Fabien Morvan,2004.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://www.utc.fr/,<na>,None,None,NaN,None,None,NaN
1517672,openalex::W2766068608,HAL,openalex,<NA>,None,http://hal.univ-reunion.fr/hal-01620054,preprint,None,A multi-physics optimization problem in natura...,Delphine Ramalingom; Pierre-Henri Cocquet; Rez...,2017.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hal.univ-reunion.fr/hal-01620054,<na>,None,None,NaN,None,None,NaN
1682624,openalex::W4394975358,HAL,openalex,<NA>,None,http://hal.univ-reunion.fr/hal-01620054,preprint,None,A multi-objective optimization problem in natu...,Delphine Ramalingom; Pierre-Henri Cocquet; Rez...,2018.0,2024-04-21T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hal.univ-reunion.fr/hal-01620054,<na>,None,None,NaN,None,None,NaN
788281,openalex::W4298862399,HAL,openalex,<NA>,None,http://atief.org,preprint,None,Un site web pour l'enseignement interdisciplin...,Sandrine Charles; Michel Ney; Dominique Mouchi...,2003.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://atief.org,<na>,None,None,NaN,None,None,NaN


In [152]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='RePEc: Research Papers in Economics'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
2470845,openalex::W5759233,RePEc: Research Papers in Economics,openalex,<NA>,None,https://www.rimisp.org/wp-content/files_mf/137...,preprint,None,Caracterización de los actores de Chiloé Central,Eduardo Ramı́rez; Félix Modrego; Julie Claire ...,2009.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://www.rimisp.org/wp-content/files_mf/137...,<na>,None,None,NaN,None,None,NaN
2557886,openalex::W2396705024,RePEc: Research Papers in Economics,openalex,<NA>,None,https://www.rimisp.org/wp-content/files_mf/137...,article,None,Caracterización de los actores de Chiloé Central,C Revaz; Aude Favier du Noyer,2009.0,2016-06-24T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://www.rimisp.org/wp-content/files_mf/137...,<na>,None,None,NaN,None,None,NaN
2558499,openalex::W25120768,RePEc: Research Papers in Economics,openalex,<NA>,None,https://www.rba.gov.au/publications/confs/2006...,preprint,None,Wrap-up Discussion,Gary Burtless; James K. Glassman; Adair Turner,2006.0,2016-06-24T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://www.rba.gov.au/publications/confs/2006...,<na>,None,None,NaN,None,None,NaN
2943237,openalex::W1440470170,RePEc: Research Papers in Economics,openalex,<NA>,None,https://www.rba.gov.au/publications/confs/2006...,article,None,Overexpression of MAGE-D4 in colorectal cancer...,Qingmei Zhang; Shu-Jia He; Ning Shen; Bin Luo;...,2014.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://www.rba.gov.au/publications/confs/2006...,<na>,None,None,NaN,None,None,NaN
2460544,openalex::W2418493038,RePEc: Research Papers in Economics,openalex,<NA>,None,https://www.nber.org/chapters/c9003.pdf,article,None,"Introduction to ""Concentrated Corporate Owners...",Felix Sahm,2000.0,2016-06-24T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://www.nber.org/chapters/c9003.pdf,<na>,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2988192,openalex::W96404234,RePEc: Research Papers in Economics,openalex,<NA>,None,http://asers.eu/journals/jemt/jemt-issues.html,article,None,DEMAND OF REGIONAL TOURISTS VISITING LAO PEOPL...,Sakkarin Nonthapot; Thanet Wattanakul,2016.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://asers.eu/journals/jemt/jemt-issues.html,<na>,None,None,NaN,None,None,NaN
2952367,openalex::W1564586308,RePEc: Research Papers in Economics,openalex,<NA>,None,http://asers.eu/journals/jemt/jemt-issues.html,article,None,HOW THE ECONOMIC CRISIS AFFECTS THE ENVIRONMENT,Cristina Barbu,2016.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://asers.eu/journals/jemt/jemt-issues.html,<na>,None,None,NaN,None,None,NaN
2943728,openalex::W1521022285,RePEc: Research Papers in Economics,openalex,<NA>,None,http://asers.eu/journals/jemt/jemt-issues.html,article,None,AN INVESTIGATION INTO MOTIVATIONAL FACTORS THA...,Abu Bashar; Abdelnaser Omran,2011.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://asers.eu/journals/jemt/jemt-issues.html,<na>,None,None,NaN,None,None,NaN
2950061,openalex::W113954667,RePEc: Research Papers in Economics,openalex,<NA>,None,http://asers.eu/journals/jasf/jasf-issues.html,article,None,EFFICIENCY ANALYSIS OF TURKISH BANKING SYSTEM,Ayşe Altıok Yilmaz,2013.0,2025-10-10T00:00:00,None,None,None,None,N

In [153]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='arXiv'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
6966259,datacite::10.48550/arxiv.1911.02734,arXiv,datacite,10.48550/arxiv.1911.02734,https://doi.org/10.48550/arxiv.1911.02734,https://arxiv.org/abs/1911.02734,Preprint,Article,Dipolar condensed atomic mixtures and miscibil...,"Tomio, Lauro; Kumar, Ramavarmaraja Kishor; Gam...",2019.0,2022-02-27,[],2,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""arxiv.content"", ""t...",parent,https://arxiv.org/abs/1911.02734,10.48550/arxiv.1911.02734,None,None,NaN,None,None,NaN
7001093,datacite::10.48550/arxiv.1905.11483,arXiv,datacite,10.48550/arxiv.1905.11483,https://doi.org/10.48550/arxiv.1905.11483,https://arxiv.org/abs/1905.11483,Preprint,Article,Scaling properties of firearm homicides in Bra...,"Deppman, Airton",2019.0,2022-02-28,[],1,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""arxiv.content"", ""t...",parent,https://arxiv.org/abs/1905.11483,10.48550/arxiv.1905.11483,None,None,NaN,None,None,NaN
6999379,datacite::10.48550/arxiv.1903.00148,arXiv,datacite,10.48550/arxiv.1903.00148,https://doi.org/10.48550/arxiv.1903.00148,https://arxiv.org/abs/1903.00148,Preprint,Article,Comment on Daya Bay's definition and use of De...,"Parke, Stephen J.; Funchal, Renata Zukanovich",2019.0,2022-02-28,[],1,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""arxiv.content"", ""t...",parent,https://arxiv.org/abs/1903.00148,10.48550/arxiv.1903.00148,None,None,NaN,None,None,NaN
7023615,datacite::10.48550/arxiv.1901.02401,arXiv,datacite,10.48550/arxiv.1901.02401,https://doi.org/10.48550/arxiv.1901.02401,https://arxiv.org/abs/1901.02401,Preprint,Article,The Buzzard Flock: Dark Energy Survey Syntheti...,"DeRose, Joseph; Wechsler, Risa H.; Becker, Mat...",2019.0,2022-03-01,[],1,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""arxiv.content"", ""t...",parent,https://arxiv.org/abs/1901.02401,10.48550/arxiv.1901.02401,None,None,NaN,None,None,NaN
7033269,datacite::10.48550/arxiv.1812.08127,arXiv,datacite,10.48550/arxiv.1812.08127,https://doi.org/10.48550/arxiv.1812.08127,https://arxiv.org/abs/1812.08127,Preprint,Article,Nuclear Dependence of Transverse Single-Spin A...,"Pate, Stephen",2018.0,2022-03-01,[],1,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""arxiv.content"", ""t...",parent,https://arxiv.org/abs/1812.08127,10.48550/arxiv.1812.08127,None,None,NaN,None,None,NaN
7032980,datacite::10.48550/arxiv.1812.07964,arXiv,datacite,10.48550/arxiv.1812.07964,https://doi.org/10.48550/arxiv.1812.07964,https://arxiv.org/abs/1812.07964,Preprint,Article,Searches for Higgs bosons with dark matter at ...,"Gallinaro, Michele",2018.0,2022-03-01,[],1,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""arxiv.content"", ""t...",parent,https://arxiv.org/abs/1812.07964,10.48550/arxiv.1812.07964,None,None,NaN,None,None,NaN
7029387,datacite::10.48550/arxiv.1812.05819,arXiv,datacite,10.48550/arxiv.1812.05819,https://doi.org/10.48550/arxiv.1812.05819,https://arxiv.org/abs/1812.05819,Preprint,Article,Top quark properties,"Van Mulders, Petra",2018.0,2022-03-01,[],1,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""arxiv.content"", ""t...",parent,https://arxiv.org/abs/1812.05819,10.48550/arxiv.1812.05819,None,None,NaN,None,None,NaN
7037251,datacite::10.48550/arxiv.1811.10215,arXiv,datacite,10.48550/arxiv.1811.10215,https://doi.org/10.48550/arxiv.1811.10215,https://arxiv.org/abs/1811.10215,Preprint,Article,Higgs boson measurements at the LHC,"Unal, Guillaume",2018.0,2022-03-01,[],1,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""arxiv.content"", ""t

In [154]:
df_mirror[df_mirror['landing_page_url']=='https://arxiv.org/abs/1703.02360']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
7142272,datacite::10.48550/arxiv.1703.02360,arXiv,datacite,10.48550/arxiv.1703.02360,https://doi.org/10.48550/arxiv.1703.02360,https://arxiv.org/abs/1703.02360,Preprint,Article,Single-top quark cross-section measurements in...,"Hirschbuehl, Dominic",2017.0,2022-03-04,[],1,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""arxiv.content"", ""t...",parent,https://arxiv.org/abs/1703.02360,10.48550/arxiv.1703.02360,None,None,NaN,None,None,NaN
2009527,openalex::W4300531380,Munich Personal RePEc Archive,openalex,<NA>,None,https://arxiv.org/abs/1703.02360,preprint,None,Single-top quark cross-section measurements in...,D. Hirschbuehl,2017.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://arxiv.org/abs/1703.02360,<na>,None,None,NaN,None,None,NaN


In [155]:
df_mirror[df_mirror['landing_page_url']=='https://www.nber.org/chapters/c9003.pdf']

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
2457161,openalex::W13745406,RePEc: Research Papers in Economics,openalex,<NA>,None,https://www.nber.org/chapters/c9003.pdf,preprint,None,"Introduction to ""Concentrated Corporate Owners...",Randall Mørck,2000.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://www.nber.org/chapters/c9003.pdf,<na>,None,None,NaN,None,None,NaN
2460544,openalex::W2418493038,RePEc: Research Papers in Economics,openalex,<NA>,None,https://www.nber.org/chapters/c9003.pdf,article,None,"Introduction to ""Concentrated Corporate Owners...",Felix Sahm,2000.0,2016-06-24T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://www.nber.org/chapters/c9003.pdf,<na>,None,None,NaN,None,None,NaN


In [156]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='Qeios'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
2412169,crossref::10.32388/subst.test.1,Qeios,crossref,10.32388/subst.test.1,https://doi.org/10.32388/subst.test.1,https://www.qeios.com/tmp-test-subst-1,posted-content,preprint,The History of Chocolate: From Ancient Beginni...,"Marinello, Gabriele",2025.0,2025-08-05,None,None,,,,,false,None,None,None,None,parent,https://www.qeios.com/tmp-test-subst-1,10.32388/subst.test.1,None,None,NaN,None,None,NaN
2411669,crossref::10.32388/subst.test.160056748,Qeios,crossref,10.32388/subst.test.160056748,https://doi.org/10.32388/subst.test.160056748,https://www.qeios.com/tmp-test-subst-1,posted-content,preprint,The History of Chocolate: From Ancient Beginni...,"Marinello, Gabriele",2025.0,2025-03-28,None,None,,,,,false,None,None,None,None,parent,https://www.qeios.com/tmp-test-subst-1,10.32388/subst.test.160056748,None,None,NaN,None,None,NaN
2412175,crossref::10.32388/subst.test.3,Qeios,crossref,10.32388/subst.test.3,https://doi.org/10.32388/subst.test.3,https://www.qeios.com/tmp-test-subst-1,posted-content,preprint,The History of Chocolate: From Ancient Beginni...,"Marinello, Gabriele",2025.0,2025-08-06,None,None,,,,,false,None,None,None,None,parent,https://www.qeios.com/tmp-test-subst-1,10.32388/subst.test.3,None,None,NaN,None,None,NaN
2412224,crossref::10.32388/subst.lyons.modern_cinema.2025,Qeios,crossref,10.32388/subst.lyons.modern_cinema.2025,https://doi.org/10.32388/subst.lyons.modern_ci...,https://qeios.com/read/substack-lyons-snails-i...,posted-content,preprint,Snails in Modern Cinema: From Speed Dreams to ...,"Lyons, Mansel",2025.0,2025-08-29,None,None,,,,,false,None,None,None,None,parent,https://qeios.com/read/substack-lyons-snails-i...,10.32388/subst.lyons.modern_cinema.2025,None,None,NaN,None,None,NaN
2412195,crossref::10.32388/subst.lyons.modern_cinema_s...,Qeios,crossref,10.32388/subst.lyons.modern_cinema_snails.2025,https://doi.org/10.32388/subst.lyons.modern_ci...,https://qeios.com/read/substack-lyons-snails-i...,posted-content,preprint,Snails in Modern Cinema: From Speed Dreams to ...,"Lyons, Mansel",2025.0,2025-08-13,None,None,,,,,false,None,None,None,None,parent,https://qeios.com/read/substack-lyons-snails-i...,10.32388/subst.lyons.modern_cinema_snails.2025,None,None,NaN,None,None,NaN
2412223,crossref::10.32388/subst.lyons.knight_vs_snail...,Qeios,crossref,10.32388/subst.lyons.knight_vs_snail.2025,https://doi.org/10.32388/subst.lyons.knight_vs...,https://qeios.com/read/substack-lyons-knights-...,posted-content,preprint,"Knights vs. Snails: A History of a Tiny, Tenac...","Lyons, Mansel",2025.0,2025-08-29,None,None,,,,,false,None,None,None,None,parent,https://qeios.com/read/substack-lyons-knights-...,10.32388/subst.lyons.knight_vs_snail.2025,None,None,NaN,None,None,NaN
2412194,crossref::10.32388/subst.lyons.knights_vs_snai...,Qeios,crossref,10.32388/subst.lyons.knights_vs_snails.2025,https://doi.org/10.32388/subst.lyons.knights_v...,https://qeios.com/read/substack-lyons-knights-...,posted-content,preprint,"Knights vs. Snails: A History of a Tiny, Tenac...","Lyons, Mansel",2025.0,2025-08-13,None,None,,,,,false,None,None,None,None,parent,https://qeios.com/read/substack-lyons-knights-...,10.32388/subst.lyons.knights_vs_snails.2025,None,None,NaN,None,None,NaN


In [157]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='ResearchHub'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
3296188,crossref::10.55277/researchhub.8i2kymwh,ResearchHub,crossref,10.55277/researchhub.8i2kymwh,https://doi.org/10.55277/researchhub.8i2kymwh,https://www.staging.researchhub.com/post/321/r...,report,None,02.26 Preregistration template,"Tytarenko, Mykola",2025.0,2025-02-26,None,None,,,,,false,None,None,None,None,parent,https://www.staging.researchhub.com/post/321/r...,10.55277/researchhub.8i2kymwh,None,None,NaN,None,None,NaN
3296190,crossref::10.55277/researchhub.7ewvh0rf,ResearchHub,crossref,10.55277/researchhub.7ewvh0rf,https://doi.org/10.55277/researchhub.7ewvh0rf,https://www.staging.researchhub.com/post/321/r...,report,None,02.26 Preregistration template v 3,"Tytarenko, Mykola",2025.0,2025-02-26,None,None,,,,,false,None,None,None,None,parent,https://www.staging.researchhub.com/post/321/r...,10.55277/researchhub.7ewvh0rf,None,None,NaN,None,None,NaN
3296189,crossref::10.55277/researchhub.2ihlexrg,ResearchHub,crossref,10.55277/researchhub.2ihlexrg,https://doi.org/10.55277/researchhub.2ihlexrg,https://www.staging.researchhub.com/post/321/r...,report,None,02.26 Preregistration template,"Tytarenko, Mykola",2025.0,2025-02-26,None,None,,,,,false,None,None,None,None,parent,https://www.staging.researchhub.com/post/321/r...,10.55277/researchhub.2ihlexrg,None,None,NaN,None,None,NaN
3296187,crossref::10.55277/researchhub.lrtyw6ah,ResearchHub,crossref,10.55277/researchhub.lrtyw6ah,https://doi.org/10.55277/researchhub.lrtyw6ah,https://www.staging.researchhub.com/post/321/r...,report,None,Research Article Title 2,"Tytarenko, Mykola",2025.0,2025-02-26,None,None,,,,,false,None,None,None,None,parent,https://www.staging.researchhub.com/post/321/r...,10.55277/researchhub.lrtyw6ah,None,None,NaN,None,None,NaN
3296387,crossref::10.55277/rhj.8ksztm1x,ResearchHub,crossref,10.55277/rhj.8ksztm1x,https://doi.org/10.55277/rhj.8ksztm1x,https://www.staging.researchhub.com/paper/3236...,report,None,In the test journal,"K, Taki",2025.0,2025-05-29,None,None,,,,,false,None,None,None,None,parent,https://www.staging.researchhub.com/paper/3236...,10.55277/rhj.8ksztm1x,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3296119,crossref::10.55277/researchhub.s485ohw7.1,ResearchHub,crossref,10.55277/researchhub.s485ohw7.1,https://doi.org/10.55277/researchhub.s485ohw7.1,https://www.researchhub.com/paper/8437957/fini...,report,None,Finite Density Black Holes in a Quantum Gravit...,"Brown, Jesse Daniel; Smith, McCade",2025.0,2025-01-09,None,None,,,,,false,None,None,None,None,parent,https://www.researchhub.com/paper/8437957/fini...,10.55277/researchhub.s485ohw7.1,None,None,NaN,None,None,NaN
3296055,crossref::10.55277/researchhub.71mugn7l,ResearchHub,crossref,10.55277/researchhub.71mugn7l,https://doi.org/10.55277/researchhub.71mugn7l,https://www.researchhub.com/paper/8420046/alte...,report,None,Alternate EoS for Finite Density Black Holes i...,"Brown, Jesse Daniel; Smith, McCade",2024.0,2024-11-21,None,None,,,,,false,None,None,None,None,parent,https://www.researchhub.com/paper/8420046/alte...,10.55277/researchhub.71mugn7l,None,None,NaN,None,None,NaN
3296114,crossref::10.55277/researchhub.71mugn7l.1,ResearchHub,crossref,10.55277/researchhub.71mugn7l.1,https://doi.org/10.55277/researchhub.71mugn7l.1,https://www.researchhub.com/paper/8420046/alte...,report,None,Alternate EoS for Finite Density Black Holes i...,"Brown, Jesse Daniel; Smith, McCade",2025.0,2025-01-09,None,None,,,,,false,None,None,None,None,parent,https

In [158]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='Munich Personal RePEc Archive'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
2009376,openalex::W4288027439,Munich Personal RePEc Archive,openalex,<NA>,None,https://arxiv.org/abs/1911.02734,preprint,None,Dipolar condensed atomic mixtures and miscibil...,Lauro Tomio; R. Kishor Kumar; A. Gammal,2019.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://arxiv.org/abs/1911.02734,<na>,None,None,NaN,None,None,NaN
2009381,openalex::W4288347547,Munich Personal RePEc Archive,openalex,<NA>,None,https://arxiv.org/abs/1905.11483,preprint,None,Scaling properties of firearms homicides in br...,A. Deppman,2019.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://arxiv.org/abs/1905.11483,<na>,None,None,NaN,None,None,NaN
2009384,openalex::W4288555292,Munich Personal RePEc Archive,openalex,<NA>,None,https://arxiv.org/abs/1903.00148,preprint,None,Comment on daya bay's de nition and use of 'de...,Stephen Parke; R. Zukanovich Funchal,2019.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://arxiv.org/abs/1903.00148,<na>,None,None,NaN,None,None,NaN
1980910,openalex::W4297781979,Munich Personal RePEc Archive,openalex,<NA>,None,https://arxiv.org/abs/1901.02401,preprint,None,The buzzard flock: dark energy survey syntheti...,Joseph DeRose; Risa H. Wechsler; M. R. Becker;...,2019.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://arxiv.org/abs/1901.02401,<na>,None,None,NaN,None,None,NaN
2009392,openalex::W4289106760,Munich Personal RePEc Archive,openalex,<NA>,None,https://arxiv.org/abs/1812.08127,preprint,None,Nuclear dependence of transverse single-spin a...,S. F. Pate,2018.0,2022-08-01T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://arxiv.org/abs/1812.08127,<na>,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1993860,openalex::W82458165,Munich Personal RePEc Archive,openalex,<NA>,None,http://www.theses.fr/2010INPT0096/document,preprint,None,"Gestion autonomique de performance, d'énergie ...",Rémi Sharrock,2010.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://www.theses.fr/2010inpt0096/document,<na>,None,None,NaN,None,None,NaN
2048163,openalex::W4395677280,Munich Personal RePEc Archive,openalex,<NA>,None,http://www.theses.fr/2010INPT0029/document,preprint,None,Architectures innovantes de systèmes de comman...,Manel Sghairi Haouati,2010.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://www.theses.fr/2010inpt0029/document,<na>,None,None,NaN,None,None,NaN
2021984,openalex::W4392339766,Munich Personal RePEc Archive,openalex,<NA>,None,http://www.theses.fr/2010INPT0029/document,preprint,None,Innovative Architectures of Flight Control Sys...,Manel Sghairi Haouati,2010.0,2024-03-05T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://www.theses.fr/2010inpt0029/document,<na>,None,None,NaN,None,None,NaN
1997529,openalex::W20541881,Munich Personal RePEc Archive,openalex,<NA>,None,http://www.theses.fr/2010CLF22073/document,preprint,None,"Synthèse d'aminocyclitols, inhibiteurs potenti...",Flora Camps Bres,2010.0,2016-06-24T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://www.theses.fr/2010clf22073/document,<na>,None,None,NaN,None,None,NaN


In [159]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='AgEcon Search'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
59301,datacite::10.22004/ag.econ.98631,AgEcon Search,datacite,10.22004/ag.econ.98631,https://doi.org/10.22004/ag.econ.98631,https://ageconsearch.umn.edu/record/98631,Text,Text,What do Haitians need after the earthquake?,"Andre, Rock; Lusk, Jayson L.",2011.0,2019-08-24,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""tind.agecon"", ""typ...",parent,https://ageconsearch.umn.edu/record/98631,10.22004/ag.econ.98631,None,None,NaN,None,None,NaN
58155,datacite::10.22004/ag.econ.95334,AgEcon Search,datacite,10.22004/ag.econ.95334,https://doi.org/10.22004/ag.econ.95334,https://ageconsearch.umn.edu/record/95334,Text,Text,Análisis de la ampliación de los recursos loca...,"Marin-Sanchez, Maria Del Mar",2010.0,2019-08-24,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""tind.agecon"", ""typ...",parent,https://ageconsearch.umn.edu/record/95334,10.22004/ag.econ.95334,None,None,NaN,None,None,NaN
58152,datacite::10.22004/ag.econ.95331,AgEcon Search,datacite,10.22004/ag.econ.95331,https://doi.org/10.22004/ag.econ.95331,https://ageconsearch.umn.edu/record/95331,Text,Text,LAS VIAS PECUARIAS Y LA PLANIFICACION TERRITORIAL,"Guaita Pradas, Inmaculada; Barrachina Martinez...",2010.0,2019-08-24,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""tind.agecon"", ""typ...",parent,https://ageconsearch.umn.edu/record/95331,10.22004/ag.econ.95331,None,None,NaN,None,None,NaN
58089,datacite::10.22004/ag.econ.95214,AgEcon Search,datacite,10.22004/ag.econ.95214,https://doi.org/10.22004/ag.econ.95214,https://ageconsearch.umn.edu/record/95214,Text,Text,Los cambios en la división internacional del t...,"Pensado Leglise, Mario Del Roble",2010.0,2019-08-24,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""tind.agecon"", ""typ...",parent,https://ageconsearch.umn.edu/record/95214,10.22004/ag.econ.95214,None,None,NaN,None,None,NaN
19699,datacite::10.22004/ag.econ.6543,AgEcon Search,datacite,10.22004/ag.econ.6543,https://doi.org/10.22004/ag.econ.6543,https://ageconsearch.umn.edu/record/6543,Text,Text,School District and Municipal Reorganization: ...,"Scorsone, Eric",2007.0,2019-08-23,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""tind.agecon"", ""typ...",parent,https://ageconsearch.umn.edu/record/6543,10.22004/ag.econ.6543,None,None,NaN,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60304,datacite::10.22004/ag.econ.103153,AgEcon Search,datacite,10.22004/ag.econ.103153,https://doi.org/10.22004/ag.econ.103153,https://ageconsearch.umn.edu/record/103153,Text,Text,"PRODUÇÃO DE FLORES EM UMUARAMA, NOROESTE DO PA...","Morita, Daniela Alves Dos Santos; Dias-Arieira...",2008.0,2019-08-24,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""tind.agecon"", ""typ...",parent,https://ageconsearch.umn.edu/record/103153,10.22004/ag.econ.103153,None,None,NaN,None,None,NaN
60294,datacite::10.22004/ag.econ.103117,AgEcon Search,datacite,10.22004/ag.econ.103117,https://doi.org/10.22004/ag.econ.103117,https://ageconsearch.umn.edu/record/103117,Text,Text,A Trajetória das Pesquisas com Práticas Agríco...,"Borges Filho, Epaminondas Luiz",2008.0,2019-08-24,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""tind.agecon"", ""typ...",parent,https://ageconsearch.umn.edu/record/103117,10.22004/ag.econ.103117,None,None,NaN,None,None,NaN
60258,datacite::10.22004/ag.econ.102769,AgEcon Search,datacite,10.22004/ag.econ.102769,https:

In [160]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='PsyArXiv'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
2357889,crossref::10.31234/osf.io/zq3fa_v1,PsyArXiv,crossref,10.31234/osf.io/zq3fa_v1,https://doi.org/10.31234/osf.io/zq3fa_v1,https://osf.io/zq3fa_v1,posted-content,preprint,Raising the Bar: Improving Methodological Rigo...,"Pennington, Charlotte Rebecca; Jones, Andrew; ...",2021.0,2025-05-26,None,None,,,,,false,None,None,None,None,parent,https://osf.io/zq3fa_v1,10.31234/osf.io/zq3fa_v1,_v1,explicit_version,1.0,NaN,NaN,NaN
2384658,crossref::10.31234/osf.io/zq3fa,PsyArXiv,crossref,10.31234/osf.io/zq3fa,https://doi.org/10.31234/osf.io/zq3fa,https://osf.io/zq3fa_v1,posted-content,preprint,Raising the Bar: Improving Methodological Rigo...,"Pennington, Charlotte Rebecca; Jones, Andrew; ...",2021.0,2021-06-29,None,None,,,,,false,None,None,None,None,parent,https://osf.io/zq3fa_v1,10.31234/osf.io/zq3fa,_v1,explicit_version,1.0,NaN,NaN,NaN
2358124,crossref::10.31234/osf.io/zphx9_v1,PsyArXiv,crossref,10.31234/osf.io/zphx9_v1,https://doi.org/10.31234/osf.io/zphx9_v1,https://osf.io/zphx9_v1,posted-content,preprint,The adventure of running experiments with teen...,"Alfonso, Antonio; Branas-Garza, Pablo; Jorrat,...",2022.0,2025-06-26,None,None,,,,,false,None,None,None,None,parent,https://osf.io/zphx9_v1,10.31234/osf.io/zphx9_v1,_v1,explicit_version,1.0,NaN,NaN,NaN
2393930,crossref::10.31234/osf.io/zphx9,PsyArXiv,crossref,10.31234/osf.io/zphx9,https://doi.org/10.31234/osf.io/zphx9,https://osf.io/zphx9_v1,posted-content,preprint,The adventure of running experiments with teen...,"Alfonso, Antonio; Branas-Garza, Pablo; Jorrat,...",2022.0,2022-11-14,None,None,,,,,false,None,None,None,None,parent,https://osf.io/zphx9_v1,10.31234/osf.io/zphx9,_v1,explicit_version,1.0,NaN,NaN,NaN
2357945,crossref::10.31234/osf.io/y39xu_v1,PsyArXiv,crossref,10.31234/osf.io/y39xu_v1,https://doi.org/10.31234/osf.io/y39xu_v1,https://osf.io/y39xu_v1,posted-content,preprint,The Effects of Patients’ Expectations on Surge...,"Laferton, Johannes Andreas Christoph; Oeltjen,...",2020.0,2025-06-04,None,None,,,,,false,None,None,None,None,parent,https://osf.io/y39xu_v1,10.31234/osf.io/y39xu_v1,_v1,explicit_version,1.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2358131,crossref::10.31234/osf.io/37a9q_v1,PsyArXiv,crossref,10.31234/osf.io/37a9q_v1,https://doi.org/10.31234/osf.io/37a9q_v1,https://osf.io/37a9q_v1,posted-content,preprint,Access to meaning from visual input: Object an...,"Gregorova, Klara; Turini Volonghi, Jacopo; Gag...",2021.0,2025-06-26,None,None,,,,,false,None,None,None,None,parent,https://osf.io/37a9q_v1,10.31234/osf.io/37a9q_v1,_v1,explicit_version,1.0,NaN,NaN,NaN
2396244,crossref::10.31234/osf.io/2cvk7,PsyArXiv,crossref,10.31234/osf.io/2cvk7,https://doi.org/10.31234/osf.io/2cvk7,https://osf.io/2cvk7_v1,posted-content,preprint,No Evidence that Working Memory Modulates the ...,"Meyers, Elke; Alves, Maryna; Teugels, Anouk; T...",2023.0,2023-01-27,None,None,,,,,false,None,None,None,None,parent,https://osf.io/2cvk7_v1,10.31234/osf.io/2cvk7,_v1,explicit_version,1.0,NaN,NaN,NaN
2358110,crossref::10.31234/osf.io/2cvk7_v1,PsyArXiv,crossref,10.31234/osf.io/2cvk7_v1,https://doi.org/10.31234/osf.io/2cvk7_v1,https://osf.io/2cvk7_v1,posted-content,preprint,No Evidence that Working Memory Modulates the ...,"Meyers, Elke; alves, maryna; Teugels, Anouk; T...",2023.0,2025-06-24,None,None,,,,,false,None,None,None,None,parent,https://osf.io/2cvk7_v1,10.31234/osf.io/2cvk7_v1,_v1,explicit_version,1.0,NaN,NaN,NaN
2358051,crossref::10.312

In [161]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='Earth and Space Science Open Archive'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
401215,crossref::10.22541/essoar.169720482.25977228/v1,Earth and Space Science Open Archive,crossref,10.22541/essoar.169720482.25977228/v1,https://doi.org/10.22541/essoar.169720482.2597...,https://essopenarchive.org/users/672150/articl...,posted-content,preprint,Long-term trends in the distribution of ocean ...,"Zhai, Dongran; Beaulieu, Claudie; Kudela, Raph...",2023.0,2023-10-13,None,None,,,,,false,None,None,None,None,parent,https://essopenarchive.org/users/672150/articl...,10.22541/essoar.169720482.25977228/v1,None,None,NaN,/v1,explicit_version,1.0
401178,crossref::10.22541/essoar.169711704.45388674/v1,Earth and Space Science Open Archive,crossref,10.22541/essoar.169711704.45388674/v1,https://doi.org/10.22541/essoar.169711704.4538...,https://essopenarchive.org/users/672150/articl...,posted-content,preprint,Long-term trends in the distribution of ocean ...,"Zhai, Dongran; Beaulieu, Claudie; Kudela, Raph...",2023.0,2023-10-12,None,None,,,,,false,None,None,None,None,parent,https://essopenarchive.org/users/672150/articl...,10.22541/essoar.169711704.45388674/v1,None,None,NaN,/v1,explicit_version,1.0
401168,crossref::10.22541/essoar.169711691.17315936/v1,Earth and Space Science Open Archive,crossref,10.22541/essoar.169711691.17315936/v1,https://doi.org/10.22541/essoar.169711691.1731...,https://essopenarchive.org/users/671111/articl...,posted-content,preprint,Seasonal variability of kelp dissolved organic...,"Carlson, Andrew Kalani; Yoshimura, Takeshi; Ku...",2023.0,2023-10-12,None,None,,,,,false,None,None,None,None,parent,https://essopenarchive.org/users/671111/articl...,10.22541/essoar.169711691.17315936/v1,None,None,NaN,/v1,explicit_version,1.0
401210,crossref::10.22541/essoar.169720309.94232981/v1,Earth and Space Science Open Archive,crossref,10.22541/essoar.169720309.94232981/v1,https://doi.org/10.22541/essoar.169720309.9423...,https://essopenarchive.org/users/671111/articl...,posted-content,preprint,Seasonal variability of kelp dissolved organic...,"Carlson, Andrew Kalani; Yoshimura, Takeshi; Ku...",2023.0,2023-10-13,None,None,,,,,false,None,None,None,None,parent,https://essopenarchive.org/users/671111/articl...,10.22541/essoar.169720309.94232981/v1,None,None,NaN,/v1,explicit_version,1.0
401217,crossref::10.22541/essoar.169722116.61393621/v1,Earth and Space Science Open Archive,crossref,10.22541/essoar.169722116.61393621/v1,https://doi.org/10.22541/essoar.169722116.6139...,https://essopenarchive.org/users/669866/articl...,posted-content,preprint,Projecting Future Chronic Coastal Hazard Impac...,"Leung, Meredith C; Cagigal, Laura; Méndez, Fer...",2023.0,2023-10-13,None,None,,,,,false,None,None,None,None,parent,https://essopenarchive.org/users/669866/articl...,10.22541/essoar.169722116.61393621/v1,None,None,NaN,/v1,explicit_version,1.0
401198,crossref::10.22541/essoar.169711756.69292862/v1,Earth and Space Science Open Archive,crossref,10.22541/essoar.169711756.69292862/v1,https://doi.org/10.22541/essoar.169711756.6929...,https://essopenarchive.org/users/669866/articl...,posted-content,preprint,Projecting Future Chronic Coastal Hazard Impac...,"Leung, Meredith C; Cagigal, Laura; Méndez, Fer...",2023.0,2023-10-12,None,None,,,,,false,None,None,None,None,parent,https://essopenarchive.org/users/669866/articl...,10.22541/essoar.169711756.69292862/v1,None,None,NaN,/v1,explicit_version,1.0
401214,crossref::10.22541/essoar.169720476.67987203/v1,Earth and Space Science Open Archive,crossref,10.22541/essoar.169720476.67987203/v1,https://doi.org/10.22541/essoar.169720476.6798...,https://essop

In [162]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='ScienceOpen Preprints'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi


In [163]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='TechRxiv'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
6249216,crossref::10.36227/techrxiv.174613156.61394999/v1,TechRxiv,crossref,10.36227/techrxiv.174613156.61394999/v1,https://doi.org/10.36227/techrxiv.174613156.61...,https://www.techrxiv.org/users/918312/articles...,posted-content,preprint,Spatio-Temporal Gaze Analysis in VR: Comparing...,"Sun, Xiaoxiao; Shi, Xinran; Koorathota, Sharat...",2025.0,2025-05-01,None,None,,,,,false,None,None,None,None,parent,https://www.techrxiv.org/users/918312/articles...,10.36227/techrxiv.174613156.61394999/v1,None,None,NaN,/v1,explicit_version,1.0
6249142,crossref::10.36227/techrxiv.174594289.98518288/v1,TechRxiv,crossref,10.36227/techrxiv.174594289.98518288/v1,https://doi.org/10.36227/techrxiv.174594289.98...,https://www.techrxiv.org/users/918312/articles...,posted-content,preprint,Spatio-Temporal Gaze Analysis in VR: Comparing...,"Sun, Xiaoxiao; Shi, Xinran; Koorathota, Sharat...",2025.0,2025-04-29,None,None,,,,,false,None,None,None,None,parent,https://www.techrxiv.org/users/918312/articles...,10.36227/techrxiv.174594289.98518288/v1,None,None,NaN,/v1,explicit_version,1.0
6249139,crossref::10.36227/techrxiv.174593999.98386877/v1,TechRxiv,crossref,10.36227/techrxiv.174593999.98386877/v1,https://doi.org/10.36227/techrxiv.174593999.98...,https://www.techrxiv.org/users/917216/articles...,posted-content,preprint,Free-wheeling offline and online identificatio...,"Pfeifer, Bernd; Hackl, Christoph M.",2025.0,2025-04-29,None,None,,,,,false,None,None,None,None,parent,https://www.techrxiv.org/users/917216/articles...,10.36227/techrxiv.174593999.98386877/v1,None,None,NaN,/v1,explicit_version,1.0
6249214,crossref::10.36227/techrxiv.174613113.39438286/v1,TechRxiv,crossref,10.36227/techrxiv.174613113.39438286/v1,https://doi.org/10.36227/techrxiv.174613113.39...,https://www.techrxiv.org/users/917216/articles...,posted-content,preprint,Free-wheeling offline and online identificatio...,"Pfeifer, Bernd; Hackl, Christoph M.",2025.0,2025-05-01,None,None,,,,,false,None,None,None,None,parent,https://www.techrxiv.org/users/917216/articles...,10.36227/techrxiv.174613113.39438286/v1,None,None,NaN,/v1,explicit_version,1.0
6249213,crossref::10.36227/techrxiv.174613072.24623819/v1,TechRxiv,crossref,10.36227/techrxiv.174613072.24623819/v1,https://doi.org/10.36227/techrxiv.174613072.24...,https://www.techrxiv.org/users/916644/articles...,posted-content,preprint,Artificial Intelligence Applied to Risk Manage...,"Uehara, Marcelo Sousa",2025.0,2025-05-01,None,None,,,,,false,None,None,None,None,parent,https://www.techrxiv.org/users/916644/articles...,10.36227/techrxiv.174613072.24623819/v1,None,None,NaN,/v1,explicit_version,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6244928,crossref::10.36227/techrxiv.171925125.59769255/v1,TechRxiv,crossref,10.36227/techrxiv.171925125.59769255/v1,https://doi.org/10.36227/techrxiv.171925125.59...,https://www.techrxiv.org/users/681556/articles...,posted-content,preprint,Electrodynamics for Non-Relativistic Point Cha...,"Kühn, Steffen",2024.0,2024-07-03,None,None,,,,,false,None,None,None,None,parent,https://www.techrxiv.org/users/681556/articles...,10.36227/techrxiv.171925125.59769255/v1,None,None,NaN,/v1,explicit_version,1.0
6244858,crossref::10.36227/techrxiv.171863936.68554917/v1,TechRxiv,crossref,10.36227/techrxiv.171863936.68554917/v1,https://doi.org/10.36227/techrxiv.171863936.68...,https://www.techrxiv.org/users/681556/articles...,posted-content,preprint,Electrodynamics for Non-Relativistic Point Cha..

In [164]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='F1000Research'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
530501,crossref::10.12688/f1000research.3979.1,F1000Research,crossref,10.12688/f1000research.3979.1,https://doi.org/10.12688/f1000research.3979.1,http://f1000research.com/articles/3-94/v1,journal-article,None,Data publication consensus and controversies,"Kratz, John; Strasser, Carly",2014.0,2014-06-17,None,None,,,,,false,None,None,None,None,parent,http://f1000research.com/articles/3-94/v1,10.12688/f1000research.3979.1,/v1,explicit_version,1.0,NaN,NaN,NaN
530503,crossref::10.12688/f1000research.4264,F1000Research,crossref,10.12688/f1000research.4264,https://doi.org/10.12688/f1000research.4264,http://f1000research.com/articles/3-94/v1,journal-article,None,Data publication consensus and controversies,"Kratz, John; Strasser, Carly",2014.0,2014-05-08,None,None,,,,,false,None,None,None,None,parent,http://f1000research.com/articles/3-94/v1,10.12688/f1000research.4264,/v1,explicit_version,1.0,NaN,NaN,NaN
530637,crossref::10.12688/f1000research.4162,F1000Research,crossref,10.12688/f1000research.4162,https://doi.org/10.12688/f1000research.4162,http://f1000research.com/articles/3-91/v1,journal-article,None,Ten things to get right for marine conservatio...,"Weeks, Rebecca; Pressey, Robert L.; Wilson, Jo...",2014.0,2014-05-12,None,None,,,,,false,None,None,None,None,parent,http://f1000research.com/articles/3-91/v1,10.12688/f1000research.4162,/v1,explicit_version,1.0,NaN,NaN,NaN
530636,crossref::10.12688/f1000research.3886.1,F1000Research,crossref,10.12688/f1000research.3886.1,https://doi.org/10.12688/f1000research.3886.1,http://f1000research.com/articles/3-91/v1,journal-article,None,Ten things to get right for marine conservatio...,"Weeks, Rebecca; Pressey, Robert L.; Wilson, Jo...",2014.0,2014-06-17,None,None,,,,,false,None,None,None,None,parent,http://f1000research.com/articles/3-91/v1,10.12688/f1000research.3886.1,/v1,explicit_version,1.0,NaN,NaN,NaN
530406,crossref::10.12688/f1000research.4019,F1000Research,crossref,10.12688/f1000research.4019,https://doi.org/10.12688/f1000research.4019,http://f1000research.com/articles/3-83/v1,journal-article,None,Recommendations to enable drug development for...,"Sames, Lori; Moore, Allison; Arnold, Renee; Ek...",2014.0,2014-04-03,None,None,,,,,false,None,None,None,None,parent,http://f1000research.com/articles/3-83/v1,10.12688/f1000research.4019,/v1,explicit_version,1.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
530451,crossref::10.12688/f1000research.1-2.v1,F1000Research,crossref,10.12688/f1000research.1-2.v1,https://doi.org/10.12688/f1000research.1-2.v1,http://f1000research.com/articles/1-2/v1,journal-article,None,Considerations for clinical read alignment and...,"Oliver, Gavin R",2012.0,2013-05-09,None,None,,,,,false,None,None,None,None,parent,http://f1000research.com/articles/1-2/v1,10.12688/f1000research.1-2.v1,/v1,explicit_version,1.0,NaN,NaN,NaN
530461,crossref::10.3410/f1000research.1-12.v1,F1000Research,crossref,10.3410/f1000research.1-12.v1,https://doi.org/10.3410/f1000research.1-12.v1,http://f1000research.com/articles/1-12/v1,journal-article,None,Diversion at the ER: How Plasmodium falciparum...,"Römisch, Karin",2012.0,2015-06-24,None,None,,,,,false,None,None,None,None,parent,http://f1000research.com/articles/1-12/v1,10.3410/f1000research.1-12.v1,/v1,explicit_version,1.0,NaN,NaN,NaN
530450,crossref::10.12688/f1000research.1-12.v1,F1000Research,crossref,10.12688/f1000research.1-12.v1,https://doi.org/10.12688/f1000research.1-12.v1,http://f1000res

In [165]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='DSpace@MIT'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
338716,openalex::W4297798537,DSpace@MIT,openalex,<NA>,None,https://arxiv.org/abs/1802.04480,preprint,None,RoboChain: A Secure Data-Sharing Framework for...,Eduardo Castelló Ferrer; Ognjen Rudovic; Thoma...,2018.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://arxiv.org/abs/1802.04480,<na>,None,None,NaN,None,None,NaN
338590,openalex::W3102404986,DSpace@MIT,openalex,<NA>,None,http://hdl.handle.net/1721.1/77925,article,None,Coherency Strain and the Kinetics of Phase Sep...,Daniel A. Cogswell; Martin Z. Bazant,2013.0,2020-11-23T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hdl.handle.net/1721.1/77925,<na>,None,None,NaN,None,None,NaN
335603,openalex::W1499052255,DSpace@MIT,openalex,<NA>,None,http://hdl.handle.net/1721.1/77925,article,None,Coherency Strain and the Kinetics of Phase Sep...,Daniel A. Cogswell; Martin Z. Bazant,2012.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hdl.handle.net/1721.1/77925,<na>,None,None,NaN,None,None,NaN
337600,openalex::W3024898576,DSpace@MIT,openalex,<NA>,None,http://hdl.handle.net/1721.1/60550,article,None,Spin and Valence States of Iron in Mg0.8Fe0.2S...,Brent Grocholski; Seung‐Bo Shim; Jie Zhao; W. ...,2009.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hdl.handle.net/1721.1/60550,<na>,None,None,NaN,None,None,NaN
340441,openalex::W1586102975,DSpace@MIT,openalex,<NA>,None,http://hdl.handle.net/1721.1/60550,article,None,Spin and valence states of iron in (Mg[subscri...,Brent Grocholski; Sang-Heon Dan Shim; W. Sturh...,2009.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hdl.handle.net/1721.1/60550,<na>,None,None,NaN,None,None,NaN
339508,openalex::W2905992706,DSpace@MIT,openalex,<NA>,None,http://hdl.handle.net/1721.1/110743,article,None,Kinetic isotope effects of 12CH3D + OH and 13...,L. M. T. Joelsson; Johan A. Schmidt; Elna J. K...,2016.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hdl.handle.net/1721.1/110743,<na>,None,None,NaN,None,None,NaN
343507,openalex::W2737630491,DSpace@MIT,openalex,<NA>,None,http://hdl.handle.net/1721.1/110743,article,None,Kinetic isotope effects of [superscript 12]CH[...,L. M. T. Joelsson; Johan A. Schmidt; Elna J. K...,2016.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hdl.handle.net/1721.1/110743,<na>,None,None,NaN,None,None,NaN
343453,openalex::W2734455347,DSpace@MIT,openalex,<NA>,None,http://hdl.handle.net/1721.1/110608,article,None,The rotation-vibration structure of the SO[sub...,Jun Jiang; George Barratt Park; Robert W. Field,2016.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hdl.handle.net/1721.1/110608,<na>,None,None,NaN,None,None,NaN
346886,openalex::W3215857213,DSpace@MIT,openalex,<NA>,None,http://hdl.handle.net/1721.1/110608,article,None,The rotation-vibration structure of the SO2 C ...,Jun Jiang; George Park; Robert W. Field,2016.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hdl.handle.net/1721.1/110608,<na>,None,None,NaN,None,None,NaN
343533,openalex::W2739208627,DSpace@MIT,openalex,<NA>,None,http://hdl.handle.net/1721.1/110227,article,None,Observation of B[subscript c][superscript +]→J...,R. Aaij; B. Adeva; M. Adinolfi; Z. Ajaltouni; ...,2017.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http:

In [166]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='Authorea Inc.'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
268002,crossref::10.22541/au.166013641.15972664/v1,Authorea Inc.,crossref,10.22541/au.166013641.15972664/v1,https://doi.org/10.22541/au.166013641.15972664/v1,https://www.techrxiv.org/doi/full/10.36227/tec...,posted-content,preprint,"Human-Centered Artificial Intelligence, a review","Domfeh, Emmanuel Adjei; Weyori, Benjamin; APPI...",2022.0,2022-08-10,None,None,,,,,false,None,None,None,None,parent,https://www.techrxiv.org/doi/full/10.36227/tec...,10.22541/au.166013641.15972664/v1,.v1,explicit_version,1.0,NaN,NaN,NaN
264900,crossref::10.22541/au.174111107.77253989/v1,Authorea Inc.,crossref,10.22541/au.174111107.77253989/v1,https://doi.org/10.22541/au.174111107.77253989/v1,https://www.authorea.com/users/898379/articles...,posted-content,preprint,Inhibition of CaN/FoxO1/FABP4 pathway prevents...,"Zhu, Beibei; Luo, Shuangxue; Su, Hang; Zhang, ...",2025.0,2025-03-04,None,None,,,,,false,None,None,None,None,parent,https://www.authorea.com/users/898379/articles...,10.22541/au.174111107.77253989/v1,None,None,NaN,/v1,explicit_version,1.0
264891,crossref::10.22541/au.174110059.99710888/v1,Authorea Inc.,crossref,10.22541/au.174110059.99710888/v1,https://doi.org/10.22541/au.174110059.99710888/v1,https://www.authorea.com/users/898379/articles...,posted-content,preprint,Inhibition of CaN/FoxO1/FABP4 pathway prevents...,"Zhu, Beibei; Luo, Shuangxue; Su, Hang; Zhang, ...",2025.0,2025-03-04,None,None,,,,,false,None,None,None,None,parent,https://www.authorea.com/users/898379/articles...,10.22541/au.174110059.99710888/v1,None,None,NaN,/v1,explicit_version,1.0
245823,crossref::10.22541/au.169754906.69651455/v1,Authorea Inc.,crossref,10.22541/au.169754906.69651455/v1,https://doi.org/10.22541/au.169754906.69651455/v1,https://www.authorea.com/users/672717/articles...,posted-content,preprint,"Quality of randomised controlled trials, syste...","Jiang, Wilson; Wang, Bill; Sperandei, Sandro; ...",2023.0,2023-10-17,None,None,,,,,false,None,None,None,None,parent,https://www.authorea.com/users/672717/articles...,10.22541/au.169754906.69651455/v1,None,None,NaN,/v1,explicit_version,1.0
245616,crossref::10.22541/au.169710870.00119117/v1,Authorea Inc.,crossref,10.22541/au.169710870.00119117/v1,https://doi.org/10.22541/au.169710870.00119117/v1,https://www.authorea.com/users/672717/articles...,posted-content,preprint,"Quality of randomised controlled trials, syste...","Jiang, Wilson; Wang, Bill; Sperandei, Sandro; ...",2023.0,2023-10-12,None,None,,,,,false,None,None,None,None,parent,https://www.authorea.com/users/672717/articles...,10.22541/au.169710870.00119117/v1,None,None,NaN,/v1,explicit_version,1.0
245834,crossref::10.22541/au.169754941.18437909/v1,Authorea Inc.,crossref,10.22541/au.169754941.18437909/v1,https://doi.org/10.22541/au.169754941.18437909/v1,https://www.authorea.com/users/672139/articles...,posted-content,preprint,Solus: An end-to-end AI software developer,"Blumenfeld, Adam",2023.0,2023-10-17,None,None,,,,,false,None,None,None,None,parent,https://www.authorea.com/users/672139/articles...,10.22541/au.169754941.18437909/v1,None,None,NaN,/v1,explicit_version,1.0
245620,crossref::10.22541/au.169710892.20614312/v1,Authorea Inc.,crossref,10.22541/au.169710892.20614312/v1,https://doi.org/10.22541/au.169710892.20614312/v1,https://www.authorea.com/users/672139/articles...,posted-content,preprint,Solus: An end-to-end AI software developer,"Blumenfeld, Adam",2023.0,2023-10-12,None,None,,,,,false,None,None,None,None,parent,https://www.authorea.com/users/672139/articles...,10.22541/au.169710892.20614312/v1,None,N

In [167]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='Digital Access to Scholarship at Harvard (DASH) (Harvard University)'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
353418,openalex::W7112397115,Digital Access to Scholarship at Harvard (DASH...,openalex,<NA>,None,http://nrs.harvard.edu/urn-3:hul.eresource:Cla...,article,None,What GN owes OMD,"Nagy, Gregory",2018.0,2025-12-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://nrs.harvard.edu/urn-3:hul.eresource:cla...,<na>,None,None,NaN,None,None,NaN
354904,openalex::W7113012407,Digital Access to Scholarship at Harvard (DASH...,openalex,<NA>,None,http://nrs.harvard.edu/urn-3:hul.eresource:Cla...,article,None,How are the epic verses of the Hesiodic Suitor...,"Nagy, Gregory",2021.0,2025-12-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://nrs.harvard.edu/urn-3:hul.eresource:cla...,<na>,None,None,NaN,None,None,NaN
354419,openalex::W7112816552,Digital Access to Scholarship at Harvard (DASH...,openalex,<NA>,None,http://nrs.harvard.edu/urn-3:hul.eresource:Cla...,article,None,"What on earth did Helen ever see in Ajax, her ...","Nagy, Gregory",2021.0,2025-12-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://nrs.harvard.edu/urn-3:hul.eresource:cla...,<na>,None,None,NaN,None,None,NaN
354348,openalex::W7112788517,Digital Access to Scholarship at Harvard (DASH...,openalex,<NA>,None,http://nrs.harvard.edu/urn-3:hul.eresource:Cla...,article,None,A question of “reception”: how could Homer eve...,"Nagy, Gregory",2021.0,2025-12-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://nrs.harvard.edu/urn-3:hul.eresource:cla...,<na>,None,None,NaN,None,None,NaN
354924,openalex::W7113021182,Digital Access to Scholarship at Harvard (DASH...,openalex,<NA>,None,http://nrs.harvard.edu/urn-3:hul.eresource:Cla...,article,None,Death of an Amazon,"Nagy, Gregory",2020.0,2025-12-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://nrs.harvard.edu/urn-3:hul.eresource:cla...,<na>,None,None,NaN,None,None,NaN
354065,openalex::W7112670930,Digital Access to Scholarship at Harvard (DASH...,openalex,<NA>,None,http://nrs.harvard.edu/urn-3:hul.eresource:Cla...,article,None,Can Sappho be freed from receivership? Part Two,"Nagy, Gregory",2021.0,2025-12-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://nrs.harvard.edu/urn-3:hul.eresource:cla...,<na>,None,None,NaN,None,None,NaN
354032,openalex::W7112658728,Digital Access to Scholarship at Harvard (DASH...,openalex,<NA>,None,http://nrs.harvard.edu/urn-3:hul.eresource:Cla...,article,None,Mages and Ionians,"Nagy, Gregory",2017.0,2025-12-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://nrs.harvard.edu/urn-3:hul.eresource:cla...,<na>,None,None,NaN,None,None,NaN
353580,openalex::W7112462899,Digital Access to Scholarship at Harvard (DASH...,openalex,<NA>,None,http://nrs.harvard.edu/urn-3:hul.eresource:Cla...,article,None,"Sappho’s Aphrodite, the goddess Chryse, and a ...","Nagy, Gregory",2021.0,2025-12-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://nrs.harvard.edu/urn-3:hul.eresource:cla...,<na>,None,None,NaN,None,None,NaN
353482,openalex::W7112425060,Digital Access to Scholarship at Harvard (DASH...,openalex,<NA>,None,http://nrs.harvard.edu/urn-3:hul.eresource:Cla...,article,None,On Visualizing Heavenly Origins for Particular...,"Nagy, Gregory",2021.0,2025-12-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://nrs.harvard.edu/urn-3:hul.eresource:cla...,<na>,None,None,NaN,None,None,NaN
353456,openalex::W7112414740,Digital Access to 

In [168]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='EarthArXiv'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
387180,crossref::10.31223/x5kt7m,EarthArXiv,crossref,10.31223/x5kt7m,https://doi.org/10.31223/x5kt7m,https://eartharxiv.org/repository/view/6898/,posted-content,preprint,Can spinodal decomposition occur during decomp...,"Nishiwaki, Mizuki",2025.0,2025-08-13,None,None,,,,,false,None,None,None,None,parent,https://eartharxiv.org/repository/view/6898/,10.31223/x5kt7m,None,None,NaN,None,None,NaN
387409,crossref::10.31223/x58h6b,EarthArXiv,crossref,10.31223/x58h6b,https://doi.org/10.31223/x58h6b,https://eartharxiv.org/repository/view/6898/,posted-content,preprint,Can spinodal decomposition occur during decomp...,"Nishiwaki, Mizuki",2025.0,2024-03-28,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.1016/j.epsl.2025.119655,,,true,None,None,None,None,parent,https://eartharxiv.org/repository/view/6898/,10.31223/x58h6b,None,None,NaN,None,None,NaN
385778,crossref::10.31223/x5fd5g,EarthArXiv,crossref,10.31223/x5fd5g,https://doi.org/10.31223/x5fd5g,https://eartharxiv.org/repository/view/6821/,posted-content,preprint,Moving graphs: Predicting barchan dune migrati...,"Beelen, Daan",2024.0,2024-03-07,None,None,,,,,false,None,None,None,None,parent,https://eartharxiv.org/repository/view/6821/,10.31223/x5fd5g,None,None,NaN,None,None,NaN
385779,crossref::10.31223/x52t1p,EarthArXiv,crossref,10.31223/x52t1p,https://doi.org/10.31223/x52t1p,https://eartharxiv.org/repository/view/6821/,posted-content,preprint,Moving graphs: Predicting barchan dune migrati...,"Beelen, Daan",2024.0,2023-10-17,None,None,,,,,false,None,None,None,None,parent,https://eartharxiv.org/repository/view/6821/,10.31223/x52t1p,None,None,NaN,None,None,NaN
385044,crossref::10.31223/x54s90,EarthArXiv,crossref,10.31223/x54s90,https://doi.org/10.31223/x54s90,https://eartharxiv.org/repository/view/5038/,posted-content,preprint,Statistical precursor signals for Dansgaard-Oe...,"Mitsui, Takahito; Boers, Niklas",2023.0,2023-02-14,None,None,,,,,false,None,None,None,None,parent,https://eartharxiv.org/repository/view/5038/,10.31223/x54s90,None,None,NaN,None,None,NaN
384765,crossref::10.31223/x58m1g,EarthArXiv,crossref,10.31223/x58m1g,https://doi.org/10.31223/x58m1g,https://eartharxiv.org/repository/view/5038/,posted-content,preprint,Predictability of abrupt northern-hemisphere c...,"Mitsui, Takahito; Boers, Niklas",2023.0,2023-02-14,None,None,,,,,false,None,None,None,None,parent,https://eartharxiv.org/repository/view/5038/,10.31223/x58m1g,None,None,NaN,None,None,NaN
383593,crossref::10.31223/x59922,EarthArXiv,crossref,10.31223/x59922,https://doi.org/10.31223/x59922,http://eartharxiv.org/repository/view/2855/,posted-content,preprint,The Influence of Grain Shape and Size on the R...,"Payton, Ryan; Chiarella, Domenico; Kingdon, An...",2021.0,2021-11-12,None,None,,,,,false,None,None,None,None,parent,http://eartharxiv.org/repository/view/2855/,10.31223/x59922,None,None,NaN,None,None,NaN
383600,crossref::10.31223/x5gs6z,EarthArXiv,crossref,10.31223/x5gs6z,https://doi.org/10.31223/x5gs6z,http://eartharxiv.org/repository/view/2855/,posted-content,preprint,The Influence of Grain Shape and Size on the R...,"Payton, Ryan; Chiarella, Domenico; Kingdon, An...",2021.0,2021-11-18,None,None,,,,,false,None,None,None,None,parent,http://eartharxiv.org/repository/view/2855/,10.31223/x5gs6z,None,None,NaN,None,None,NaN
384056,crossref::10.31223/x5fk9q,EarthArXiv,crossref,10.31223/x5fk9q,https://doi.org/10.31223/x5fk9q,http://eartharxiv.org/repository/view/2730/,posted-content,preprint,Evaluating the Evolution of ECMWF Precipitatio...,"Ghajarnia, Navid

In [169]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='IACR Cryptology ePrint Archive'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
1911981,openalex::W3207917365,IACR Cryptology ePrint Archive,openalex,<NA>,None,https://eprint.iacr.org/2016/161.pdf,preprint,None,Revisiting Structure Graphs: Applications to C...,Ashwin Jha; Mridul Nandi,2016.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://eprint.iacr.org/2016/161.pdf,<na>,None,None,NaN,None,None,NaN
1905088,openalex::W2952233364,IACR Cryptology ePrint Archive,openalex,<NA>,None,https://eprint.iacr.org/2016/161.pdf,preprint,None,Revisiting Structure Graph and Its Application...,Ashwin Jha; Mridul Nandi,2016.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://eprint.iacr.org/2016/161.pdf,<na>,None,None,NaN,None,None,NaN
1907291,openalex::W3217563045,IACR Cryptology ePrint Archive,openalex,<NA>,None,https://eprint.iacr.org/2016/008.pdf,preprint,None,cMix: Mixing with Minimal Real-Time Asymmetric...,David Chaum; Debajyoti Das; Farid Javani; Anik...,2016.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://eprint.iacr.org/2016/008.pdf,<na>,None,None,NaN,None,None,NaN
1901246,openalex::W2398691649,IACR Cryptology ePrint Archive,openalex,<NA>,None,https://eprint.iacr.org/2016/008.pdf,preprint,None,cMix: Anonymization byHigh-Performance Scalabl...,David Chaum; Farid Javani; Aniket Kate; Anna K...,2016.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://eprint.iacr.org/2016/008.pdf,<na>,None,None,NaN,None,None,NaN
1907246,openalex::W3207746821,IACR Cryptology ePrint Archive,openalex,<NA>,None,https://eprint.iacr.org/2015/971.pdf,preprint,None,Attacks on the Search-RLWE problem with small ...,Hao Chen; Kristin Lauter; Katherine E. Stange,2015.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://eprint.iacr.org/2015/971.pdf,<na>,None,None,NaN,None,None,NaN
1901098,openalex::W2403700287,IACR Cryptology ePrint Archive,openalex,<NA>,None,https://eprint.iacr.org/2015/971.pdf,preprint,None,Attacks on Search RLWE.,Hao Chen; Kristin Lauter; Katherine E. Stange,2015.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://eprint.iacr.org/2015/971.pdf,<na>,None,None,NaN,None,None,NaN
1904776,openalex::W2791898560,IACR Cryptology ePrint Archive,openalex,<NA>,None,https://eprint.iacr.org/2015/942.pdf,preprint,None,Secrecy and independence for election schemes.,Ben Smyth,2015.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://eprint.iacr.org/2015/942.pdf,<na>,None,None,NaN,None,None,NaN
1903820,openalex::W3210407932,IACR Cryptology ePrint Archive,openalex,<NA>,None,https://eprint.iacr.org/2015/942.pdf,preprint,None,"Ballot secrecy: Security definition, sufficien...",Ben Smyth,2015.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://eprint.iacr.org/2015/942.pdf,<na>,None,None,NaN,None,None,NaN
1911891,openalex::W3204639541,IACR Cryptology ePrint Archive,openalex,<NA>,None,https://eprint.iacr.org/2015/806.pdf,preprint,None,Fault Space Transformation: A Generic Approach...,Sikhar Patranabis; Abhishek Chakraborty; Debde...,2015.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://eprint.iacr.org/2015/806.pdf,<na>,None,None,NaN,None,None,NaN
1901691,openalex::W2402265787,IACR Cryptology ePrint Archive,openalex,<NA>,None,https://eprint.iacr.org/2015/806.pdf,preprint,None,Using State Space Encoding To Counter Biased F...,Sikhar Patranabis;

In [170]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='Organic Eprints'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi


In [171]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='Open Science Framework'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
2115304,crossref::10.31219/osf.io/wcks6_v1,Open Science Framework,crossref,10.31219/osf.io/wcks6_v1,https://doi.org/10.31219/osf.io/wcks6_v1,https://osf.io/wcks6_v1,posted-content,preprint,Value-based decision-making in regular alcohol...,"Copeland, Amber; Stafford, Tom; Field, Matt",2023.0,2025-05-26,None,None,,,,,false,None,None,None,None,parent,https://osf.io/wcks6_v1,10.31219/osf.io/wcks6_v1,_v1,explicit_version,1.0,NaN,NaN,NaN
2186518,crossref::10.31219/osf.io/wcks6,Open Science Framework,crossref,10.31219/osf.io/wcks6,https://doi.org/10.31219/osf.io/wcks6,https://osf.io/wcks6_v1,posted-content,preprint,Value-based decision-making in regular alcohol...,"Copeland, Amber; Stafford, Tom; Field, Matt",2023.0,2023-08-24,None,None,,,,,false,None,None,None,None,parent,https://osf.io/wcks6_v1,10.31219/osf.io/wcks6,_v1,explicit_version,1.0,NaN,NaN,NaN
2083024,crossref::10.31219/osf.io/w8kue,Open Science Framework,crossref,10.31219/osf.io/w8kue,https://doi.org/10.31219/osf.io/w8kue,https://osf.io/w8kue,posted-content,preprint,Quality-of-life in dementia: Assessment in low...,"Chua, Kia-Chong; Böhnke, Jan R.; Prince, Marti...",2017.0,2018-07-02,None,None,,,,,false,None,None,None,None,parent,https://osf.io/w8kue,10.31219/osf.io/w8kue,None,None,NaN,None,None,NaN
2092736,crossref::10.31219/osf.io/w4mhc,Open Science Framework,crossref,10.31219/osf.io/w4mhc,https://doi.org/10.31219/osf.io/w4mhc,https://osf.io/w4mhc,posted-content,preprint,Chinese Immersion Teachers in the U.S: Percept...,"Chen, Mengyao; Li, Jiahang; Gorke, Yongling Z",2022.0,2022-03-07,None,None,,,,,false,None,None,None,None,parent,https://osf.io/w4mhc,10.31219/osf.io/w4mhc,None,None,NaN,None,None,NaN
2116175,crossref::10.31219/osf.io/sdzfq_v1,Open Science Framework,crossref,10.31219/osf.io/sdzfq_v1,https://doi.org/10.31219/osf.io/sdzfq_v1,https://osf.io/sdzfq_v1,posted-content,preprint,Exploring the Design Space of BioFabric Visual...,"Fuchs, Johannes; Dennig, Frederik L.; Heinle, ...",2024.0,2025-06-30,None,None,,,,,false,None,None,None,None,parent,https://osf.io/sdzfq_v1,10.31219/osf.io/sdzfq_v1,_v1,explicit_version,1.0,NaN,NaN,NaN
2185740,crossref::10.31219/osf.io/sdzfq,Open Science Framework,crossref,10.31219/osf.io/sdzfq,https://doi.org/10.31219/osf.io/sdzfq,https://osf.io/sdzfq_v1,posted-content,preprint,Exploring the Design Space of BioFabric Visual...,"Fuchs, Johannes; Dennig, Frederik L.; Heinle, ...",2024.0,2024-03-21,None,None,,,,,false,None,None,None,None,parent,https://osf.io/sdzfq_v1,10.31219/osf.io/sdzfq,_v1,explicit_version,1.0,NaN,NaN,NaN
2104184,crossref::10.31219/osf.io/rfvy5,Open Science Framework,crossref,10.31219/osf.io/rfvy5,https://doi.org/10.31219/osf.io/rfvy5,https://osf.io/rfvy5,posted-content,preprint,Assessing the Psychometric Properties and Vali...,"Haddox, Dawson; Mackin, Daniel; Griffin, Tess;...",2024.0,2024-11-04,None,None,,,,,false,None,None,None,None,parent,https://osf.io/rfvy5,10.31219/osf.io/rfvy5,None,None,NaN,None,None,NaN
2186549,crossref::10.31219/osf.io/p5gm4,Open Science Framework,crossref,10.31219/osf.io/p5gm4,https://doi.org/10.31219/osf.io/p5gm4,https://osf.io/p5gm4_v1,posted-content,preprint,Young children's screen time during the first ...,"Bergmann, Christina; Dimitrova, Nevena; Alasla...",2021.0,2021-05-31,None,None,,,,,false,None,None,None,None,parent,https://osf.io/p5gm4_v1,10.31219/osf.io/p5gm4,_v1,explicit_version,1.0,NaN,NaN,NaN
2115469,crossref::10.31219/osf.io/p5gm4_v1,Open Science Framework,crossref,10.31219/osf.io/p5gm4_v1,https://doi.org/10.31219/osf.io/p5gm4_v1,

In [172]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='SciELO Preprints'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
6170519,crossref::10.1590/scielopreprints9984,SciELO Preprints,crossref,10.1590/scielopreprints9984,https://doi.org/10.1590/scielopreprints9984,https://preprints.scielo.org/index.php/scielo/...,posted-content,preprint,ESTUDANTES COTISTAS NO CURSO DE MEDICINA DA UF...,"Vilar Bonaldi, Eduardo; Viricimo, Luan",2024.0,2024-09-26,None,None,,,,,false,None,None,None,None,parent,https://preprints.scielo.org/index.php/scielo/...,10.1590/scielopreprints9984,None,None,NaN,None,None,NaN
6170542,crossref::10.1590/scielopreprints.9984,SciELO Preprints,crossref,10.1590/scielopreprints.9984,https://doi.org/10.1590/scielopreprints.9984,https://preprints.scielo.org/index.php/scielo/...,posted-content,preprint,ESTUDANTES COTISTAS NO CURSO DE MEDICINA DA UF...,"Vilar Bonaldi, Eduardo; Viricimo, Luan",2024.0,2024-09-14,None,None,,,,,false,None,None,None,None,parent,https://preprints.scielo.org/index.php/scielo/...,10.1590/scielopreprints.9984,None,None,NaN,None,None,NaN
6167884,crossref::10.1590/scielopreprints.2690,SciELO Preprints,crossref,10.1590/scielopreprints.2690,https://doi.org/10.1590/scielopreprints.2690,https://preprints.scielo.org/index.php/scielo/...,posted-content,preprint,Prácticas de evaluación en entornos virtuales ...,"Picón, Gerardo Armando; Rodríguez, Nimia; Oliv...",2021.0,2021-07-26,None,None,,,,,false,None,None,None,None,parent,https://preprints.scielo.org/index.php/scielo/...,10.1590/scielopreprints.2690,None,None,NaN,None,None,NaN
6167620,crossref::10.1590/scielopreprints.1690,SciELO Preprints,crossref,10.1590/scielopreprints.1690,https://doi.org/10.1590/scielopreprints.1690,https://preprints.scielo.org/index.php/scielo/...,posted-content,preprint,Prácticas de evaluación en entornos virtuales ...,"Picón, Gerardo Armando; Rodríguez, Nimia; Oliv...",2021.0,2021-07-26,None,None,,,,,false,None,None,None,None,parent,https://preprints.scielo.org/index.php/scielo/...,10.1590/scielopreprints.1690,None,None,NaN,None,None,NaN
6171465,crossref::10.1590/2596-304x202527e20251386,SciELO Preprints,crossref,10.1590/2596-304x202527e20251386,https://doi.org/10.1590/2596-304x202527e20251386,https://preprints.scielo.org/index.php/scielo/...,posted-content,preprint,Surrealism and architecture: the transatlantic...,"Naumann Machado, Nara Helena; Ponge, Robert",2025.0,2025-11-06,None,None,,,,,false,None,None,None,None,parent,https://preprints.scielo.org/index.php/scielo/...,10.1590/2596-304x202527e20251386,None,None,NaN,None,None,NaN
6171448,crossref::10.1590/scielopreprints.13933,SciELO Preprints,crossref,10.1590/scielopreprints.13933,https://doi.org/10.1590/scielopreprints.13933,https://preprints.scielo.org/index.php/scielo/...,posted-content,preprint,Surrealism and architecture: the transatlantic...,"Naumann Machado, Nara Helena; Ponge, Robert",2025.0,2025-10-30,None,None,,,,,false,None,None,None,None,parent,https://preprints.scielo.org/index.php/scielo/...,10.1590/scielopreprints.13933,None,None,NaN,None,None,NaN
6171466,crossref::10.1590/2596-304x202527e20251278,SciELO Preprints,crossref,10.1590/2596-304x202527e20251278,https://doi.org/10.1590/2596-304x202527e20251278,https://preprints.scielo.org/index.php/scielo/...,posted-content,preprint,"Orfeu Negro e Emicida AmarElo: canto, comunida...","Silva Menezes, Roniere",2025.0,2025-11-06,None,None,,,,,false,None,None,None,None,parent,https://preprints.scielo.org/index.php/scielo/...,10.1590/2596-304x202527e20251278,None,None,NaN,None,None,NaN
6171447,crossref::10.1590/scielopreprints.13919,SciELO Preprints,crossref,10.1590/scielopreprints.13919,ht

In [173]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='EcoEvoRxiv'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
414171,crossref::10.31219/osf.io/wu5vz,EcoEvoRxiv,crossref,10.31219/osf.io/wu5vz,https://doi.org/10.31219/osf.io/wu5vz,https://osf.io/wu5vz,posted-content,preprint,Test,"Rosenblatt, Rebecca P.",2018.0,2018-11-09,None,None,,,,,false,None,None,None,None,parent,https://osf.io/wu5vz,10.31219/osf.io/wu5vz,None,None,NaN,None,None,NaN
414175,crossref::10.32942/osf.io/wu5vz,EcoEvoRxiv,crossref,10.32942/osf.io/wu5vz,https://doi.org/10.32942/osf.io/wu5vz,https://osf.io/wu5vz,posted-content,preprint,<NA>,<NA>,2018.0,2018-11-09,None,None,,,,,false,None,None,None,None,parent,https://osf.io/wu5vz,10.32942/osf.io/wu5vz,None,None,NaN,None,None,NaN
414174,crossref::10.32942/osf.io/k85eq,EcoEvoRxiv,crossref,10.32942/osf.io/k85eq,https://doi.org/10.32942/osf.io/k85eq,https://osf.io/k85eq,posted-content,preprint,<NA>,<NA>,2018.0,2018-11-09,None,None,,,,,false,None,None,None,None,parent,https://osf.io/k85eq,10.32942/osf.io/k85eq,None,None,NaN,None,None,NaN
414172,crossref::10.31219/osf.io/k85eq,EcoEvoRxiv,crossref,10.31219/osf.io/k85eq,https://doi.org/10.31219/osf.io/k85eq,https://osf.io/k85eq,posted-content,preprint,testtest,"Rosenblatt, Rebecca P.",2018.0,2018-11-09,None,None,,,,,false,None,None,None,None,parent,https://osf.io/k85eq,10.31219/osf.io/k85eq,None,None,NaN,None,None,NaN
414173,crossref::10.31219/osf.io/gzunx,EcoEvoRxiv,crossref,10.31219/osf.io/gzunx,https://doi.org/10.31219/osf.io/gzunx,https://osf.io/gzunx,posted-content,preprint,testtesttest,"Rosenblatt, Rebecca P.",2018.0,2018-11-09,None,None,,,,,false,None,None,None,None,parent,https://osf.io/gzunx,10.31219/osf.io/gzunx,None,None,NaN,None,None,NaN
414176,crossref::10.32942/osf.io/gzunx,EcoEvoRxiv,crossref,10.32942/osf.io/gzunx,https://doi.org/10.32942/osf.io/gzunx,https://osf.io/gzunx,posted-content,preprint,<NA>,<NA>,2018.0,2018-11-09,None,None,,,,,false,None,None,None,None,parent,https://osf.io/gzunx,10.32942/osf.io/gzunx,None,None,NaN,None,None,NaN
412672,crossref::10.32942/x2qc8z,EcoEvoRxiv,crossref,10.32942/x2qc8z,https://doi.org/10.32942/x2qc8z,https://ecoevorxiv.org/repository/view/6403/,posted-content,preprint,The trade-offs of honest and dishonest signals,"Zachar, István; Penn, Dustin",2023.0,2023-12-13,None,None,,,,,false,None,None,None,None,parent,https://ecoevorxiv.org/repository/view/6403/,10.32942/x2qc8z,None,None,NaN,None,None,NaN
412680,crossref::10.32942/x2pc91,EcoEvoRxiv,crossref,10.32942/x2pc91,https://doi.org/10.32942/x2pc91,https://ecoevorxiv.org/repository/view/6403/,posted-content,preprint,The trade-offs of honest and dishonest signals,"Számadó, Szabolcs; Zachar, István; Penn, Dustin",2023.0,2023-12-14,None,None,,,,,false,None,None,None,None,parent,https://ecoevorxiv.org/repository/view/6403/,10.32942/x2pc91,None,None,NaN,None,None,NaN
412638,crossref::10.32942/x23s40,EcoEvoRxiv,crossref,10.32942/x23s40,https://doi.org/10.32942/x23s40,https://ecoevorxiv.org/repository/view/6292/,posted-content,preprint,Amazonian soundscapes: unravelling the secrets...,"Do Nascimento, Leandro; Pérez-Granados, Cristi...",2023.0,2023-11-28,None,None,,,,,false,None,None,None,None,parent,https://ecoevorxiv.org/repository/view/6292/,10.32942/x23s40,None,None,NaN,None,None,NaN
412631,crossref::10.32942/x2102p,EcoEvoRxiv,crossref,10.32942/x2102p,https://doi.org/10.32942/x2102p,https://ecoevorxiv.org/repository/view/6292/,posted-content,preprint,Amazonian soundscapes: unravelling the secrets...,"Do Nascimento, Leandro; Pérez-Granados, Cristi...",2023.0,2023-11-22,None,None,,,,,false,None,None,None,None,parent,https://ecoevorxiv.org/rep

In [174]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='UCL Open Environment'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
6258122,crossref::10.14324/111.444/ucloe.000068,UCL Open Environment,crossref,10.14324/111.444/ucloe.000068,https://doi.org/10.14324/111.444/ucloe.000068,https://journals.uclpress.co.uk/ucloe/article/...,journal-article,None,Use of evidence and expertise in UK climate go...,"Willis, Rebecca",2024.0,2024-02-08,None,None,,,,,false,None,None,None,None,parent,https://journals.uclpress.co.uk/ucloe/article/...,10.14324/111.444/ucloe.000068,None,None,NaN,None,None,NaN
6258121,crossref::10.14324/ucloe.1982,UCL Open Environment,crossref,10.14324/ucloe.1982,https://doi.org/10.14324/ucloe.1982,https://journals.uclpress.co.uk/ucloe/article/...,journal-article,None,Use of evidence and expertise in UK climate go...,"Willis, Rebecca",2024.0,2024-01-30,None,None,,,,,false,None,None,None,None,parent,https://journals.uclpress.co.uk/ucloe/article/...,10.14324/ucloe.1982,None,None,NaN,None,None,NaN


In [175]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='Humanities Commons CORE'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
1883185,datacite::10.17613/bah4-vw03,Humanities Commons CORE,datacite,10.17613/bah4-vw03,https://doi.org/10.17613/bah4-vw03,https://hcommons.org/deposits/removed/,Text,,<NA>,"N/A, N/A",2026.0,2024-05-03,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""msu.core"", ""type"":...",parent,https://hcommons.org/deposits/removed/,10.17613/bah4-vw03,None,None,NaN,None,None,NaN
1876175,datacite::10.17613/hs98-7t33,Humanities Commons CORE,datacite,10.17613/hs98-7t33,https://doi.org/10.17613/hs98-7t33,https://hcommons.org/deposits/removed/,Other,,<NA>,<NA>,2020.0,2021-11-30,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""msu.core"", ""type"":...",parent,https://hcommons.org/deposits/removed/,10.17613/hs98-7t33,None,None,NaN,None,None,NaN
1892016,datacite::10.17613/sbns-7n85,Humanities Commons CORE,datacite,10.17613/sbns-7n85,https://doi.org/10.17613/sbns-7n85,https://hcommons.org/deposits/removed/,Other,None,<NA>,"N/A, N/A",2024.0,2024-07-20,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""msu.core"", ""type"":...",parent,https://hcommons.org/deposits/removed/,10.17613/sbns-7n85,None,None,NaN,None,None,NaN
1876180,datacite::10.17613/w01d-y281,Humanities Commons CORE,datacite,10.17613/w01d-y281,https://doi.org/10.17613/w01d-y281,https://hcommons.org/deposits/removed/,Other,None,<NA>,<NA>,2021.0,2021-12-01,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""msu.core"", ""type"":...",parent,https://hcommons.org/deposits/removed/,10.17613/w01d-y281,None,None,NaN,None,None,NaN
1876183,datacite::10.17613/x5jz-ym79,Humanities Commons CORE,datacite,10.17613/x5jz-ym79,https://doi.org/10.17613/x5jz-ym79,https://hcommons.org/deposits/removed/,Other,None,<NA>,<NA>,2022.0,2021-12-02,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""msu.core"", ""type"":...",parent,https://hcommons.org/deposits/removed/,10.17613/x5jz-ym79,None,None,NaN,None,None,NaN
1873293,datacite::10.17613/0smm-ms52,Humanities Commons CORE,datacite,10.17613/0smm-ms52,https://doi.org/10.17613/0smm-ms52,https://hcommons.org/deposits/removed,Other,None,<NA>,"N/A, N/A",2019.0,2019-09-15,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""msu.core"", ""type"":...",parent,https://hcommons.org/deposits/removed,10.17613/0smm-ms52,None,None,NaN,None,None,NaN
1872153,datacite::10.17613/m67w67553,Humanities Commons CORE,datacite,10.17613/m67w67553,https://doi.org/10.17613/m67w67553,https://hcommons.org/deposits/removed,Other,None,<NA>,"N/A, N/A",2017.0,2018-10-14,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""msu.core"", ""type"":...",parent,https://hcommons.org/deposits/removed,10.17613/m67w67553,None,None,NaN,None,None,NaN
1872731,datacite::10.17613/mxjy-1d38,Humanities Commons CORE,datacite,10.17613/mxjy-1d38,https://doi.org/10.17613/mxjy-1d38,https://hcommons.org/deposits/removed,Other,None,<NA>,"N/A, N/A",2015.0,2019-04-12,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""msu.core"", ""type"":...",parent,https://hcommons.org/deposits/removed,10.17613/mxjy-1d38,None,None,NaN,None,None,NaN
1873386,datacite::10.17613/pyew-7h37,Humanities Commons CORE,datacite,10.17613/pyew-7h37,https://doi.org/10.17613/pyew-7h37,https://hcommons.org/deposits/removed,Other,None,<NA>,"N/A, N/A",2019.0,2019-10-19,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""msu.core"", ""type"":...",parent,https://hcommons.org/deposits/removed,10.17613/pyew-7h37,None,None,NaN,None,None,NaN
1882736,datacit

In [176]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='SocArXiv'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
6176498,crossref::10.31235/osf.io/w4mhc,SocArXiv,crossref,10.31235/osf.io/w4mhc,https://doi.org/10.31235/osf.io/w4mhc,https://osf.io/w4mhc,posted-content,preprint,Chinese Immersion Teachers in the U.S: Percept...,"Chen, Mengyao; Li, Jiahang; Gorke, Yongling Z",2022.0,2022-03-14,None,None,,,,,false,None,None,None,None,parent,https://osf.io/w4mhc,10.31235/osf.io/w4mhc,None,None,NaN,None,None,NaN
6195145,crossref::10.31235/osf.io/8zejr,SocArXiv,crossref,10.31235/osf.io/8zejr,https://doi.org/10.31235/osf.io/8zejr,https://osf.io/8zejr_v1,posted-content,preprint,The Financial Geography of Sustainability Data...,"Dimmelmeier, Andreas",2023.0,2023-12-16,None,None,,,,,false,None,None,None,None,parent,https://osf.io/8zejr_v1,10.31235/osf.io/8zejr,_v1,explicit_version,1.0,NaN,NaN,NaN
6179731,crossref::10.31235/osf.io/8zejr_v1,SocArXiv,crossref,10.31235/osf.io/8zejr_v1,https://doi.org/10.31235/osf.io/8zejr_v1,https://osf.io/8zejr_v1,posted-content,preprint,The Financial Geography of Sustainability Data...,"Dimmelmeier, Andreas",2023.0,2025-06-12,None,None,,,,,false,None,None,None,None,parent,https://osf.io/8zejr_v1,10.31235/osf.io/8zejr_v1,_v1,explicit_version,1.0,NaN,NaN,NaN
6189214,crossref::10.31235/osf.io/8hvm6,SocArXiv,crossref,10.31235/osf.io/8hvm6,https://doi.org/10.31235/osf.io/8hvm6,https://osf.io/8hvm6_v1,posted-content,preprint,Mahatma Gandhi and Basic Education,"Jena, Pravat Kumar",2021.0,2021-01-07,None,None,,,,,false,None,None,None,None,parent,https://osf.io/8hvm6_v1,10.31235/osf.io/8hvm6,_v1,explicit_version,1.0,NaN,NaN,NaN
6179770,crossref::10.31235/osf.io/8hvm6_v1,SocArXiv,crossref,10.31235/osf.io/8hvm6_v1,https://doi.org/10.31235/osf.io/8hvm6_v1,https://osf.io/8hvm6_v1,posted-content,preprint,Mahatma Gandhi and Basic Education,"Jena, Pravat Kumar",2021.0,2025-06-17,None,None,,,,,false,None,None,None,None,parent,https://osf.io/8hvm6_v1,10.31235/osf.io/8hvm6_v1,_v1,explicit_version,1.0,NaN,NaN,NaN
6190543,crossref::10.31235/osf.io/4xpza,SocArXiv,crossref,10.31235/osf.io/4xpza,https://doi.org/10.31235/osf.io/4xpza,https://osf.io/4xpza_v1,posted-content,preprint,Cognitive biases in strategic decision-making,"Midtgård, Kenneth; Selart, Marcus",2024.0,2024-03-02,None,None,,,,,false,None,None,None,None,parent,https://osf.io/4xpza_v1,10.31235/osf.io/4xpza,_v1,explicit_version,1.0,NaN,NaN,NaN
6179752,crossref::10.31235/osf.io/4xpza_v1,SocArXiv,crossref,10.31235/osf.io/4xpza_v1,https://doi.org/10.31235/osf.io/4xpza_v1,https://osf.io/4xpza_v1,posted-content,preprint,Cognitive biases in strategic decision-making,"Midtgård, Kenneth; Selart, Marcus",2024.0,2025-06-13,None,None,,,,,false,None,None,None,None,parent,https://osf.io/4xpza_v1,10.31235/osf.io/4xpza_v1,_v1,explicit_version,1.0,NaN,NaN,NaN
6194988,crossref::10.31235/osf.io/3bzex,SocArXiv,crossref,10.31235/osf.io/3bzex,https://doi.org/10.31235/osf.io/3bzex,https://osf.io/3bzex_v1,posted-content,preprint,Dark Money and Politician Learning,"Schnakenberg, Keith; Turner, Ian R",2023.0,2023-01-07,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.1086/734548,,,true,None,None,None,None,parent,https://osf.io/3bzex_v1,10.31235/osf.io/3bzex,_v1,explicit_version,1.0,NaN,NaN,NaN
6179847,crossref::10.31235/osf.io/3bzex_v1,SocArXiv,crossref,10.31235/osf.io/3bzex_v1,https://doi.org/10.31235/osf.io/3bzex_v1,https://osf.io/3bzex_v1,posted-content,preprint,Dark Money and Politician Learning,"Schnakenberg, Keith; Turner, Ian R",2023.0,2025-06-24,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.1086/734548,,,true,No

In [177]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='ART-Dok'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
5776,datacite::10.11588/artdok.00002463,ART-Dok,datacite,10.11588/artdok.00002463,https://doi.org/10.11588/artdok.00002463,https://www.ub.uni-heidelberg.de/helios/digi/r...,Text,Monograph,"Sprachtäter, Ausschließensmechanismus, Reine\r...","Riahi, Azam; Zafani Rad, Nika",2014.0,2017-02-15,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""gesis.ubhd"", ""type...",parent,https://www.ub.uni-heidelberg.de/helios/digi/r...,10.11588/artdok.00002463,None,None,NaN,None,None,NaN
5777,datacite::10.11588/artdok.00002464,ART-Dok,datacite,10.11588/artdok.00002464,https://doi.org/10.11588/artdok.00002464,https://www.ub.uni-heidelberg.de/helios/digi/r...,Text,Monograph,Die nicht zum Ausdruck gekommene Moderne und e...,"Riahi, Azam",2014.0,2017-02-15,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""gesis.ubhd"", ""type...",parent,https://www.ub.uni-heidelberg.de/helios/digi/r...,10.11588/artdok.00002464,None,None,NaN,None,None,NaN
10597,datacite::10.11588/artdok.00008068,ART-Dok,datacite,10.11588/artdok.00008068,https://doi.org/10.11588/artdok.00008068,https://archiv.ub.uni-heidelberg.de/artdok/id/...,Text,BookSection,Malarstwo witrażowe,"Labuda, Adam S. [Hrsg.]; Secomska, Krystyna [H...",2023.0,2022-12-07,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""gesis.ubhd"", ""type...",parent,https://archiv.ub.uni-heidelberg.de/artdok/id/...,10.11588/artdok.00008068,None,None,NaN,None,None,NaN
10685,datacite::10.11588/artdok.00008143,ART-Dok,datacite,10.11588/artdok.00008143,https://doi.org/10.11588/artdok.00008143,https://archiv.ub.uni-heidelberg.de/artdok/id/...,Text,BookSection,Malarstwo witrażowe,"Labuda, Adam S. [Hrsg.]; Secomska, Krystyna [H...",2023.0,2023-02-08,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""gesis.ubhd"", ""type...",parent,https://archiv.ub.uni-heidelberg.de/artdok/id/...,10.11588/artdok.00008143,None,None,NaN,None,None,NaN
6074,datacite::10.11588/artdok.00002777,ART-Dok,datacite,10.11588/artdok.00002777,https://doi.org/10.11588/artdok.00002777,https://archiv.ub.uni-heidelberg.de/artdok/id/...,Text,BookSection,Visuelle Topoi um 1600. Annibale Carracci zwis...,"Dickhut, Wolfgang [Hrsg.]; Manns, Stefan [Hrsg...",2022.0,2017-02-15,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""gesis.ubhd"", ""type...",parent,https://archiv.ub.uni-heidelberg.de/artdok/id/...,10.11588/artdok.00002777,None,None,NaN,None,None,NaN
10294,datacite::10.11588/artdok.00007786,ART-Dok,datacite,10.11588/artdok.00007786,https://doi.org/10.11588/artdok.00007786,https://archiv.ub.uni-heidelberg.de/artdok/id/...,Text,BookSection,Visuelle Topoi um 1600. Annibale Carracci zwis...,"Dickhut, Wolfgang [Hrsg.]; Manns, Stefan [Hrsg...",2022.0,2022-04-25,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""gesis.ubhd"", ""type...",parent,https://archiv.ub.uni-heidelberg.de/artdok/id/...,10.11588/artdok.00007786,None,None,NaN,None,None,NaN
9151,datacite::10.11588/artdok.00006572,ART-Dok,datacite,10.11588/artdok.00006572,https://doi.org/10.11588/artdok.00006572,http://archiv.ub.uni-heidelberg.de/artdok/id/e...,Text,Article,Una versione sconosciuta della tela di Giovann...,"Kienlechner, Susanne",2019.0,2019-09-17,[],None,,,,,false,None,None,None,"{""client"": {""data"": {""id"": ""gesis.ubhd"", ""type...",parent,http://archiv.ub.uni-heidelberg.de/artdok/id/e...,10.11588/artdok.00006572,None,None,NaN,None,None,NaN
9203,datacite::10.11588/artdok.00006627,ART-Dok,datacite,10.11588/artdok.00006627,https://doi.or

In [178]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='EconStor Preprints'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
463445,openalex::W2241195487,EconStor Preprints,openalex,<NA>,None,https://econpapers.repec.org/RePEc:zbw:espost:...,article,None,Electric Vehicles in Imperfect Electricity Mar...,Wolf-Peter Schill,2011.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://econpapers.repec.org/repec:zbw:espost:...,<na>,None,None,NaN,None,None,NaN
428479,openalex::W2888303629,EconStor Preprints,openalex,<NA>,None,https://econpapers.repec.org/RePEc:zbw:espost:...,article,None,Electric Vehicles in Imperfect Electricity Mar...,Wolf-Peter Schill,2011.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://econpapers.repec.org/repec:zbw:espost:...,<na>,None,None,NaN,None,None,NaN
476286,openalex::W3049474313,EconStor Preprints,openalex,<NA>,None,http://hdl.handle.net/10419/218808,article,None,Industrie dämpft die konjunkturelle Erholung,Schmidt Torsten; György Barabás; Boris Blagov;...,2019.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hdl.handle.net/10419/218808,<na>,None,None,NaN,None,None,NaN
485114,openalex::W7112520020,EconStor Preprints,openalex,<NA>,None,http://hdl.handle.net/10419/218808,article,None,Industrie dämpft die konjunkturelle Erholung,"Schmidt Torsten; Barabás, György; Blagov, Bori...",2019.0,2025-12-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hdl.handle.net/10419/218808,<na>,None,None,NaN,None,None,NaN
471237,openalex::W2756173876,EconStor Preprints,openalex,<NA>,None,http://hdl.handle.net/10419/172735,article,None,"Neue Ordnung, neues Glück? Ordnungs- und fiska...",Markus Breuer; Luca Rebeggiani,2017.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hdl.handle.net/10419/172735,<na>,None,None,NaN,None,None,NaN
471531,openalex::W2774257793,EconStor Preprints,openalex,<NA>,None,http://hdl.handle.net/10419/172735,article,None,"Neue Ordnung, neues Glück?New Rules, New Luck?...",Luca Rebeggiani; Markus Breuer,2017.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hdl.handle.net/10419/172735,<na>,None,None,NaN,None,None,NaN
427444,openalex::W2265687887,EconStor Preprints,openalex,<NA>,None,http://hdl.handle.net/10419/124252,preprint,None,Technological Progress and Economic Geography_...,Jacques Thisse; Takatoshi Tabuchi; Xiwei Zhu,2014.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hdl.handle.net/10419/124252,<na>,None,None,NaN,None,None,NaN
484906,openalex::W7112003676,EconStor Preprints,openalex,<NA>,None,http://hdl.handle.net/10419/124252,other,None,Technological Progress and Economic Geography,Thisse Jacques; Tabuchi Takatoshi; Zhu Xiwei,2014.0,2025-12-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://hdl.handle.net/10419/124252,<na>,None,None,NaN,None,None,NaN


In [179]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='viXra'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
9772179,openalex::W2419579416,viXra,openalex,<NA>,None,https://vixra.org/pdf/1405.0153v1.pdf,article,None,Pregnancy hormones in cardiovascular disease.,Denise Hilfiker‐Kleiner,2015.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://vixra.org/pdf/1405.0153v1.pdf,<na>,None,None,NaN,None,None,NaN
9782498,openalex::W26151073,viXra,openalex,<NA>,None,https://vixra.org/pdf/1405.0153v1.pdf,preprint,None,Structure of Chromatic Polynomials on Quasi - ...,R.V.N. SrinivasaRao; J. VenkateswaraRao; T. Na...,2014.0,2016-06-24T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://vixra.org/pdf/1405.0153v1.pdf,<na>,None,None,NaN,None,None,NaN
9772315,openalex::W18990831,viXra,openalex,<NA>,None,https://vixra.org/pdf/1405.0117v1.pdf,review,None,Optimization of Green Sand Casting Process Par...,Sanjay S. Jamkar; M. J. Deshmukh; N.A. Vidhate,2014.0,2016-06-24T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://vixra.org/pdf/1405.0117v1.pdf,<na>,None,None,NaN,None,None,NaN
9791191,openalex::W2994514029,viXra,openalex,<NA>,None,https://vixra.org/pdf/1405.0117v1.pdf,article,None,Optimization of Green Sand Casting Process Par...,John Casillas,2014.0,2019-12-13T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://vixra.org/pdf/1405.0117v1.pdf,<na>,None,None,NaN,None,None,NaN
9772851,openalex::W2412976284,viXra,openalex,<NA>,None,http://article.aascit.org/file/pdf/9090755.pdf,article,None,Ether-medium and a new constant on photons rad...,Edward F. Donnelly; T N Chase,2014.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://article.aascit.org/file/pdf/9090755.pdf,<na>,None,None,NaN,None,None,NaN
9796033,openalex::W4706860,viXra,openalex,<NA>,None,http://article.aascit.org/file/pdf/9090755.pdf,preprint,None,Ether-medium and a new constant on photons rad...,Jian Ding; HU Xiuqin,2014.0,2016-06-24T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,http://article.aascit.org/file/pdf/9090755.pdf,<na>,None,None,NaN,None,None,NaN


In [180]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='PeerJ Preprints'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
2218360,crossref::10.7287/peerj.preprints.3516,PeerJ Preprints,crossref,10.7287/peerj.preprints.3516,https://doi.org/10.7287/peerj.preprints.3516,https://peerj.com/preprints/3516,posted-content,preprint,Factors affecting silk production in Japanese ...,"Zuko, Yeti; Maeda, Kazuo",2018.0,2018-01-09,None,None,,,,,false,None,None,None,None,parent,https://peerj.com/preprints/3516,10.7287/peerj.preprints.3516,None,None,NaN,None,None,NaN
2218361,crossref::10.7287/peerj.preprints.3516v1,PeerJ Preprints,crossref,10.7287/peerj.preprints.3516v1,https://doi.org/10.7287/peerj.preprints.3516v1,https://peerj.com/preprints/3516,posted-content,preprint,Factors affecting silk production in Japanese ...,"Zuko, Yeti; Maeda, Kazuo",2018.0,2018-01-09,None,None,,,,,false,None,None,None,None,parent,https://peerj.com/preprints/3516,10.7287/peerj.preprints.3516v1,None,None,NaN,None,None,NaN
2218366,crossref::10.7287/peerj.preprints.26897,PeerJ Preprints,crossref,10.7287/peerj.preprints.26897,https://doi.org/10.7287/peerj.preprints.26897,https://peerj.com/preprints/26897,posted-content,preprint,The effect of intestinal <i>Bifidobacterium</i...,"Zuko, Yeti; Maeda, Kazuo",2018.0,2018-04-26,None,None,,,,,false,None,None,None,None,parent,https://peerj.com/preprints/26897,10.7287/peerj.preprints.26897,None,None,NaN,None,None,NaN
2218365,crossref::10.7287/peerj.preprints.26897v1,PeerJ Preprints,crossref,10.7287/peerj.preprints.26897v1,https://doi.org/10.7287/peerj.preprints.26897v1,https://peerj.com/preprints/26897,posted-content,preprint,The effect of intestinal <i>Bifidobacterium</i...,"Zuko, Yeti; Maeda, Kazuo",2018.0,2018-04-26,None,None,,,,,false,None,None,None,None,parent,https://peerj.com/preprints/26897,10.7287/peerj.preprints.26897v1,None,None,NaN,None,None,NaN
2218363,crossref::10.7287/peerj.preprints.26769,PeerJ Preprints,crossref,10.7287/peerj.preprints.26769,https://doi.org/10.7287/peerj.preprints.26769,https://peerj.com/preprints/26769,posted-content,preprint,The effect of cyclical stimulation on the prim...,"Zuko, Yeti; Maeda, Kazuo",2018.0,2018-03-25,None,None,,,,,false,None,None,None,None,parent,https://peerj.com/preprints/26769,10.7287/peerj.preprints.26769,None,None,NaN,None,None,NaN
2218362,crossref::10.7287/peerj.preprints.26769v1,PeerJ Preprints,crossref,10.7287/peerj.preprints.26769v1,https://doi.org/10.7287/peerj.preprints.26769v1,https://peerj.com/preprints/26769,posted-content,preprint,The effect of cyclical stimulation on the prim...,"Zuko, Yeti; Maeda, Kazuo",2018.0,2018-03-25,None,None,,,,,false,None,None,None,None,parent,https://peerj.com/preprints/26769,10.7287/peerj.preprints.26769v1,None,None,NaN,None,None,NaN


In [181]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='MetaArXiv'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
1973849,crossref::10.31222/osf.io/uxf39,MetaArXiv,crossref,10.31222/osf.io/uxf39,https://doi.org/10.31222/osf.io/uxf39,https://osf.io/uxf39_v1,posted-content,preprint,Do Pre-Registration and Pre-analysis Plans Red...,"Brodeur, Abel; Cook, Nikolai; Hartley, Jonatha...",2022.0,2022-08-11,None,None,,,,,false,None,None,None,None,parent,https://osf.io/uxf39_v1,10.31222/osf.io/uxf39,_v1,explicit_version,1.0,NaN,NaN,NaN
1973472,crossref::10.31222/osf.io/uxf39_v1,MetaArXiv,crossref,10.31222/osf.io/uxf39_v1,https://doi.org/10.31222/osf.io/uxf39_v1,https://osf.io/uxf39_v1,posted-content,preprint,Do Pre-Registration and Pre-analysis Plans Red...,"Brodeur, Abel; Cook, Nikolai; Hartley, Jonatha...",2022.0,2025-06-23,None,None,,,,,false,None,None,None,None,parent,https://osf.io/uxf39_v1,10.31222/osf.io/uxf39_v1,_v1,explicit_version,1.0,NaN,NaN,NaN
1973847,crossref::10.31222/osf.io/a9vhr,MetaArXiv,crossref,10.31222/osf.io/a9vhr,https://doi.org/10.31222/osf.io/a9vhr,https://osf.io/a9vhr_v1,posted-content,preprint,We Need to Talk about Mechanical Turk: What 22...,"Brodeur, Abel; Cook, Nikolai; Heyes, Anthony",2022.0,2022-08-11,None,None,,,,,false,None,None,None,None,parent,https://osf.io/a9vhr_v1,10.31222/osf.io/a9vhr,_v1,explicit_version,1.0,NaN,NaN,NaN
1973473,crossref::10.31222/osf.io/a9vhr_v1,MetaArXiv,crossref,10.31222/osf.io/a9vhr_v1,https://doi.org/10.31222/osf.io/a9vhr_v1,https://osf.io/a9vhr_v1,posted-content,preprint,We Need to Talk about Mechanical Turk: What 22...,"Brodeur, Abel; Cook, Nikolai; Heyes, Anthony",2022.0,2025-06-23,None,None,,,,,false,None,None,None,None,parent,https://osf.io/a9vhr_v1,10.31222/osf.io/a9vhr_v1,_v1,explicit_version,1.0,NaN,NaN,NaN
1973853,crossref::10.31222/osf.io/8ya3m,MetaArXiv,crossref,10.31222/osf.io/8ya3m,https://doi.org/10.31222/osf.io/8ya3m,https://osf.io/8ya3m_v1,posted-content,preprint,The influence of journal submission guidelines...,"Giofrè, David; Boedker, Ingrid; Cumming, Geoff...",2022.0,2022-03-07,None,None,,,,,false,None,None,None,None,parent,https://osf.io/8ya3m_v1,10.31222/osf.io/8ya3m,_v1,explicit_version,1.0,NaN,NaN,NaN
1973478,crossref::10.31222/osf.io/8ya3m_v1,MetaArXiv,crossref,10.31222/osf.io/8ya3m_v1,https://doi.org/10.31222/osf.io/8ya3m_v1,https://osf.io/8ya3m_v1,posted-content,preprint,The influence of journal submission guidelines...,"Giofrè, David; Boedker, Ingrid; Cumming, Geoff...",2022.0,2025-06-25,None,None,,,,,false,None,None,None,None,parent,https://osf.io/8ya3m_v1,10.31222/osf.io/8ya3m_v1,_v1,explicit_version,1.0,NaN,NaN,NaN


In [182]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='WikiJournal of Humanities'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi


In [183]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='WikiJournal of Medicine'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi


In [184]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='Bepress Legal Repository'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi


In [185]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='Social Science Open Access Repository'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
6219709,openalex::W2948778104,Social Science Open Access Repository,openalex,10.5167/uzh-160317,https://doi.org/10.5167/uzh-160317,https://www.ssoar.info/ssoar/handle/document/6...,article,None,"Kommentar zu Meiser, T. et al. (2018). Positio...",Natalie Nagowski; Peter Kirsch; Andrea Kübler;...,2018.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://www.ssoar.info/ssoar/handle/document/6...,10.5167/uzh-160317,None,None,NaN,None,None,NaN
6222277,openalex::W3198948865,Social Science Open Access Repository,openalex,<NA>,None,https://www.ssoar.info/ssoar/handle/document/6...,article,None,Fachgruppe Gesundheitspsychologie: Methoden si...,Petra Warschburger; Gudrun Sproesser; Daniela ...,2018.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://www.ssoar.info/ssoar/handle/document/6...,<na>,None,None,NaN,None,None,NaN
6199687,openalex::W2748208804,Social Science Open Access Repository,openalex,<NA>,None,https://www.ssoar.info/ssoar/handle/document/6...,article,None,Deutscher Alterssurvey (DEAS): Kurzbeschreibun...,Heribert Engstler; Nicole Hameister,2019.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://www.ssoar.info/ssoar/handle/document/6...,<na>,None,None,NaN,None,None,NaN
6222461,openalex::W3208177612,Social Science Open Access Repository,openalex,<NA>,None,https://www.ssoar.info/ssoar/handle/document/6...,article,None,Deutscher Alterssurvey (DEAS): Kurzbeschreibun...,Heribert Engstler; Nicole Hameister,2021.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://www.ssoar.info/ssoar/handle/document/6...,<na>,None,None,NaN,None,None,NaN


In [186]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='Advance'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
16416,crossref::10.31124/advance.171863606.69452326/v1,Advance,crossref,10.31124/advance.171863606.69452326/v1,https://doi.org/10.31124/advance.171863606.694...,https://advance.sagepub.com/users/719316/artic...,posted-content,preprint,CLUSTER DEVELOPMENT ECONOMIC POLICY,"Lomsadze, Tinatin",2024.0,2024-06-17,None,None,,,,,false,None,None,None,None,parent,https://advance.sagepub.com/users/719316/artic...,10.31124/advance.171863606.69452326/v1,None,None,NaN,/v1,explicit_version,1.0
16420,crossref::10.31124/advance.171897777.74136883/v1,Advance,crossref,10.31124/advance.171897777.74136883/v1,https://doi.org/10.31124/advance.171897777.741...,https://advance.sagepub.com/users/719316/artic...,posted-content,preprint,CLUSTER DEVELOPMENT ECONOMIC POLICY,"Lomsadze, Tinatin",2024.0,2024-07-03,None,None,,,,,false,None,None,None,None,parent,https://advance.sagepub.com/users/719316/artic...,10.31124/advance.171897777.74136883/v1,None,None,NaN,/v1,explicit_version,1.0


In [187]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='Preprints.org'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
2231072,crossref::10.20944/preprints201904.0246.v1,Preprints.org,crossref,10.20944/preprints201904.0246.v1,https://doi.org/10.20944/preprints201904.0246.v1,http://www.preprints.org/manuscript/201904.024...,posted-content,preprint,Mathematical Models for Possible Roles of Oxyt...,"Gottlieb, Mark",2019.0,2019-04-22,None,None,,,,,false,None,None,None,None,parent,http://www.preprints.org/manuscript/201904.024...,10.20944/preprints201904.0246.v1,/v1,explicit_version,1.0,NaN,NaN,NaN
2231078,crossref::10.20944/preprints201904.0246.v2,Preprints.org,crossref,10.20944/preprints201904.0246.v2,https://doi.org/10.20944/preprints201904.0246.v2,http://www.preprints.org/manuscript/201904.024...,posted-content,preprint,Mathematical Models for Possible Roles of Oxyt...,"Gottlieb, Mark",2019.0,2019-04-24,None,None,,,,,false,None,None,None,None,parent,http://www.preprints.org/manuscript/201904.024...,10.20944/preprints201904.0246.v2,/v1,explicit_version,1.0,NaN,NaN,NaN


In [188]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='JMIR Preprints'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
1930190,crossref::10.2196/iproc.8586,JMIR Preprints,crossref,10.2196/iproc.8586,https://doi.org/10.2196/iproc.8586,http://preprints.jmir.org/preprint/8989,posted-content,preprint,Barriers and facilitators to patient portal im...,"Kooij, Laura; Groen, Wim G; van Harten, Wim H",2017.0,2017-09-25,None,None,,,,,false,None,None,None,None,parent,http://preprints.jmir.org/preprint/8989,10.2196/iproc.8586,None,None,NaN,None,None,NaN
1931127,crossref::10.2196/preprints.8989,JMIR Preprints,crossref,10.2196/preprints.8989,https://doi.org/10.2196/preprints.8989,http://preprints.jmir.org/preprint/8989,posted-content,preprint,Barriers and Facilitators Affecting Patient Po...,"Kooij, Laura; Groen, Wim G; van Harten, Wim H",2017.0,2017-09-26,"{""is-preprint-of"": [{""asserted-by"": ""subject"",...",None,,10.2196/jmir.8989,,,true,None,None,None,None,parent,http://preprints.jmir.org/preprint/8989,10.2196/preprints.8989,None,None,NaN,None,None,NaN


In [189]:
dupes_landing_page_url[dupes_landing_page_url['server_name']=='National Bureau of Economic Research'].sort_values(by='landing_page_url', ascending=False)

,record_id,server_name,backend,doi,doi_url,landing_page_url,type_backend_raw,subtype_backend_raw,title,authors_flat,publication_year,date_created,relations_json,version_label,is_version_of,is_preprint_of,has_preprint,has_review,has_published_version,published_version_ids_json,version_of_ids_json,update_to_json,raw_relationships_json,records_hierarchy,landing_norm,doi_norm,version_token_lp,token_kind_lp,vnum_lp,version_token_doi,token_kind_doi,vnum_doi
2075189,openalex::W2567960855,National Bureau of Economic Research,openalex,<NA>,None,https://eric.ed.gov/?id=ED524978,article,None,"Creating ""No Excuses"" (Traditional) Public Sch...",Roland G. Fryer,2011.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://eric.ed.gov/?id=ed524978,<na>,None,None,NaN,None,None,NaN
2075195,openalex::W2913912915,National Bureau of Economic Research,openalex,<NA>,None,https://eric.ed.gov/?id=ED524978,article,None,Injecting Successful Charter School Strategies...,Roland G. Fryer,2011.0,2025-10-10T00:00:00,None,None,None,None,None,None,None,None,None,None,None,parent,https://eric.ed.gov/?id=ed524978,<na>,None,None,NaN,None,None,NaN


#### function

In [190]:
import pandas as pd
import re

# -----------------------------
# 0) Define primary servers
# -----------------------------
# PRIMARY_SERVERS = {
#     "arXiv",
#     "bioRxiv",
#     "medRxiv",
#     "SSRN",
#     "TechRxiv",
#     "Wellcome Open Research",
#     "AgriRxiv",
#     "EarthArXiv",
#     "Law Archive",
#     "SocArXiv",
#     "Thesis Commons",
#     "Research Square",
#     "Open Research Europe",
#     "Oroboros Instruments",
#     "ResearchGate",
#     "AgEcon Search",
#     "Zenodo",
#     "Open Science Framework",
#     "Humanities Commons CORE",
#     "CERN document server",
#     "eLife",
# }

PRIMARY_SERVERS = set(primary_servers) 
# -----------------------------
# 1) Normalize landing_page_url
# -----------------------------
url_norm = (
    df_mirror["landing_page_url"]
    .astype('string')
    .str.strip()
    .str.lower()
    .replace({"": pd.NA, "none": pd.NA, "nan": pd.NA, "null": pd.NA})
    # .replace({"": pd.NA, "none": pd.NA})
    # remove query strings and fragments
    .str.replace(r"[?#].*$", "", regex=True)
    # remove trailing slash
    .str.rstrip("/")
)


# -----------------------------
# 2) Flags
# -----------------------------
is_primary = df_mirror["server_name"].isin(PRIMARY_SERVERS)

# duplicated URL groups
is_dup = url_norm.notna() & url_norm.duplicated(keep=False)

# -----------------------------
# 3) Build URL → primary server mapping (FAST)
#    Only once, only for duplicated URLs
# -----------------------------
primary_by_url = (
    df_mirror.loc[is_primary & is_dup, ["server_name"]]
    .assign(url=url_norm[is_primary & is_dup])
    .dropna(subset=["url"])
    .groupby("url")["server_name"]
    .first()   # deterministic + fast
)

# -----------------------------
# 4) Label mirrors
#    - non-primary
#    - duplicated URL
#    - duplicated with at least one primary
# -----------------------------
mask_mirror = (
    is_dup
    & ~is_primary
    & url_norm.isin(primary_by_url.index)
)

df_mirror.loc[mask_mirror, "records_hierarchy"] = (
    "mirror (" + url_norm[mask_mirror].map(primary_by_url) + ")"
)

# -----------------------------
# 5) Sanity checks
# -----------------------------
print("Mirror rows set:", mask_mirror.sum())
print(df_mirror["records_hierarchy"].value_counts(dropna=False))


Mirror rows set: 2757
records_hierarchy
parent                              7825713
version                              102319
publish_version                       12888
mirror (AgEcon Search)                 6607
mirror (arXiv)                         6419
part_of                                5921
NaN                                    2593
child                                  2061
mirror (ResearchGate)                   827
correction                              354
comment                                 242
mirror (Zenodo)                         191
mirror (bioRxiv)                         29
review                                   27
mirror (SSRN)                            23
mirror (Open Science Framework)          21
mirror (Humanities Commons CORE)         16
others                                   12
parent_duplicate                          4
mirror (EarthArXiv)                       2
mirror (AfricArXiv)                       2
mirror (Research Square)            

# Resolve duplicates across OSF + OSF communities using the OSF id (osf.io/<id>)

In [191]:
## Quick proof (diagnostic)

In [192]:
# import pandas as pd
# import re

# def extract_osf_id(x):
#     if pd.isna(x):
#         return pd.NA
#     m = re.search(r"osf\.io/([a-z0-9]+)", str(x).lower())
#     return m.group(1) if m else pd.NA

# tmp = df_mirror.copy()

# tmp["osf_id_from_doi"] = tmp["doi"].map(extract_osf_id) if "doi" in tmp.columns else pd.NA
# tmp["osf_id_from_lp"]  = tmp["landing_page_url"].map(extract_osf_id) if "landing_page_url" in tmp.columns else pd.NA

# tmp["osf_id"] = tmp["osf_id_from_doi"].fillna(tmp["osf_id_from_lp"])

# print("Rows with OSF id:", tmp["osf_id"].notna().sum())
# print("Duplicated OSF id rows:", tmp["osf_id"].notna().sum() - tmp["osf_id"].dropna().nunique())
# print(tmp.loc[tmp["osf_id"].notna(), "server_name"].value_counts().head(20))


In [193]:
## function: resolve OSF duplicates by OSF ID (works across different DOI prefixes)

In [194]:
# import pandas as pd
# import numpy as np
# import re

# def resolve_osf_duplicates_fast(
#     df: pd.DataFrame,
#     preferred_parent_servers: set,
#     prefer_backend: str = "crossref",
#     choose_parent: str = "oldest",           # "oldest" or "most_recent"
#     date_col: str = "date_created",          # if missing -> record_id fallback
#     overwrite_mode: str = "parent_only",     # "any" | "parent_only" | "unlabeled_only"
#     # columns
#     server_col: str = "server_name",
#     backend_col: str = "backend",
#     record_id_col: str = "record_id",
#     doi_col: str = "doi",
#     landing_col: str = "landing_page_url",
#     hierarchy_col: str = "records_hierarchy",
#     parent_id_col: str = "parent_record_id",
#     # perf knobs
#     coarse_filter: str = "osf.io/",          # cheap contains() filter before regex
#     min_group_size: int = 2
# ) -> pd.DataFrame:
#     """
#     Fast OSF duplicate resolver for huge frames (millions of rows).

#     Key idea:
#       - Avoid regex over the full dataframe.
#       - First, cheaply filter rows that likely contain 'osf.io/' in doi or landing_page_url.
#       - Only then extract OSF id and group to resolve duplicates.
#       - Finally, write results back only for touched rows.

#     Parent selection priority:
#       1) preferred community servers first (SocArXiv, PsyArXiv, etc.)
#       2) prefer backend == prefer_backend (crossref)
#       3) date (oldest/most_recent)
#       4) record_id numeric key (tie-break)
#     """

#     out = df.copy()

#     # Ensure output cols exist
#     if hierarchy_col not in out.columns:
#         out[hierarchy_col] = pd.NA
#     if parent_id_col not in out.columns:
#         out[parent_id_col] = pd.NA

#     # Eligibility mask (run only where you allow overwriting)
#     h = out[hierarchy_col]
#     if overwrite_mode == "any":
#         eligible = pd.Series(True, index=out.index)
#     elif overwrite_mode == "parent_only":
#         eligible = h.astype(str).str.strip().str.lower().eq("parent")
#     elif overwrite_mode == "unlabeled_only":
#         eligible = h.isna()
#     else:
#         raise ValueError("overwrite_mode must be: any | parent_only | unlabeled_only")

#     # ---- 1) Coarse filter: only rows likely to be OSF-related (very fast)
#     # Use fillna("") so .str.contains doesn't create object issues
#     m_osf = pd.Series(False, index=out.index)

#     if doi_col in out.columns:
#         m_osf |= out[doi_col].fillna("").astype(str).str.contains(coarse_filter, case=False, regex=False)
#     if landing_col in out.columns:
#         m_osf |= out[landing_col].fillna("").astype(str).str.contains(coarse_filter, case=False, regex=False)

#     m = eligible & m_osf
#     if not m.any():
#         return out

#     # Work on subset only
#     sub = out.loc[m, [server_col, backend_col, record_id_col]].copy()

#     # ---- 2) Extract OSF id from subset (regex only on ~235k rows, not 8M)
#     pat = re.compile(r"osf\.io/([a-z0-9]+)", re.I)

#     def extract_osf_id_series(s: pd.Series) -> pd.Series:
#         return s.fillna("").astype(str).str.lower().str.extract(pat, expand=False)

#     osf_id = pd.Series(pd.NA, index=sub.index, dtype="object")
#     if doi_col in out.columns:
#         osf_id = extract_osf_id_series(out.loc[m, doi_col])
#     if landing_col in out.columns:
#         osf_id = osf_id.fillna(extract_osf_id_series(out.loc[m, landing_col]))

#     # drop rows with no extracted id (coarse filter can include a few false positives)
#     sub["_osf_id"] = osf_id
#     sub = sub[sub["_osf_id"].notna()].copy()
#     if sub.empty:
#         return out

#     # ---- 3) Prepare sort keys on subset
#     sub["_is_pref_server"] = sub[server_col].isin(preferred_parent_servers)

#     # backend preference (vectorized)
#     sub["_is_pref_backend"] = (
#         sub[backend_col].fillna("").astype(str).str.lower().eq(str(prefer_backend).lower())
#     )

#     # date key
#     if date_col in out.columns:
#         sub["_dt"] = pd.to_datetime(out.loc[sub.index, date_col], errors="coerce")
#     else:
#         sub["_dt"] = pd.NaT

#     # record_id numeric key (extract digits)
#     rid_digits = sub[record_id_col].astype(str).str.extract(r"(\d+)")[0]
#     sub["_rid_key"] = pd.to_numeric(rid_digits, errors="coerce")

#     if choose_parent not in {"oldest", "most_recent"}:
#         raise ValueError("choose_parent must be: oldest | most_recent")
#     date_asc = (choose_parent == "oldest")

#     # ---- 4) Only group IDs that actually have duplicates (saves time)
#     # value_counts on 235k rows is cheap
#     dup_ids = sub["_osf_id"].value_counts()
#     dup_ids = dup_ids[dup_ids >= min_group_size].index
#     sub = sub[sub["_osf_id"].isin(dup_ids)].copy()
#     if sub.empty:
#         return out

#     # ---- 5) Resolve per OSF id
#     # Sorting once, then picking first per group is faster than looping all groups with Python
#     sub_sorted = sub.sort_values(
#         by=["_osf_id", "_is_pref_server", "_is_pref_backend", "_dt", "_rid_key"],
#         ascending=[True, False, False, date_asc, True],
#         na_position="last"
#     )

#     # Parent idx per osf_id = first row after sorting
#     parent_idx_by_id = sub_sorted.groupby("_osf_id", sort=False).head(1)
#     parent_map_rid = parent_idx_by_id.set_index("_osf_id")[record_id_col]
#     parent_map_srv = parent_idx_by_id.set_index("_osf_id")[server_col]

#     # For all rows in sub_sorted, map parent rid & parent server
#     sub_sorted["_parent_rid"] = sub_sorted["_osf_id"].map(parent_map_rid)
#     sub_sorted["_parent_srv"] = sub_sorted["_osf_id"].map(parent_map_srv)

#     # Identify which rows are parent vs child
#     is_parent_row = sub_sorted[record_id_col].eq(sub_sorted["_parent_rid"])

#     # Write back to OUT
#     parent_rows = sub_sorted.index[is_parent_row]
#     child_rows  = sub_sorted.index[~is_parent_row]

#     out.loc[parent_rows, hierarchy_col] = "parent"
#     out.loc[parent_rows, parent_id_col] = pd.NA

#     out.loc[child_rows, hierarchy_col] = "mirror (" + sub_sorted.loc[child_rows, "_parent_srv"].astype(str) + ")"
#     out.loc[child_rows, parent_id_col] = sub_sorted.loc[child_rows, "_parent_rid"].values

#     return out


In [195]:
# PREFERRED_OSF_PARENT = {
#     "SocArXiv","Law Archive","PsyArXiv","EdArXiv","EarthArXiv","Thesis Commons",
#     "LIS Scholarship Archive","SportRxiv","INA-Rxiv","Arabixiv","engrXiv","MetaArXiv",
#     "MindRxiv","MarXiv","AgriRxiv","NutriXiv","ECSarXiv","FocUS Archive","Frenxiv",
#     "EcoEvoRxiv","IndiaRxiv","PaleorXiv","AfricArXiv","BioHackrXiv","MediArXiv"
# }

# df_mirror = resolve_osf_duplicates_fast(
#     df=df_mirror,
#     preferred_parent_servers=PREFERRED_OSF_PARENT,
#     prefer_backend="crossref",
#     choose_parent="oldest",
#     date_col="date_created",
#     overwrite_mode="parent_only",
# )

# print(df_mirror["records_hierarchy"].value_counts(dropna=False).head(30))


# save

In [196]:
records_hierarchy_df = df_mirror[['record_id','server_name','records_hierarchy']]
records_hierarchy_df

,record_id,server_name,records_hierarchy
399051,crossref::10.1002/essoar.10500000.1,Earth and Space Science Open Archive,parent
399052,crossref::10.1002/essoar.10500002.1,Earth and Space Science Open Archive,parent
399047,crossref::10.1002/essoar.10500004.1,Earth and Space Science Open Archive,parent
399073,crossref::10.1002/essoar.10500007.1,Earth and Space Science Open Archive,parent
399068,crossref::10.1002/essoar.10500009.1,Earth and Space Science Open Archive,parent
...,...,...,...
2598815,openalex::W999921877,RePEc: Research Papers in Economics,parent
974376,openalex::W999947037,HAL,parent
2505522,openalex::W999974616,RePEc: Research Papers in Economics,parent
973276,openalex::W999989114,HAL,parent


In [197]:
records_hierarchy_df.to_csv("outputs_new/records_hierarchy_df.csv", index=False)
records_hierarchy_df.to_pickle("outputs_new/records_hierarchy_df.pkl")

In [198]:
ccc

NameError: name 'ccc' is not defined

In [ ]:
some paper have same title and author infos, but different may have different doi or landing page url in the same servers. 
    we need to clean title first or decide that the title have to be equal at 90% per example to overcome some little typo difference
    could it be possibe to write a code that will get one version of rows, 
        the version choose will be label parent and others childs (duplicate),
        we need also to work server by server and a way for each server to decide if we get the most recent or the old on as parent. 



how to have a function, that, we can add as input, server, or list of server, the columns one or multiples columns we need to use to find duplicates, wich records id to choose, the most recent or old one by sorting

# VeriXiv and Gate

In [ ]:
data[data['server_name']=='VeriXiv']

In [ ]:
gate_data = data[data['server_name']=='Gates Open Research']
gate_data

In [ ]:
pattern = "10.12688/verixiv.244"

mask = data['doi'].str.contains(pattern, regex=False, na=False)
result = data[mask]
result

In [ ]:
pattern = "10.12688/gatesopenres.16372"

mask = data['doi'].str.contains(pattern, regex=False, na=False)
result = data[mask]
result

In [ ]:
result['relations_json'][636828]

In [ ]:
pattern = "10.12688/verixiv.244.3"

mask = data['doi'].str.contains(pattern, regex=False, na=False)
result = data[mask]
result

In [ ]:
gate_data

In [ ]:
pattern = "10.12688/verixiv."

mask = gate_data['relations_json'].str.contains(pattern, regex=False, na=False)
result = gate_data[mask]
result

In [ ]:
result.shape

In [ ]:
pattern = "has-preprint"

mask = gate_data['relations_json'].str.contains(pattern, regex=False, na=False)
result = gate_data[mask]
print(result.shape)
result

In [ ]:
gate_data2025 = gate_data[gate_data['publication_year'] == '2025.0']
print(gate_data2025.shape)
gate_data2025

In [ ]:
gate_data2025first = gate_data2025[gate_data2025['is_version_of']=='']
print(gate_data2025first.shape)
gate_data2025first

In [ ]:
pattern = "has-preprint"

mask = ~gate_data2025['relations_json'].str.contains(pattern, regex=False, na=False)
result = gate_data2025[mask]
print(result.shape)
result

In [ ]:
pattern = "10.12688/gatesopenres.15431.1"

mask = data['doi'].str.contains(pattern, regex=False, na=False)
result = data[mask]
print(result.shape)
result

## Keldysh Institute Preprints

In [ ]:
data[data['server_name']=='Keldysh Institute Preprints']

# elife_data

In [ ]:
file_path = r"/mnt/c/SCHOLCOMMLAB/APPS/preprint-harvester/data/by_server/eLife/eLife_rule72_1990-01-01_2025-12-31_crossref.parquet"
elife_data = pd.read_parquet(file_path) 
elife_data

In [ ]:
elife_data["type_backend_raw"].value_counts()

In [ ]:
elife_data["issn"].value_counts()

In [ ]:
elife_data[elife_data['type_backend_raw']=='journal-article']

In [ ]:
elife_data[elife_data['type_backend_raw']=='posted-content']

In [ ]:
elife_data_issn = elife_data[elife_data['issn'].notna()]
elife_data_issn

In [ ]:
elife_data_no_issn = elife_data[elife_data['issn'].isna()]
elife_data_no_issn